# **Document Preprocessing**

## Pipeline A — Turning Everything Into Text

This is the first of the three pipelines in `Book_Writer_System_Design.md`. Its job is small to
state and easy to get wrong:

> Take a pile of heterogeneous material — PDFs, slide decks, images, and **recorded lectures** —
> and turn it into one clean text corpus, *without losing the information that makes the material
> worth using in the first place.*

Everything downstream is built on what this pipeline produces. To quote the design document:

> *"If a lecture is transcribed with the wrong technical vocabulary, or a diagram is described as
> 'a flowchart with boxes and arrows', no amount of clever memory design later will recover it."*

So this section is worth being fussy about. The rest of the book writer can only be as good as
the corpus this stage hands it.

---

### What Pipeline A must produce

Four artifacts. The first one is the obvious one; the other three are the ones the earlier version
of this notebook forgot, and they are what Pipeline C's memory layers actually run on.

| Artifact | What it is | Who consumes it |
|---|---|---|
| `raw_consolidated_text.txt` | The whole corpus as one text file, document boundaries marked | Pipeline B's `TextChunker` |
| `document_index.json` | Exact character span of every document, plus its **provenance** | Layer 3 (Source Memory) — "which document did this claim come from?" |
| `figure_store.json` + `data/figures/*.png` | The retained image crops and their descriptions | Layer 4 (Context Assembler) — the Writer is multimodal and gets to *see* the diagram |
| `data/transcripts/*.raw.json` | Whisper output with **timestamps kept** | Provenance down to the minute: *"as covered around the 34-minute mark"* |

---

### The corrected architecture

```
Input: data/raw_sources/
├── papers, books, articles   (PDF, DOCX, MD, TXT)
├── slide decks               (PPTX)
├── standalone diagrams       (PNG, JPG)
└── recorded lectures         (MP4, MP3, WAV, M4A)    <-- was missing entirely
        |
        |------------------------------+
        v                              v
┌────────────────────────────┐  ┌────────────────────────────────────────┐
│ STAGE 1  SPEECH -> TEXT    │  │ STAGE 2  LAYOUT PARSING                │
│ Whisper large-v3           │  │ MinerU / Docling                       │
│ - vocabulary priming       │  │ - text blocks in reading order         │
│ - condition_on_prev=False  │  │ - image / table / equation regions     │
│ - word timestamps KEPT     │  │ - IMAGE CROPS KEPT ON DISK  <-- fixed  │
│ -> data/transcripts/*.json │  │ -> data/parsed/*.json                  │
└────────────────────────────┘  └────────────────────────────────────────┘
        |                              |
        +--------------+---------------+
                       v
┌──────────────────────────────────────────────────────────────────────┐
│ STAGE 3  MULTIMODAL -> TEXT   (Qwen VLM)                             │
│  a. drop layout junk ('discarded' blocks)              <-- fixed     │
│  b. cheap pre-filter: icons/avatars never reach the GPU <-- new      │
│  c. describe each figure WITH THE PAGE AROUND IT        <-- fixed    │
│  d. let the model answer DECORATIVE and emit nothing    <-- new      │
│  e. copy the crop into the figure store, keep the id    <-- new      │
│  f. conservative transcript cleanup (raw kept alongside) <-- new     │
│ -> data/converted/*.txt  +  figure_store.json                        │
└──────────────────────────────────────────────────────────────────────┘
                       |
                       v
┌──────────────────────────────────────────────────────────────────────┐
│ STAGE 4  CONSOLIDATION + PROVENANCE                                  │
│  - one text file, boundaries marked                                  │
│  - EXACT character offsets (verified by assertion)      <-- fixed    │
│  - source_document / source_type / timestamp per span                │
│  - build_chunk_metadata(): the Layer-3 join for Pipeline B  <-- new  │
│ -> raw_consolidated_text.txt + document_index.json                   │
└──────────────────────────────────────────────────────────────────────┘
```

---

### What was wrong before, and what changed

The previous version of this section ran end to end without raising a single exception — which is
exactly what made its problems hard to see. Every defect below is visible in the outputs the
notebook itself saved.

| # | What went wrong | Evidence from the previous run | Fix |
|---|---|---|---|
| 1 | **No audio path at all.** Lectures are described by the design as "the highest-value and lowest-quality input", and the pipeline could not read them | Stage 2 config had no media extensions | **Stage 1** added: Whisper large-v3, vocabulary-primed |
| 2 | **`discarded` blocks were treated as content.** MinerU labels page headers, footers and page numbers `discarded`; the code hit its `else` branch and wrote `[UNKNOWN CONTENT TYPE: discarded]` into the corpus | `discarded: 678 (48.1%)` of all parsed items — **half the corpus was junk markers** | Explicit drop-list, counted in stats, never emitted |
| 3 | **Figures described in isolation** with a generic prompt, so decorative furniture got 400 tokens of earnest analysis | A newsletter signup widget became *"a table with two columns: 'Type your email' and 'Subscribe'…"* | Page-context window + an explicit `DECORATIVE` escape hatch + a size pre-filter |
| 4 | **The same image was described over and over.** A Medium article repeats the author avatar, the clap icon, the follow button on every section | 95 image calls on 2 documents, ~16 s each | Content-hash cache: identical crops cost one call |
| 5 | **Image crops were thrown away.** `output_dir=None` meant the parser's crops landed somewhere temporary, and nothing was ever copied out | No `data/figures/` existed | Explicit per-document parse dir, crops copied into a figure store with stable ids |
| 6 | **Retries never retried.** `parse_single_document` catches its own exceptions and *returns* `status="failed"`, so the retry loop saw a normal return value and exited on attempt 1 | Silent: failures looked like clean failures | Retry now keys on the returned status, not on an exception |
| 7 | **One parser shared by 4 threads.** MinerU holds model state; this is not thread-safe | Silent corruption risk | Thread-local parser instances |
| 8 | **Character offsets in `document_index.json` were wrong.** Parts were joined with `"\n"` but positions were accumulated with hand-written `+1`s that did not match | Silent: the index looked plausible and pointed at the wrong text | Offsets measured from the string actually built, then **asserted** |
| 9 | **`config` was rebound three times.** `Stage2Config` → `Stage3Config` → `Stage4Config`, all under the name `config`, in one shared notebook namespace. Re-running any earlier cell silently used the wrong settings | Latent | One config object per stage, each with its own name |
| 10 | **No provenance.** Nothing recorded which document, which page, or which minute a piece of text came from | `document_index.json` had offsets only | `source_document`, `source_type`, `timestamp_index`, `figure_ids` per span |
| 11 | **`flash_attention_2` in an un-failable `try`.** The `try` wrapped a dictionary assignment, which cannot raise; the real failure happened later inside `from_pretrained` | Would crash the whole stage on a machine without flash-attn | Attention backend is tried for real, falling back `flash_attention_2 → sdpa → eager` |

There is one more thing worth saying plainly, because it is not a bug and cannot be fixed here:
the PDF text itself is lossy. The previous run produced lines like *"I did a deep dive in 201 I
have been a bit stale"* — a sentence the layout parser truncated. We normalise typography and we
flag suspicious blocks, but a book writer built on OCR-quality text will inherit OCR-quality
sentences. Check a sample of Stage 2's output by eye before you trust a run.

## Stage 0 — Shared Setup

Before any stage runs, three things get fixed in one place: **where files live**, **how logging
works**, and **the domain vocabulary**.

That third one is not obvious, so it is worth pausing on. Whisper accepts an `initial_prompt` and
conditions its transcription on it. If we seed it with the technical terms we know appear in this
corpus, "kay-vee cache" comes out as `KV cache` instead of `Katie cache`. It costs nothing and it
is the difference between a lecture being usable source material and noisy source material. The
same list is useful to Stage 3 as a hint about what the material is *about*.

The design document notes that once Pipeline B has run once, its canonical tag vocabulary is a far
better seed list than anything hand-written — so a second run of the corpus can feed
`normalized_tags.json` straight back into `DOMAIN_VOCABULARY`.

In [ ]:
# ============================================================================
# Stage 0.1 — Installation
# ============================================================================
# Run once per machine. Commented out so re-running the notebook is cheap.
#
#   raganything[all]  -> MinerU / Docling layout parsers (Stage 2)
#   faster-whisper    -> Whisper large-v3, CTranslate2 backend (Stage 1)
#   transformers      -> Qwen VLM (Stage 3)
#   ffmpeg            -> audio extraction from video containers (Stage 1)
#
# flash-attn is OPTIONAL. Stage 3 falls back to sdpa and then to eager
# attention if it is missing, so do not let a failed flash-attn build stop you.

# !pip install -q 'raganything[all]'
# !pip install -q faster-whisper
# !pip install -q 'transformers>=4.57.0' accelerate pillow
# !pip install -q flash-attn --no-build-isolation   # optional
# !apt-get -qq install -y ffmpeg

In [ ]:
# ============================================================================
# Stage 0.2 — Paths, logging, and the domain vocabulary
# ============================================================================
"""
One place that decides where everything lives.

The previous version of this section scattered directory strings across three
separate config dataclasses that were all bound to the same variable name
`config`. Re-running an earlier cell after a later one silently pointed the
earlier stage at the later stage's directories. Here the layout is declared
once, and each stage reads it.
"""

import os
import re
import json
import time
import shutil
import hashlib
import logging
import subprocess
import threading
import traceback
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple, Iterable

from tqdm.auto import tqdm


# ---------------------------------------------------------------- paths -----
@dataclass
class PipelinePaths:
    """The on-disk layout for Pipeline A, mirroring `Files on Disk` in the design doc."""

    root: Path = Path("./data")

    @property
    def raw_sources(self) -> Path:      # what you drop in: PDFs, slides, lectures
        return self.root / "raw_sources"

    @property
    def transcripts(self) -> Path:      # Stage 1: Whisper output, timestamps kept
        return self.root / "transcripts"

    @property
    def parsed(self) -> Path:           # Stage 2: layout JSON, one dir per document
        return self.root / "parsed"

    @property
    def converted(self) -> Path:        # Stage 3: plain text, one file per document
        return self.root / "converted"

    @property
    def figures(self) -> Path:          # Stage 3b: the retained image crops
        return self.root / "figures"

    @property
    def chunks(self) -> Path:           # Stage 4: chunk_0001.txt ... (Layer 3)
        return self.root / "chunks"

    @property
    def source_index(self) -> Path:     # Stage 5: Chroma over the chunks (Layer 3)
        return self.root / "source_index"

    @property
    def checkpoints(self) -> Path:
        return self.root / "checkpoints"

    @property
    def consolidated_text(self) -> Path:
        return self.root / "raw_consolidated_text.txt"

    @property
    def document_index(self) -> Path:
        return self.root / "document_index.json"

    @property
    def chunk_metadata(self) -> Path:   # L3 - provenance, timestamps, tags, figures
        return self.root / "chunk_metadata.json"

    # -- Pipeline B artifacts. The design's `Files on Disk` puts the tag files
    # -- under data/ (they describe the sources) and toc.json under storage/
    # -- (it describes the book).
    @property
    def normalized_tags(self) -> Path:    # canonical -> aliases, seeds the ledger
        return self.root / "normalized_tags.json"

    @property
    def tag_relationships(self) -> Path:  # canonical -> prerequisites
        return self.root / "tag_relationships.json"

    @property
    def chunk_tags(self) -> Path:         # chunk -> canonical tags
        return self.root / "chunk_tags.json"

    @property
    def storage(self) -> Path:
        return Path("./storage")

    @property
    def toc(self) -> Path:
        return self.storage / "toc.json"

    @property
    def figure_store(self) -> Path:
        return self.root / "figure_store.json"

    def mkdirs(self) -> None:
        for p in (self.raw_sources, self.transcripts, self.parsed,
                  self.converted, self.figures, self.chunks, self.checkpoints,
                  self.storage):
            p.mkdir(parents=True, exist_ok=True)


PATHS = PipelinePaths()
PATHS.mkdirs()


# -------------------------------------------------------------- logging -----
def make_logger(name: str, logfile: str) -> logging.Logger:
    """
    One logger per stage, each with its own file.

    Jupyter re-runs cells, and `logging.basicConfig` is a no-op the second time
    it is called — which is why the previous version's per-stage log files
    quietly all went to whichever file was configured first. Building the
    logger explicitly and clearing old handlers avoids that.
    """
    lg = logging.getLogger(name)
    lg.setLevel(logging.INFO)
    lg.handlers.clear()
    lg.propagate = False

    fmt = logging.Formatter("%(asctime)s | %(name)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")

    fh = logging.FileHandler(logfile, encoding="utf-8")
    fh.setFormatter(fmt)
    lg.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    lg.addHandler(sh)
    return lg


# ------------------------------------------------- the domain vocabulary ----
# Seed Whisper with the terms you already know are in the corpus. Anything you
# leave out, Whisper will guess at phonetically.
#
# After Pipeline B has run once, replace this list with the canonical tag
# vocabulary from `normalized_tags.json` -- it is a strictly better seed and it
# costs nothing, because that LLM bill has already been paid.
DOMAIN_VOCABULARY: List[str] = [
    "transformer", "attention", "self-attention", "multi-head attention",
    "KV cache", "tokenizer", "embedding", "positional encoding",
    "RAG", "retrieval augmented generation", "vector database", "chunking",
    "reranker", "LLM", "fine-tuning", "LoRA", "quantization",
    "ReAct loop", "agent", "tool calling", "function calling",
    "PyTorch", "CUDA", "Hugging Face", "prompt engineering",
]


def load_vocabulary_from_pipeline_b(normalized_tags_path: Path) -> List[str]:
    """
    If a previous run produced Pipeline B's canonical tag vocabulary, use it.
    Falls back to the hand-written list above.
    """
    if not normalized_tags_path.exists():
        return DOMAIN_VOCABULARY
    try:
        tags = json.loads(normalized_tags_path.read_text(encoding="utf-8"))
        canonical = list(tags.keys()) if isinstance(tags, dict) else list(tags)
        return sorted(set(DOMAIN_VOCABULARY) | set(canonical))
    except Exception:
        return DOMAIN_VOCABULARY


# ------------------------------------------------- source type detection ----
# `source_type` is provenance, and provenance changes how the Writer phrases
# things. The design doc puts it well: a reader can tell the difference between
# "as Vaswani et al. put it" and "as the lecture demonstrated on the whiteboard"
# -- but only if the Writer knows which is which.
SOURCE_TYPE_BY_EXT: Dict[str, str] = {
    ".pdf": "document", ".docx": "document", ".doc": "document",
    ".md": "text", ".txt": "text",
    ".pptx": "slides", ".ppt": "slides",
    ".xlsx": "spreadsheet", ".xls": "spreadsheet",
    ".png": "image", ".jpg": "image", ".jpeg": "image",
    ".gif": "image", ".bmp": "image", ".tiff": "image", ".tif": "image",
    ".webp": "image",
    ".mp4": "transcript", ".mkv": "transcript", ".mov": "transcript",
    ".avi": "transcript", ".webm": "transcript",
    ".mp3": "transcript", ".wav": "transcript", ".m4a": "transcript",
    ".flac": "transcript", ".ogg": "transcript",
}

MEDIA_EXTENSIONS = {e for e, t in SOURCE_TYPE_BY_EXT.items() if t == "transcript"}
DOCUMENT_EXTENSIONS = {e for e, t in SOURCE_TYPE_BY_EXT.items() if t != "transcript"}


def source_type_for(path: Path) -> str:
    return SOURCE_TYPE_BY_EXT.get(path.suffix.lower(), "unknown")


def doc_slug(name: str) -> str:
    """
    A short, stable, filesystem-safe id derived from a filename.

    Truncation alone is not enough. "...Advanced RAG (Part 10)..." and
    "...(Part 11)..." share their first 40 characters, so a truncated slug
    silently MERGES the two documents: the second one looks "already parsed",
    gets skipped, and simply never enters the corpus. A short hash of the FULL
    filename keeps every id distinct while staying readable and stable.
    """
    stem = Path(name).stem.lower()
    slug = re.sub(r"[^a-z0-9]+", "_", stem).strip("_")
    digest = hashlib.md5(name.encode("utf-8")).hexdigest()[:6]
    return f"{(slug[:33] or 'doc')}_{digest}"


# --------------------------------------------------- text normalisation -----
_SMART_QUOTES = {
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": " -- ", "…": "...", " ": " ",
    "ﬁ": "fi", "ﬂ": "fl",
}


def normalise_text(text: str) -> str:
    """
    Conservative typographic cleanup.

    Deliberately does NOT touch wording, line structure, or anything a language
    model would have an opinion about. Smart quotes and ligatures are the ones
    worth fixing here because they break naive string matching later -- the
    ledger's alias resolution in Pipeline C is exact string comparison.
    """
    for bad, good in _SMART_QUOTES.items():
        text = text.replace(bad, good)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{4,}", "\n\n\n", text)
    return text.strip()


# ------------------------------------------------------------ the model ----
# The design document (section 1, "The models") specifies TWO models and one of
# them does everything that is not speech:
#
#   | Speech to text                      | Whisper large-v3 |
#   | Everything else - figure description, tagging, planning, writing,
#     editing, cataloguing                | Qwen3.6-27B (multimodal, 256K)     |
#
# Using one language model for the whole pipeline is a deliberate choice:
#
#   "The model that describes a diagram during preprocessing is the same model
#    that later writes the chapter about it. That alone does more for
#    consistency than any amount of prompt tuning."
#
# So the id lives here, once, and every stage and every agent reads it. Change
# it in this cell and the whole system changes with it.
BOOK_MODEL = "Qwen/Qwen3.6-27B-Instruct"

# If BOOK_MODEL cannot be pulled on this machine, the loader steps down this
# ladder rather than dying -- but it logs a warning and every stage report
# prints the model it actually ran on, so a downgrade is never silent.
BOOK_MODEL_FALLBACKS = [
    "Qwen/Qwen3-VL-30B-A3B-Instruct",
    "Qwen/Qwen3-VL-8B-Instruct",
    "Qwen/Qwen2-VL-7B-Instruct",
]

WHISPER_MODEL = "large-v3"


# ------------------------------------------------------ token accounting ----
# Chunk sizes are specified in tokens, not characters, so we need a counter.
# tiktoken's cl100k_base is not Qwen's tokenizer, but it is close enough for
# deciding whether a chunk is 1,500 or 2,500 tokens, and it needs no GPU.
try:
    import tiktoken
    _ENCODING = tiktoken.get_encoding("cl100k_base")
except Exception:
    _ENCODING = None


def count_tokens(text: str) -> int:
    """Token count, falling back to the ~4-characters-per-token approximation."""
    if _ENCODING is not None:
        return len(_ENCODING.encode(text, disallowed_special=()))
    return max(1, len(text) // 4)


print("Stage 0 ready.")
print(f"  Corpus root      : {PATHS.root.resolve()}")
print(f"  Drop sources in  : {PATHS.raw_sources.resolve()}")
print(f"  Vocabulary terms : {len(DOMAIN_VOCABULARY)}")
print(f"  Model (design)   : {BOOK_MODEL}")
print(f"  Token counter    : {'tiktoken cl100k_base' if _ENCODING else 'approximate (len/4)'}")

## Stage 1 — Speech to Text (Whisper large-v3)

This stage did not exist in the previous version of the notebook, which is a strange omission for a
book built partly from recorded lectures. The design document is blunt about why it matters:

> *"Recorded talks are the highest-value and lowest-quality input in a technical corpus. A lecture
> contains explanations that exist nowhere in the slides."*

Three settings carry almost all of the quality, and each one is worth understanding rather than
copying.

**`initial_prompt=", ".join(vocab)`** — Whisper conditions its decoding on whatever you put here.
It is not an instruction; it is a prior. Seeding it with `KV cache`, `CUDA`, `PyTorch` makes those
spellings *available* to the decoder, and the mangled versions much less likely. This is the
cheapest quality win in the whole pipeline.

**`condition_on_previous_text=False`** — by default Whisper feeds each window its own previous
output as context. On hour-long recordings this occasionally falls into a repetition loop, emitting
the same sentence for two minutes straight. Turning it off costs a little cross-sentence coherence
and removes the failure that actually ruins a transcript.

**`word_timestamps=True`** — not for the reader. For provenance. A chunk that came from a lecture
carries its timestamp, so the book can say *"as covered around the 34-minute mark"*, and so you can
go and verify a claim in seconds rather than re-watching an hour of video.

And one rule that is easy to skip and expensive to skip:

> **Keep the timestamped segments, not just the flat text.** Segment boundaries are the only
> structural signal a transcript has. Throw them away and the chunker has to guess where a topic
> ended.

So Stage 1 writes `<name>.raw.json` containing the segments, and *derives* the flat text from them
later — never the other way round.

In [ ]:
# ============================================================================
# Stage 1 — Configuration
# ============================================================================

log1 = make_logger("stage1.speech", "stage1_speech.log")


@dataclass
class Stage1Config:
    """Configuration for Stage 1 - Speech to Text."""

    # Model
    model_size: str = "large-v3"
    device: str = "auto"              # "auto" | "cuda" | "cpu"
    compute_type: str = "float16"     # "float16" on GPU, "int8" on CPU

    # Decoding -- see the markdown above for why each of these is set this way
    vad_filter: bool = True                   # drop silence; large speedup on lectures
    word_timestamps: bool = True              # provenance, not readability
    condition_on_previous_text: bool = False  # stops repetition loops on long files
    beam_size: int = 5
    language: Optional[str] = None            # None = autodetect

    # Audio extraction
    sample_rate: int = 16_000
    keep_extracted_wav: bool = False          # WAVs are large; delete once transcribed

    # Resume
    skip_existing: bool = True


stage1_cfg = Stage1Config()

log1.info("Stage 1 configuration:")
log1.info(f"  model         : whisper {stage1_cfg.model_size}")
log1.info(f"  vad_filter    : {stage1_cfg.vad_filter}")
log1.info(f"  word_stamps   : {stage1_cfg.word_timestamps}")
log1.info(f"  cond_on_prev  : {stage1_cfg.condition_on_previous_text}")

In [ ]:
# ============================================================================
# Stage 1 — Audio extraction and transcription
# ============================================================================

def extract_audio(media_path: Path, out_dir: Path, sample_rate: int = 16_000) -> Path:
    """
    Pull a mono 16 kHz WAV out of any media container using ffmpeg.

    Whisper resamples internally anyway, but doing it once here means we do not
    pay for it on every retry, and it makes an MP4 and an MP3 look identical to
    everything downstream.

        -ac 1     mono         (Whisper is mono; stereo just doubles the bytes)
        -ar 16000 16 kHz       (Whisper's native rate)
        -vn       no video     (we only want the audio stream)
        -y        overwrite
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_path = out_dir / f"{media_path.stem}.wav"

    if wav_path.exists():
        return wav_path

    cmd = ["ffmpeg", "-y", "-i", str(media_path),
           "-ac", "1", "-ar", str(sample_rate), "-vn", str(wav_path)]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        # ffmpeg writes everything to stderr, including normal progress output,
        # so only the tail is worth showing.
        raise RuntimeError(f"ffmpeg failed for {media_path.name}:\n{proc.stderr[-800:]}")

    return wav_path


_whisper_model = None   # loaded lazily; the model is ~3 GB


def get_whisper_model(cfg: Stage1Config):
    """Load Whisper once and reuse it. Returns None if faster-whisper is absent."""
    global _whisper_model
    if _whisper_model is not None:
        return _whisper_model

    try:
        from faster_whisper import WhisperModel
    except ImportError:
        log1.warning("faster-whisper is not installed - Stage 1 will be skipped.")
        return None

    device = cfg.device
    compute_type = cfg.compute_type
    if device == "auto":
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            device = "cpu"
    if device == "cpu" and compute_type == "float16":
        # float16 on CPU is either unsupported or catastrophically slow.
        compute_type = "int8"

    log1.info(f"Loading Whisper {cfg.model_size} on {device} ({compute_type})...")
    t0 = time.time()
    _whisper_model = WhisperModel(cfg.model_size, device=device, compute_type=compute_type)
    log1.info(f"Whisper loaded in {time.time() - t0:.1f}s")
    return _whisper_model


def transcribe_media(media_path: Path, cfg: Stage1Config,
                     vocabulary: List[str]) -> Dict[str, Any]:
    """
    Transcribe one audio or video file into the structure Pipeline A expects.

    Returns a dict with the *segments preserved*. The flat text is derived from
    them, never stored instead of them.
    """
    model = get_whisper_model(cfg)
    if model is None:
        raise RuntimeError("Whisper model unavailable")

    wav = extract_audio(media_path, PATHS.transcripts / "_wav", cfg.sample_rate)

    log1.info(f"Transcribing {media_path.name} ...")
    t0 = time.time()

    segments_iter, info = model.transcribe(
        str(wav),
        vad_filter=cfg.vad_filter,
        word_timestamps=cfg.word_timestamps,
        beam_size=cfg.beam_size,
        language=cfg.language,
        initial_prompt=", ".join(vocabulary),      # <-- the vocabulary prior
        condition_on_previous_text=cfg.condition_on_previous_text,
    )

    # faster-whisper streams segments lazily; iterating is what does the work.
    segments = []
    for s in segments_iter:
        segments.append({
            "id": len(segments),
            "start": round(float(s.start), 2),
            "end": round(float(s.end), 2),
            "text": s.text.strip(),
        })

    elapsed = time.time() - t0
    duration = float(getattr(info, "duration", 0.0) or 0.0)

    if not cfg.keep_extracted_wav:
        wav.unlink(missing_ok=True)

    result = {
        "source_document": media_path.name,
        "source_type": "transcript",
        "language": getattr(info, "language", None),
        "language_probability": round(float(getattr(info, "language_probability", 0.0) or 0.0), 3),
        "duration_seconds": round(duration, 2),
        "transcribe_seconds": round(elapsed, 2),
        "model": f"whisper-{cfg.model_size}",
        "vocabulary_terms": len(vocabulary),
        "segments": segments,
    }

    log1.info(f"  {len(segments)} segments | {duration/60:.1f} min audio "
              f"| {elapsed/60:.1f} min compute | lang={result['language']}")
    return result


def segments_to_text(segments: List[Dict[str, Any]],
                     paragraph_gap_seconds: float = 2.0) -> Tuple[str, List[Dict]]:
    """
    Flatten segments into readable text AND build a character -> timestamp index.

    The index is what lets a chunk 40,000 characters into the corpus report
    "this came from lecture_04.mp4 at 34:12". Without it, timestamps are a
    number in a JSON file that nothing can ever use.

    A pause longer than `paragraph_gap_seconds` is treated as a paragraph break.
    It is a crude heuristic, but it is the only structural signal speech has,
    and it beats one 60,000-character wall of text.
    """
    parts: List[str] = []
    index: List[Dict[str, Any]] = []
    cursor = 0
    prev_end = None

    for seg in segments:
        text = seg["text"].strip()
        if not text:
            continue

        if prev_end is not None:
            sep = "\n\n" if (seg["start"] - prev_end) > paragraph_gap_seconds else " "
            parts.append(sep)
            cursor += len(sep)

        index.append({
            "char_start": cursor,
            "char_end": cursor + len(text),
            "t_start": seg["start"],
            "t_end": seg["end"],
        })
        parts.append(text)
        cursor += len(text)
        prev_end = seg["end"]

    return "".join(parts), index


log1.info("Stage 1 functions ready.")

In [ ]:
# ============================================================================
# Stage 1 — Run
# ============================================================================

def run_stage1_speech(cfg: Stage1Config) -> Dict[str, Any]:
    """
    Transcribe every audio/video file in data/raw_sources/.

    Skips cleanly (rather than crashing) when there is no media, or when
    faster-whisper / ffmpeg are not installed -- so the rest of the notebook
    still runs on a document-only corpus.
    """
    log1.info("=" * 70)
    log1.info("STAGE 1: SPEECH TO TEXT")
    log1.info("=" * 70)

    media_files = sorted(
        p for p in PATHS.raw_sources.rglob("*")
        if p.is_file() and p.suffix.lower() in MEDIA_EXTENSIONS
    )

    summary: Dict[str, Any] = {
        "media_found": len(media_files),
        "transcribed": [], "skipped": [], "failed": [],
        "total_audio_minutes": 0.0,
    }

    if not media_files:
        log1.info("No audio or video files found - nothing to do.")
        log1.info("(This is fine. Stage 1 only runs if your corpus has recordings.)")
        return summary

    vocabulary = load_vocabulary_from_pipeline_b(PATHS.root / "normalized_tags.json")
    log1.info(f"Priming Whisper with {len(vocabulary)} domain terms")

    if get_whisper_model(cfg) is None:
        log1.warning("Skipping Stage 1 - install faster-whisper to enable it.")
        summary["skipped"] = [p.name for p in media_files]
        return summary

    for media in tqdm(media_files, desc="Transcribing"):
        out_json = PATHS.transcripts / f"{media.stem}.raw.json"

        if cfg.skip_existing and out_json.exists():
            log1.info(f"Skipping (already transcribed): {media.name}")
            summary["skipped"].append(media.name)
            continue

        try:
            result = transcribe_media(media, cfg, vocabulary)
            out_json.write_text(json.dumps(result, indent=2, ensure_ascii=False),
                                encoding="utf-8")

            # Also write the flat text now, so a run can be inspected without
            # waiting for Stage 3. Stage 3 will overwrite it with a cleaned
            # version, and keep this one alongside.
            flat, _ = segments_to_text(result["segments"])
            (PATHS.transcripts / f"{media.stem}.raw.txt").write_text(
                normalise_text(flat), encoding="utf-8")

            summary["transcribed"].append(media.name)
            summary["total_audio_minutes"] += result["duration_seconds"] / 60.0

        except Exception as exc:
            log1.error(f"Failed on {media.name}: {exc}")
            log1.debug(traceback.format_exc())
            summary["failed"].append({"file": media.name, "error": str(exc)})

    log1.info("-" * 70)
    log1.info(f"Transcribed : {len(summary['transcribed'])}")
    log1.info(f"Skipped     : {len(summary['skipped'])}")
    log1.info(f"Failed      : {len(summary['failed'])}")
    log1.info(f"Audio       : {summary['total_audio_minutes']:.1f} minutes")
    return summary


stage1_summary = run_stage1_speech(stage1_cfg)

## Stage 2 — Documents to Structured JSON (MinerU / Docling)

PDFs, DOCX and PPTX go through a layout parser, which returns text blocks *in reading order*, plus
image, table and equation regions with their bounding boxes and cropped images.

The design document adds one instruction in bold, and it is the one the previous version missed:

> **Keep the extracted image crops on disk.** Stage 3 needs them, and so does the Writer later.

The previous code called:

```python
content_list = parser.parse_document(
    file_path=file_path,
    output_dir=None,          # <-- "We don't need parser's output dir"
    method=parse_method,
)
```

We very much do need the parser's output dir. It is where the image crops go. With `output_dir=None`
the parser writes them wherever it likes — typically next to the input, or into a temporary
directory — and the `img_path` values in the returned JSON are *relative to that location*. Stage 3
then does `Path(image_path).exists()` from a different working directory, quietly fails the check,
and substitutes `[IMAGE: Content processing unavailable]`. Nothing raises. You find out much later,
when the Writer has no diagrams.

So Stage 2 now gives every document its own parse directory, and then **resolves every `img_path`
to an absolute path that we have verified exists**.

### Three more fixes in this stage

**Retries that actually retry.** `parse_single_document` catches its own exceptions and *returns*
`{"status": "failed", ...}`. The old retry loop wrapped it in `try/except` — but a normal return
value is not an exception, so the loop always exited on the first attempt. Retry logic that never
fires is worse than no retry logic, because it reads like a safety net. The fix is to branch on the
returned status.

**Thread-local parsers.** The old code created one parser and handed the same object to four
worker threads. Layout parsers hold model state and temp-file state; sharing one across threads is
undefined behaviour. Each thread now builds its own.

**Parser API drift.** `parse_document` has returned three different shapes across raganything
versions: a bare `list`, a `(content_list, markdown)` tuple, and a `dict`. Rather than pinning a
version and hoping, we normalise all three.

In [ ]:
# ============================================================================
# Stage 2 — Configuration
# ============================================================================

log2 = make_logger("stage2.parse", "stage2_parsing.log")


@dataclass
class Stage2Config:
    """Configuration for Stage 2 - Layout parsing."""

    parser_type: str = "mineru"      # "mineru" | "docling"
    parse_method: str = "auto"       # "auto" | "txt" | "ocr"

    # Which files this stage handles. Media files belong to Stage 1, so they
    # are deliberately excluded here rather than being silently "unsupported".
    supported_extensions: List[str] = field(
        default_factory=lambda: sorted(DOCUMENT_EXTENSIONS))

    # MinerU runs models on the GPU. More than one or two workers will fight
    # over VRAM and be slower than sequential, not faster.
    max_workers: int = 2

    # Error handling -- the retry now keys on the RESULT, not on an exception
    max_retries: int = 3
    retry_delay: int = 5
    continue_on_error: bool = True

    # Resume
    skip_existing: bool = True
    force_reparse: bool = False

    validate_output: bool = True


stage2_cfg = Stage2Config()

log2.info("Stage 2 configuration:")
log2.info(f"  parser      : {stage2_cfg.parser_type} ({stage2_cfg.parse_method})")
log2.info(f"  workers     : {stage2_cfg.max_workers}")
log2.info(f"  extensions  : {len(stage2_cfg.supported_extensions)} document types")

In [ ]:
# ============================================================================
# Stage 2 — Discovery, parsing, and image-path resolution
# ============================================================================

def discover_documents(cfg: Stage2Config) -> Tuple[List[Path], List[Path]]:
    """Split raw_sources into (documents for this stage, files handled elsewhere)."""
    if not PATHS.raw_sources.exists():
        raise FileNotFoundError(f"Put your sources in {PATHS.raw_sources.resolve()}")

    supported = {e.lower() for e in cfg.supported_extensions}
    documents, other = [], []

    for p in sorted(PATHS.raw_sources.rglob("*")):
        if not p.is_file() or p.name.startswith("."):
            continue
        (documents if p.suffix.lower() in supported else other).append(p)

    log2.info(f"Found {len(documents)} documents, {len(other)} other files")
    if other:
        by_ext: Dict[str, int] = {}
        for p in other:
            by_ext[p.suffix.lower()] = by_ext.get(p.suffix.lower(), 0) + 1
        for ext, n in sorted(by_ext.items(), key=lambda kv: -kv[1]):
            kind = SOURCE_TYPE_BY_EXT.get(ext, "unsupported")
            log2.info(f"  {ext or '(no ext)'}: {n} file(s) -> {kind}")

    return documents, other


# --- parser construction -----------------------------------------------------
# One parser per thread. `threading.local()` gives each worker its own slot in
# a shared object, so no locking is needed and no state is shared.
_parser_local = threading.local()


def get_parser(cfg: Stage2Config):
    """Return this thread's parser, constructing it on first use."""
    existing = getattr(_parser_local, "parser", None)
    if existing is not None:
        return existing

    from raganything.parser import MineruParser, DoclingParser

    kind = cfg.parser_type.lower()
    if kind == "mineru":
        parser = MineruParser()
    elif kind == "docling":
        parser = DoclingParser()
    else:
        raise ValueError(f"Unsupported parser type: {cfg.parser_type}")

    # check_installation() is advisory: some builds report False and still work.
    check = getattr(parser, "check_installation", None)
    if callable(check):
        try:
            if not check():
                log2.warning(f"{kind}: installation check returned False - proceeding anyway")
        except Exception as exc:
            log2.warning(f"{kind}: installation check raised ({exc}) - proceeding anyway")

    _parser_local.parser = parser
    return parser


def _normalise_parser_return(raw: Any) -> List[Dict[str, Any]]:
    """
    Accept every shape `parse_document` has returned across versions.

    list                      -> the content list itself
    (content_list, markdown)  -> element 0
    {"content_list": [...]}   -> that key
    """
    if isinstance(raw, tuple) and raw:
        raw = raw[0]
    if isinstance(raw, dict):
        raw = raw.get("content_list", raw.get("content", []))
    if not isinstance(raw, list):
        raise TypeError(f"Parser returned an unusable type: {type(raw).__name__}")
    return raw


def _resolve_image_paths(content_list: List[Dict[str, Any]], parse_dir: Path) -> int:
    """
    Turn every relative `img_path` into a verified absolute path.

    This is the fix for the silent-fallback bug. Parsers emit paths relative to
    their own output directory, and different versions nest them differently
    (`./images/x.jpg`, `auto/images/x.jpg`, ...). We try the likely locations,
    then fall back to a recursive search by filename.

    Items whose crop genuinely cannot be found are marked `img_missing` so
    Stage 3 can report them as a number instead of silently degrading.
    """
    resolved = 0
    for item in content_list:
        rel = item.get("img_path")
        if not rel:
            continue

        candidate = Path(rel)
        if candidate.is_absolute() and candidate.exists():
            resolved += 1
            continue

        found = None
        for base in (parse_dir, parse_dir / "auto", parse_dir / "images",
                     parse_dir / "auto" / "images"):
            trial = base / rel
            if trial.exists():
                found = trial
                break

        if found is None:
            matches = list(parse_dir.rglob(Path(rel).name))
            found = matches[0] if matches else None

        if found is not None:
            item["img_path"] = str(found.resolve())
            resolved += 1
        else:
            item["img_missing"] = True

    return resolved


def parse_single_document(file_path: Path, cfg: Stage2Config) -> Dict[str, Any]:
    """
    Parse one document into structured JSON, keeping its image crops on disk.

    Never raises: returns a result dict whose "status" the caller inspects.
    That is a deliberate contract, and the retry loop below honours it.
    """
    t0 = time.time()
    parse_dir = PATHS.parsed / doc_slug(file_path.name)
    parse_dir.mkdir(parents=True, exist_ok=True)

    try:
        parser = get_parser(cfg)

        raw = parser.parse_document(
            file_path=str(file_path),
            output_dir=str(parse_dir),      # <-- the fix: crops land here and stay
            method=cfg.parse_method,
        )
        content_list = _normalise_parser_return(raw)
        images_resolved = _resolve_image_paths(content_list, parse_dir)

        content_types: Dict[str, int] = {}
        for item in content_list:
            t = item.get("type", "unknown")
            content_types[t] = content_types.get(t, 0) + 1

        elapsed = time.time() - t0
        result = {
            "status": "success",
            "file_path": str(file_path),
            "filename": file_path.name,
            "doc_slug": doc_slug(file_path.name),
            "source_type": source_type_for(file_path),      # <-- provenance
            "parse_dir": str(parse_dir),
            "content_list": content_list,
            "metadata": {
                "total_items": len(content_list),
                "content_types": content_types,
                "images_resolved": images_resolved,
                "images_missing": sum(1 for i in content_list if i.get("img_missing")),
                "parse_time": round(elapsed, 2),
                "parse_method": cfg.parse_method,
                "parser_type": cfg.parser_type,
                "parsed_at": datetime.now().isoformat(timespec="seconds"),
                "file_size": file_path.stat().st_size,
            },
        }

        log2.info(f"  {file_path.name}: {len(content_list)} items in {elapsed:.1f}s "
                  f"| {content_types}")
        if result["metadata"]["images_missing"]:
            log2.warning(f"  {file_path.name}: "
                         f"{result['metadata']['images_missing']} image crops not found")
        return result

    except Exception as exc:
        log2.error(f"  {file_path.name}: {type(exc).__name__}: {exc}")
        log2.debug(traceback.format_exc())
        return {
            "status": "failed",
            "file_path": str(file_path),
            "filename": file_path.name,
            "doc_slug": doc_slug(file_path.name),
            "source_type": source_type_for(file_path),
            "content_list": [],
            "metadata": {
                "error": str(exc),
                "error_type": type(exc).__name__,
                "parse_time": round(time.time() - t0, 2),
            },
        }


def validate_parsed_output(result: Dict[str, Any]) -> Tuple[bool, Optional[str]]:
    """Structural sanity check before we trust a parse."""
    for key in ("status", "filename", "content_list", "metadata"):
        if key not in result:
            return False, f"missing key: {key}"

    if not isinstance(result["content_list"], list):
        return False, "content_list is not a list"

    if result["status"] == "success" and not result["content_list"]:
        return False, "parse succeeded but produced zero content items"

    for i, item in enumerate(result["content_list"]):
        if not isinstance(item, dict):
            return False, f"item {i} is not a dict"
        if "type" not in item:
            return False, f"item {i} has no 'type'"

    return True, None


log2.info("Stage 2 parsing functions ready.")

In [ ]:
# ============================================================================
# Stage 2 — Batch processing with retries that actually retry
# ============================================================================

def parsed_json_path(file_path: Path) -> Path:
    return PATHS.parsed / f"{doc_slug(file_path.name)}.json"


def process_document_with_retry(file_path: Path, cfg: Stage2Config) -> Dict[str, Any]:
    """
    Parse one document, retrying on failure.

    THE FIX: `parse_single_document` reports failure by RETURNING a dict, not by
    raising. The previous version wrapped it in try/except and therefore never
    saw a failure, so `max_retries=3` was decorative. We branch on the status.
    """
    out_path = parsed_json_path(file_path)
    last_error = "unknown"

    for attempt in range(1, cfg.max_retries + 1):
        result = parse_single_document(file_path, cfg)

        if result["status"] == "success" and cfg.validate_output:
            ok, msg = validate_parsed_output(result)
            if not ok:
                result["status"] = "failed"
                result["metadata"]["validation_error"] = msg
                log2.error(f"  {file_path.name}: validation failed - {msg}")

        if result["status"] == "success":
            try:
                out_path.write_text(json.dumps(result, indent=2, ensure_ascii=False),
                                    encoding="utf-8")
                result["output_path"] = str(out_path)
                result["metadata"]["attempts"] = attempt
                return result
            except Exception as exc:
                last_error = f"could not save result: {exc}"
                log2.error(f"  {file_path.name}: {last_error}")
        else:
            last_error = result["metadata"].get("error", "parse failed")

        if attempt < cfg.max_retries:
            log2.warning(f"  {file_path.name}: attempt {attempt}/{cfg.max_retries} "
                         f"failed ({last_error}); retrying in {cfg.retry_delay}s")
            time.sleep(cfg.retry_delay)

    log2.error(f"  {file_path.name}: all {cfg.max_retries} attempts failed")
    return {
        "status": "failed", "filename": file_path.name,
        "file_path": str(file_path), "content_list": [],
        "metadata": {"error": last_error, "attempts": cfg.max_retries},
    }


@dataclass
class Stage2Checkpoint:
    """
    Resumable state.

    Stored as sorted lists (JSON has no set type) but manipulated as sets, so a
    resumed run cannot append the same filename a second time. The previous
    version used plain lists and double-counted every skipped file on resume.
    """
    processed: List[str] = field(default_factory=list)
    failed: List[str] = field(default_factory=list)
    skipped: List[str] = field(default_factory=list)
    started_at: str = ""
    updated_at: str = ""

    def merge(self, bucket: str, name: str) -> None:
        current = set(getattr(self, bucket))
        current.add(name)
        setattr(self, bucket, sorted(current))

    def save(self) -> None:
        self.updated_at = datetime.now().isoformat(timespec="seconds")
        (PATHS.checkpoints / "stage2.json").write_text(
            json.dumps(asdict(self), indent=2), encoding="utf-8")

    @classmethod
    def load(cls) -> "Stage2Checkpoint":
        p = PATHS.checkpoints / "stage2.json"
        if not p.exists():
            return cls(started_at=datetime.now().isoformat(timespec="seconds"))
        try:
            return cls(**json.loads(p.read_text(encoding="utf-8")))
        except Exception as exc:
            log2.warning(f"Unreadable checkpoint ({exc}); starting fresh")
            return cls(started_at=datetime.now().isoformat(timespec="seconds"))


def run_stage2_parsing(cfg: Stage2Config) -> Dict[str, Any]:
    """Parse every document in raw_sources into structured JSON."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    log2.info("=" * 70)
    log2.info("STAGE 2: LAYOUT PARSING")
    log2.info("=" * 70)

    documents, _ = discover_documents(cfg)
    if not documents:
        log2.warning("No documents to parse.")
        return {"error": "no documents found"}

    ckpt = Stage2Checkpoint.load()

    todo: List[Path] = []
    for doc in documents:
        already = parsed_json_path(doc).exists()
        if cfg.skip_existing and not cfg.force_reparse and already:
            ckpt.merge("skipped", doc.name)
        else:
            todo.append(doc)

    log2.info(f"To parse: {len(todo)} | already parsed: {len(ckpt.skipped)}")
    if not todo:
        ckpt.save()
        return {"total": len(documents), "processed": 0,
                "skipped": len(ckpt.skipped), "failed": 0}

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=cfg.max_workers) as pool:
        futures = {pool.submit(process_document_with_retry, d, cfg): d for d in todo}

        with tqdm(total=len(todo), desc="Parsing documents") as bar:
            for fut in as_completed(futures):
                doc = futures[fut]
                try:
                    result = fut.result()
                    bucket = "processed" if result["status"] == "success" else "failed"
                except Exception as exc:
                    log2.error(f"Worker crashed on {doc.name}: {exc}")
                    bucket = "failed"

                ckpt.merge(bucket, doc.name)
                bar.set_postfix_str(f"{'ok' if bucket == 'processed' else 'FAIL'} {doc.name[:28]}")
                bar.update(1)
                ckpt.save()

                if bucket == "failed" and not cfg.continue_on_error:
                    raise RuntimeError(f"Parsing failed for {doc.name}")

    summary = {
        "total": len(documents),
        "processed": len(ckpt.processed),
        "skipped": len(ckpt.skipped),
        "failed": len(ckpt.failed),
        "failed_files": ckpt.failed,
        "elapsed_minutes": round((time.time() - t0) / 60, 2),
    }

    log2.info("-" * 70)
    for k in ("total", "processed", "skipped", "failed"):
        log2.info(f"  {k:10s}: {summary[k]}")
    log2.info(f"  elapsed   : {summary['elapsed_minutes']} min")
    if ckpt.failed:
        log2.warning(f"  failures  : {', '.join(ckpt.failed[:5])}")

    return summary


stage2_summary = run_stage2_parsing(stage2_cfg)

In [ ]:
# ============================================================================
# Stage 2 — What did we actually get?
# ============================================================================
"""
Read this output before moving on. Two numbers matter most:

  1. `images_resolved` vs `images_missing`. If crops are missing, Stage 3 will
     produce a corpus with no diagrams in it and will not complain.

  2. The `discarded` count. MinerU tags page headers, footers, page numbers and
     other layout furniture as `discarded`. In the previous run of this notebook
     that was 678 of 1,409 items -- 48% -- and every one of them was written into
     the corpus as `[UNKNOWN CONTENT TYPE: discarded]`. Stage 3 now drops them,
     but it is worth knowing how much of your corpus is furniture.
"""

def analyse_parsed_outputs() -> Dict[str, Any]:
    files = sorted(PATHS.parsed.glob("*.json"))

    stats: Dict[str, Any] = {
        "documents": 0, "items": 0,
        "types": {}, "by_source_type": {},
        "images_resolved": 0, "images_missing": 0,
        "largest": [],
    }

    for f in files:
        try:
            data = json.loads(f.read_text(encoding="utf-8"))
        except Exception:
            continue
        if data.get("status") != "success":
            continue

        stats["documents"] += 1
        items = data["content_list"]
        stats["items"] += len(items)
        stats["largest"].append((data["filename"], len(items)))

        st = data.get("source_type", "unknown")
        stats["by_source_type"][st] = stats["by_source_type"].get(st, 0) + 1

        md = data.get("metadata", {})
        stats["images_resolved"] += md.get("images_resolved", 0)
        stats["images_missing"] += md.get("images_missing", 0)

        for t, n in md.get("content_types", {}).items():
            stats["types"][t] = stats["types"].get(t, 0) + n

    stats["largest"].sort(key=lambda kv: -kv[1])
    return stats


_s = analyse_parsed_outputs()

print("=" * 70)
print("STAGE 2 OUTPUT")
print("=" * 70)
print(f"Documents parsed : {_s['documents']}")
print(f"Content items    : {_s['items']:,}")
print(f"By source type   : {_s['by_source_type']}")

print("\nContent types (what Stage 3 will have to deal with):")
for t, n in sorted(_s["types"].items(), key=lambda kv: -kv[1]):
    share = 100 * n / _s["items"] if _s["items"] else 0
    note = ""
    if t == "discarded":
        note = "   <-- layout furniture; Stage 3 drops these"
    elif t == "image":
        note = "   <-- each of these is a potential VLM call"
    print(f"  {t:12s} {n:6,}  ({share:4.1f}%){note}")

print(f"\nImage crops resolved : {_s['images_resolved']}")
print(f"Image crops MISSING  : {_s['images_missing']}"
      f"{'   <-- investigate before running Stage 3' if _s['images_missing'] else ''}")

print("\nLargest documents:")
for name, n in _s["largest"][:5]:
    print(f"  {n:6,} items   {name}")
print("=" * 70)

## Stage 3 — Describing Figures, In Context

This is where most of the corpus quality is won or lost. The design document asks for four specific
things here, and the previous version of this notebook did none of them.

Here is a real description the old code wrote into the corpus, verbatim:

> `[TABLE]`
> *"The table provided is quite simple and contains only two columns: 'Type your email' and
> 'Subscribe'. The 'Type your email' column is a text input field where users can enter their email
> address…"*

That is a newsletter signup widget on a Medium page. It cost roughly 16 seconds of GPU time and
about 400 tokens, and those tokens became permanent source material for the book.

---

### The design's two headline requirements

**1. Describe the figure with the page around it.**

> *"A model looking at a cropped architecture diagram in isolation produces 'a flowchart with boxes
> and arrows.' The same figure, passed together with the two thousand words of text surrounding it,
> produces 'the ReAct control loop from §3.2 — the observation feeds back into the reasoner rather
> than into the tool dispatcher, which is the distinction the surrounding text is making.'"*

**2. Describe the whole document in one call.**

> *"With a 256K window you can pass an entire document and all its figures at once, rather than
> making one isolated call per image. Fewer calls, better descriptions, and the model can notice
> when figure 4 is a refinement of figure 2."*

The second one is the important one, and it changes the shape of the code. Instead of a loop that
calls the model once per image, we build **one document view** — the full text with every figure
replaced by its id — and hand that to the model together with the images:

```
FULL DOCUMENT TEXT (figures marked by id):
The encoder is built from six identical layers...
<<fig_paper_a_001>>
...each with a residual connection around it, as shown above.
<<fig_paper_a_002>>
...which refines the previous diagram by adding the normalisation step.

[image]  ↑ figure id: fig_paper_a_001
[image]  ↑ figure id: fig_paper_a_002
```

The model answers with one structured object naming every figure it saw. Because it read the whole
document, it can write *"refines fig_paper_a_001 by adding the normalisation step"* — a sentence no
per-image call could produce.

### `generate_structured` — the missing capability

The design's `describe_document` calls `llm.generate_structured(content, system=..., schema=...)`.
Nothing like that existed in the notebook. It is added here: it appends the schema to the prompt,
extracts the first balanced JSON object from the reply, and on a parse failure **re-prompts with
the parser's own error message** rather than throwing the generation away. Every agent in Pipelines
B and C needs this, so it is worth building once and properly.

### `kind: "decorative"` is part of the schema

```python
schema={"figures": [{"id": "string", "description": "string",
                     "kind": "figure|table|equation|decorative"}]}
```

The design's system prompt ends with *"If a figure is decorative or unreadable, say so and write
nothing further."* So `decorative` is not an error path — it is a legitimate answer. A figure marked
decorative is **recorded in the figure store with `kind: "decorative"` and no description**, and
never enters the corpus text. It stays in the store so the count is auditable rather than invisible.

Before that, a cheaper filter: an avatar is 48×48 and a follow button is 200×40. Anything below a
size threshold never reaches the vision encoder. And because a blog post repeats the same author
photo on every section, identical crops are hashed and described once.

**What this stage runs on.** The design specifies one model for the entire system — *"the model
that describes a diagram during preprocessing is the same model that later writes the chapter about
it"* — so the model id lives in a single constant, `BOOK_MODEL`, defined back in Stage 0 and shared
by every stage that follows.

In [ ]:
# ============================================================================
# Stage 3 — Configuration
# ============================================================================

import torch
from PIL import Image

log3 = make_logger("stage3.describe", "stage3_describe.log")


@dataclass
class Stage3Config:
    """Configuration for Stage 3 - Describing figures in context."""

    # -- Model ---------------------------------------------------------------
    # One model for the whole system (design doc, section 1). Defined once in
    # Stage 0 so Pipelines B and C use the identical checkpoint.
    model_name: str = BOOK_MODEL
    dtype: str = "bfloat16"
    # False keeps the stack pure transformers + PyTorch: the loader ladder is
    # then sdpa -> eager, both native torch attention. Flip on only if the
    # optional flash-attn package is installed and you want the speedup.
    use_flash_attention: bool = False
    max_image_side: int = 1280            # downscale before the vision encoder

    # -- Whole-document description (the design's primary path) --------------
    describe_whole_document: bool = True
    max_figures_per_call: int = 8         # images per call; lower this if VRAM is tight
    max_document_chars: int = 60_000      # how much marker text travels with each call
    document_call_max_tokens: int = 3_000 # the structured reply can be long

    # -- Per-figure fallback (used when a document-level id comes back empty) -
    context_window_items: int = 6         # text items either side of the figure
    context_max_chars: int = 2_000
    figure_max_tokens: int = 400
    temperature: float = 0.2              # descriptions should be boring and stable

    # -- Structured output ---------------------------------------------------
    structured_max_attempts: int = 3      # re-prompt with the parser's own error

    # -- The cheap decorative pre-filter -------------------------------------
    min_image_width: int = 96
    min_image_height: int = 96
    min_image_pixels: int = 20_000
    max_aspect_ratio: float = 12.0        # a 600x20 strip is a divider, not a figure

    # -- Item types ----------------------------------------------------------
    drop_types: Tuple[str, ...] = ("discarded",)

    # -- Transcript cleanup --------------------------------------------------
    clean_transcripts: bool = True
    transcript_window_chars: int = 4_000
    transcript_length_tolerance: float = 0.25

    # -- Housekeeping --------------------------------------------------------
    skip_existing: bool = True
    force_reconvert: bool = False
    continue_on_error: bool = True
    log_vlm_responses: bool = False


stage3_cfg = Stage3Config()

log3.info("Stage 3 configuration:")
log3.info(f"  model            : {stage3_cfg.model_name}")
log3.info(f"  whole-document   : {stage3_cfg.describe_whole_document} "
          f"({stage3_cfg.max_figures_per_call} figures per call)")
log3.info(f"  dropping         : {stage3_cfg.drop_types}")

if torch.cuda.is_available():
    log3.info(f"  GPU              : {torch.cuda.get_device_name(0)} "
              f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    log3.warning("  No GPU detected - Stage 3 on CPU is impractically slow.")

In [ ]:
# ============================================================================
# Stage 3 — The model wrapper, with structured generation
# ============================================================================

def extract_json_object(raw: str) -> Tuple[Optional[Any], Optional[str]]:
    """
    Pull the first balanced JSON object out of a model reply.

    Models wrap JSON in prose, in ``` fences, or both. Naive `json.loads` fails
    on all of it, and a regex for `\\{.*\\}` breaks on nested braces and on any
    brace inside a string. So we scan for the first '{' and walk forward
    counting depth, skipping over string literals and their escapes.

    Returns (object, None) or (None, reason). The reason is fed back to the
    model verbatim on the retry -- an error message is a far better repair
    instruction than "try again".
    """
    text = raw.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    start = text.find("{")
    if start == -1:
        return None, "the reply contained no '{'"

    depth, in_string, escaped = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1]), None
                    except json.JSONDecodeError as exc:
                        return None, f"JSON was malformed: {exc.msg} at char {exc.pos}"
    return None, "braces never balanced - the reply was probably truncated"


class BookModel:
    """
    The single multimodal model the whole system uses.

    Two capabilities:
      * `generate()`          -- free text
      * `generate_structured()` -- a JSON object matching a schema, with repair

    Two things here were real bugs in the previous version rather than style
    preferences:

    1. ATTENTION BACKEND. The old code wrote:

           try:
               model_kwargs["attn_implementation"] = "flash_attention_2"
           except:
               logger.warning("flash_attention_2 not available")

       The `try` wraps a dictionary assignment, which cannot fail. The real
       failure happens later inside `from_pretrained`, where nothing catches
       it -- so on a machine without flash-attn the stage died with its own
       fallback sitting unused. Backends are now tried for real.

    2. VISION INPUTS. The old code called `apply_chat_template(tokenize=True)`
       and hoped the processor would locate and load the images. We use the
       documented two-step path: render the template to text, load the images
       ourselves, hand both to the processor. We need the PIL objects anyway,
       for the size filter and the content hash.
    """

    def __init__(self, cfg: Stage3Config):
        self.cfg = cfg
        self.model = None
        self.processor = None
        self.model_id = None
        self.calls = 0
        self.structured_repairs = 0
        self.structured_failures = 0
        self._load()

    # ---------------------------------------------------------------- load --
    def _load(self) -> None:
        from transformers import AutoProcessor
        try:                        # transformers >= 4.45
            from transformers import AutoModelForImageTextToText as AutoVLM
        except ImportError:
            from transformers import AutoModelForVision2Seq as AutoVLM

        dtype = {"bfloat16": torch.bfloat16,
                 "float16": torch.float16}.get(self.cfg.dtype, torch.float32)

        # transformers renamed `torch_dtype` to `dtype` in 4.56. Getting this
        # wrong is not an error: the unknown keyword is absorbed into **kwargs
        # and the model silently loads in float32 -- twice the VRAM, half the
        # speed, no warning. So pick the spelling this version understands.
        import transformers
        version = tuple(int(x) for x in transformers.__version__.split(".")[:2])
        dtype_kwarg = "dtype" if version >= (4, 56) else "torch_dtype"
        log3.info(f"transformers {transformers.__version__} -> using {dtype_kwarg}=")

        backends = (["flash_attention_2"] if self.cfg.use_flash_attention else []) \
                   + ["sdpa", "eager"]

        # The design names one model. If this machine cannot pull it, we say so
        # loudly and step down the ladder rather than dying -- but the model we
        # actually loaded is printed in every stage report, so a downgrade can
        # never pass unnoticed.
        candidates = [self.cfg.model_name] + [
            m for m in BOOK_MODEL_FALLBACKS if m != self.cfg.model_name]

        t0 = time.time()
        last_error = None
        for model_id in candidates:
            for backend in backends:
                try:
                    log3.info(f"Loading {model_id} (attn={backend}) ...")
                    self.model = AutoVLM.from_pretrained(
                        model_id, device_map="auto",
                        attn_implementation=backend, **{dtype_kwarg: dtype})
                    self.processor = AutoProcessor.from_pretrained(model_id)
                    self.model_id = model_id
                    break
                except Exception as exc:
                    last_error = exc
                    log3.warning(f"  {model_id} / attn={backend}: "
                                 f"{type(exc).__name__}: {str(exc)[:150]}")
            if self.model is not None:
                break
            log3.warning(f"Could not load {model_id}; trying the next candidate")

        if self.model is None:
            raise RuntimeError(
                f"No usable model. The design specifies {BOOK_MODEL}; none of "
                f"{candidates} could be loaded. Last error: {last_error}")

        if self.model_id != BOOK_MODEL:
            log3.warning("=" * 70)
            log3.warning(f"RUNNING ON {self.model_id}, NOT the design's {BOOK_MODEL}.")
            log3.warning("Descriptions and, later, prose will differ from the design's")
            log3.warning("assumptions. Set BOOK_MODEL in Stage 0 once the intended")
            log3.warning("checkpoint is reachable.")
            log3.warning("=" * 70)

        self.model.eval()
        n_params = sum(p.numel() for p in self.model.parameters())
        log3.info(f"Ready in {time.time() - t0:.1f}s: {self.model_id} "
                  f"({n_params / 1e9:.1f}B params)")

    # ------------------------------------------------------------- helpers --
    def _fit(self, image: Image.Image) -> Image.Image:
        side = max(image.size)
        if side <= self.cfg.max_image_side:
            return image
        scale = self.cfg.max_image_side / side
        return image.resize((max(1, int(image.width * scale)),
                             max(1, int(image.height * scale))), Image.LANCZOS)

    # ------------------------------------------------------------- chatting --
    @torch.inference_mode()
    def _chat(self, content: List[Dict[str, Any]], system: Optional[str] = None,
              max_new_tokens: int = 800, temperature: float = 0.2) -> str:
        """
        Run one turn over interleaved content blocks.

        `content` uses the design document's shape:
            {"type": "text",  "text": "..."}
            {"type": "image", "image": <path or PIL.Image>}
        """
        images: List[Image.Image] = []
        blocks: List[Dict[str, Any]] = []

        for block in content:
            if block.get("type") == "image":
                img = block["image"]
                if not isinstance(img, Image.Image):
                    img = Image.open(str(img))
                    img.load()
                images.append(self._fit(img.convert("RGB")))
                blocks.append({"type": "image"})
            else:
                blocks.append({"type": "text", "text": block.get("text", "")})

        messages: List[Dict[str, Any]] = []
        if system:
            messages.append({"role": "system",
                             "content": [{"type": "text", "text": system}]})
        messages.append({"role": "user", "content": blocks})

        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)

        proc_kwargs: Dict[str, Any] = {"text": [text], "return_tensors": "pt"}
        if images:
            proc_kwargs["images"] = images
        inputs = self.processor(**proc_kwargs).to(self.model.device)

        gen_kwargs: Dict[str, Any] = {"max_new_tokens": max_new_tokens,
                                      "do_sample": temperature > 0}
        if temperature > 0:
            # Passing temperature with do_sample=False is ignored and warns,
            # so only set it when it will actually be used.
            gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = 0.9

        output_ids = self.model.generate(**inputs, **gen_kwargs)
        self.calls += 1

        trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], output_ids)]
        return self.processor.batch_decode(
            trimmed, skip_special_tokens=True,
            clean_up_tokenization_spaces=False)[0].strip()

    # ------------------------------------------------------------ public API --
    def generate(self, prompt: str, system: Optional[str] = None,
                 images: Optional[List[Any]] = None,
                 max_new_tokens: int = 400, temperature: float = 0.2) -> str:
        content: List[Dict[str, Any]] = [{"type": "image", "image": im}
                                         for im in (images or [])]
        content.append({"type": "text", "text": prompt})
        return self._chat(content, system=system, max_new_tokens=max_new_tokens,
                          temperature=temperature)

    def generate_structured(self, content: List[Dict[str, Any]], schema: Dict[str, Any],
                            system: Optional[str] = None,
                            max_new_tokens: int = 2_000,
                            temperature: float = 0.0,
                            max_attempts: Optional[int] = None) -> Optional[Dict]:
        """
        Ask for one JSON object matching `schema`, repairing on parse failure.

        This is the capability the design assumes throughout
        (`llm.generate_structured(content, system=..., schema=...)`) and that the
        notebook did not have. Every agent in Pipelines B and C needs it.

        The repair loop is the part worth keeping: on a failure we re-prompt
        with the PARSER'S OWN ERROR appended. "braces never balanced - the reply
        was probably truncated" is a far more useful instruction than "try
        again", and it usually succeeds on the second attempt.

        Returns the object, or None once the attempts are exhausted. Callers
        must handle None -- a missing description is recoverable, a fabricated
        one is not.
        """
        attempts = max_attempts or self.cfg.structured_max_attempts

        instruction = ("Reply with ONE JSON object and nothing else: no prose, "
                       "no markdown fence, no explanation.\n"
                       "It must match this shape exactly:\n"
                       f"{json.dumps(schema, indent=2)}")
        base = list(content) + [{"type": "text", "text": instruction}]
        repair_note = None

        for attempt in range(1, attempts + 1):
            blocks = base if repair_note is None else base + [
                {"type": "text",
                 "text": f"YOUR PREVIOUS REPLY COULD NOT BE PARSED: {repair_note}\n"
                         f"Reply with only the JSON object."}]

            raw = self._chat(blocks, system=system, max_new_tokens=max_new_tokens,
                             temperature=temperature)
            obj, reason = extract_json_object(raw)

            if obj is not None:
                missing = [k for k in schema if k not in obj]
                if not missing:
                    return obj
                reason = f"missing required key(s): {missing}"

            log3.warning(f"  structured attempt {attempt}/{attempts}: {reason}")
            self.structured_repairs += 1
            repair_note = reason

        self.structured_failures += 1
        log3.error(f"  structured generation failed after {attempts} attempts")
        return None

    def cleanup(self) -> None:
        del self.model, self.processor
        self.model = self.processor = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        log3.info("Model released.")


book_model = BookModel(stage3_cfg)
print(book_model.generate("Reply with exactly: ready", max_new_tokens=8, temperature=0.0))

In [ ]:
# ============================================================================
# Stage 3 — The document view, and describing every figure in one call
# ============================================================================

# Verbatim from the design document. Every clause is load-bearing; the last
# sentence is what stops a newsletter signup box becoming 400 tokens of analysis.
DESCRIBE_SYSTEM = (
    "You are preparing source material for a technical book. For each figure, table, "
    "or equation given, write a description that could stand in for it in running text: "
    "what it shows, what the surrounding text is using it to demonstrate, and any values, "
    "labels, or axis meanings a reader would need.\n"
    "Do not invent labels that are not visible. If a figure is decorative or unreadable, "
    "say so and write nothing further."
)

# The design's schema. `decorative` is a legitimate answer, not an error.
DESCRIBE_SCHEMA = {
    "figures": [{"id": "string",
                 "description": "string - empty if decorative",
                 "kind": "figure|table|equation|decorative"}]
}

FIGURE_MARKER_RE = re.compile(r"\[(IMAGE|TABLE|EQUATION) (fig_[a-z0-9_]+)\]")
KIND_BY_ITEM_TYPE = {"image": "figure", "table": "table", "equation": "equation"}


def looks_decorative(image: Image.Image, cfg: Stage3Config) -> Optional[str]:
    """
    The cheap filter, applied before any GPU time is spent.

    An avatar is 48x48. A follow button is 200x40. A horizontal rule is 700x3.
    None is worth a place in the model's context window. Returns a reason, or
    None if the image deserves a real look.
    """
    w, h = image.size
    if w < cfg.min_image_width or h < cfg.min_image_height:
        return f"too small ({w}x{h})"
    if w * h < cfg.min_image_pixels:
        return f"too few pixels ({w * h})"
    ratio = max(w, h) / max(1, min(w, h))
    if ratio > cfg.max_aspect_ratio:
        return f"extreme aspect ratio ({ratio:.1f}:1)"
    return None


def build_document_view(parsed: Dict[str, Any],
                        cfg: Stage3Config) -> Tuple[str, List[Dict[str, Any]]]:
    """
    Build the design's `text_with_markers`: the full document as running text,
    with every figure replaced by its id.

    This single string is what lets the model notice that figure 4 refines
    figure 2 -- it can read the sentences between them.

    Returns (marker_text, figures) where each figure entry carries its id, the
    parsed item, and its pre-filter verdict.
    """
    slug = parsed.get("doc_slug") or doc_slug(parsed["filename"])
    parts: List[str] = []
    figures: List[Dict[str, Any]] = []
    counter = 0

    for index, item in enumerate(parsed["content_list"]):
        item_type = item.get("type", "unknown")

        if item_type in cfg.drop_types:          # layout furniture, never seen again
            continue

        if item_type == "text":
            text = normalise_text(item.get("text", ""))
            if text:
                parts.append(text)
            continue

        if item_type not in KIND_BY_ITEM_TYPE:
            salvage = normalise_text(str(item.get("text", "")))
            if salvage:
                parts.append(salvage)
            continue

        counter += 1
        fig_id = f"fig_{slug}_{counter:03d}"

        caption = item.get("caption") or item.get("image_caption") or []
        if isinstance(caption, list):
            caption = " ".join(str(c) for c in caption)
        # A newline inside a caption would break the one-line <<marker>> that
        # convert_document's substitution regex matches, leaving the caption
        # text stranded in the corpus. Collapse all whitespace to spaces.
        caption = re.sub(r"\s+", " ", str(caption)).strip()

        entry: Dict[str, Any] = {
            "id": fig_id,
            "kind": KIND_BY_ITEM_TYPE[item_type],
            "item": item,
            "item_index": index,
            "caption": caption or None,
            "image": None,
            "prefilter": None,
            "payload": None,
        }

        # Load the crop if there is one, and apply the cheap filter now so a
        # button never occupies a slot in the model's context.
        img_path = item.get("img_path")
        if img_path and Path(img_path).exists():
            try:
                image = Image.open(img_path)
                image.load()
                reason = looks_decorative(image, cfg)
                if reason:
                    entry["prefilter"] = reason
                else:
                    entry["image"] = image
            except Exception as exc:
                entry["prefilter"] = f"unreadable ({exc})"
        elif img_path:
            entry["prefilter"] = "crop missing on disk"

        # Items with no usable crop still carry markup worth describing.
        if entry["image"] is None:
            if item_type == "equation":
                entry["payload"] = str(item.get("latex") or item.get("text") or "")
            elif item_type == "table":
                entry["payload"] = str(item.get("table_body") or
                                       item.get("table_data") or "")

        figures.append(entry)

        marker = f"<<{fig_id}>>"
        if caption:
            marker += f"  (printed caption: {caption})"
        parts.append(marker)

    return "\n\n".join(parts), figures


def describe_document(parsed: Dict[str, Any], model: BookModel,
                      cfg: Stage3Config) -> Tuple[Dict[str, Dict[str, str]],
                                                  str, List[Dict[str, Any]]]:
    """
    The design's Stage 3: describe every figure in a document in one call.

        "With a 256K window you can pass an entire document and all its figures
         at once, rather than making one isolated call per image. Fewer calls,
         better descriptions, and the model can notice when figure 4 is a
         refinement of figure 2."

    One practical departure, stated rather than hidden: a document with 95
    figures will not fit any single GPU's activation memory, so figures are sent
    in batches of `max_figures_per_call`. Every batch still receives the WHOLE
    marker text, so cross-figure reasoning survives; only the images are split.
    With `max_figures_per_call` above the document's figure count it is exactly
    one call, as the design describes.

    Returns ({fig_id: {"description", "kind"}}, marker_text, figure_entries).
    """
    marker_text, figures = build_document_view(parsed, cfg)
    results: Dict[str, Dict[str, str]] = {}

    # Anything the pre-filter already rejected is settled without a call.
    describable = []
    for entry in figures:
        if entry["prefilter"]:
            results[entry["id"]] = {"kind": "decorative",
                                    "description": "",
                                    "why": entry["prefilter"]}
        elif entry["image"] is not None or entry["payload"]:
            describable.append(entry)
        else:
            results[entry["id"]] = {"kind": "decorative", "description": "",
                                    "why": "nothing to describe"}

    if not describable:
        return results, marker_text, figures

    view = marker_text[:cfg.max_document_chars]
    if len(marker_text) > cfg.max_document_chars:
        view += "\n\n[document truncated for this call]"

    batches = [describable[i:i + cfg.max_figures_per_call]
               for i in range(0, len(describable), cfg.max_figures_per_call)]

    log3.info(f"  {parsed['filename']}: {len(describable)} describable items "
              f"in {len(batches)} call(s)")

    for batch in batches:
        # This mirrors the design's `describe_document` content layout exactly.
        content: List[Dict[str, Any]] = [{
            "type": "text",
            "text": f"FULL DOCUMENT TEXT (figures marked by id):\n{view}"
        }, {
            "type": "text",
            "text": ("Describe ONLY the items listed below. Use the document text "
                     "above to say what each one is being used to demonstrate, and "
                     "to note when one item refines or repeats an earlier one.")
        }]

        for entry in batch:
            if entry["image"] is not None:
                content.append({"type": "image", "image": entry["image"]})
                content.append({"type": "text",
                                "text": f"^ figure id: {entry['id']}  "
                                        f"(kind: {entry['kind']})"})
            else:
                content.append({"type": "text",
                                "text": f"figure id: {entry['id']}  "
                                        f"(kind: {entry['kind']})\n"
                                        f"{entry['kind'].upper()} SOURCE:\n"
                                        f"{entry['payload'][:4000]}"})

        reply = model.generate_structured(
            content, schema=DESCRIBE_SCHEMA, system=DESCRIBE_SYSTEM,
            max_new_tokens=cfg.document_call_max_tokens, temperature=0.0)

        if not reply:
            continue

        for record in reply.get("figures", []):
            fig_id = str(record.get("id", "")).strip()
            if fig_id not in {e["id"] for e in batch}:
                continue                    # the model invented an id; ignore it
            kind = str(record.get("kind", "figure")).strip().lower()
            description = str(record.get("description", "")).strip()
            if kind == "decorative" or len(description) < 40:
                results[fig_id] = {"kind": "decorative", "description": "",
                                   "why": "model marked it decorative"}
            else:
                results[fig_id] = {"kind": kind, "description": description}

    return results, marker_text, figures


log3.info("Stage 3 document-level description ready.")

In [ ]:
# ============================================================================
# Stage 3 — The per-figure fallback
# ============================================================================
"""
The document-level call is the design's path and handles the normal case. Two
things can still leave a figure without a description:

  * the structured reply omitted an id (models do skip items in long lists), or
  * `describe_whole_document` is switched off because the GPU is small.

For those, we fall back to the design's OTHER rule -- "describe the figure with
the page around it" -- calling once per figure with a window of the surrounding
text. It is strictly worse than the document-level call, because the model
cannot see that figure 4 refines figure 2, so it is a fallback and the report
counts how often it fires.

The description cache is keyed on the SHA-256 of the image bytes, not the
filename. A blog post repeats the same author avatar under a different crop
filename in every section; hashing the pixels catches all of them.
"""


class FallbackDescriber:
    """Per-figure description with page context, plus a content-hash cache."""

    def __init__(self, model: BookModel, cfg: Stage3Config):
        self.model = model
        self.cfg = cfg
        self._cache: Dict[str, Optional[str]] = {}
        self.stats = {"calls": 0, "cache_hits": 0, "declined": 0, "errors": 0}

    def page_context(self, content_list: List[Dict[str, Any]], index: int) -> str:
        """Collect the text immediately before and after an item."""
        cfg = self.cfg
        before, after = [], []

        for j in range(index - 1, max(-1, index - 1 - cfg.context_window_items), -1):
            if content_list[j].get("type") == "text":
                before.append(content_list[j].get("text", "").strip())
        before.reverse()

        for j in range(index + 1,
                       min(len(content_list), index + 1 + cfg.context_window_items)):
            if content_list[j].get("type") == "text":
                after.append(content_list[j].get("text", "").strip())

        half = cfg.context_max_chars // 2
        parts = []
        if before:
            parts.append("TEXT IMMEDIATELY BEFORE THIS ITEM:\n"
                         + " ".join(t for t in before if t)[-half:])
        if after:
            parts.append("TEXT IMMEDIATELY AFTER THIS ITEM:\n"
                         + " ".join(t for t in after if t)[:half])
        return "\n\n".join(parts)

    def describe(self, entry: Dict[str, Any],
                 content_list: List[Dict[str, Any]]) -> Dict[str, str]:
        """Return {"kind", "description"} -- kind may be 'decorative'."""
        decorative = {"kind": "decorative", "description": "", "why": "fallback declined"}

        digest = None
        if entry["image"] is not None:
            try:
                digest = hashlib.sha256(
                    Path(entry["item"]["img_path"]).read_bytes()).hexdigest()
            except Exception:
                digest = None

        if digest and digest in self._cache:
            self.stats["cache_hits"] += 1
            cached = self._cache[digest]
            return {"kind": entry["kind"], "description": cached} if cached else decorative

        context = self.page_context(content_list, entry["item_index"])
        prompt_parts = [f"ITEM TYPE: {entry['kind']}"]
        if entry["caption"]:
            prompt_parts.append(f"CAPTION AS PRINTED: {entry['caption']}")
        if context:
            prompt_parts.append(context)
        if entry["payload"]:
            prompt_parts.append(f"{entry['kind'].upper()} SOURCE:\n{entry['payload'][:4000]}")
        prompt_parts.append(
            "Describe this item so it could stand in for the original in running text, "
            "given what the surrounding text uses it to demonstrate. "
            "If it is decorative or unreadable, reply with exactly DECORATIVE.")

        try:
            reply = self.model.generate(
                prompt="\n\n".join(prompt_parts), system=DESCRIBE_SYSTEM,
                images=[entry["image"]] if entry["image"] is not None else None,
                max_new_tokens=self.cfg.figure_max_tokens,
                temperature=self.cfg.temperature)
            self.stats["calls"] += 1
        except Exception as exc:
            log3.error(f"  fallback failed on {entry['id']}: {exc}")
            self.stats["errors"] += 1
            return decorative

        cleaned = reply.strip()
        # A two-word answer is the model shrugging: treat it as decorative.
        if cleaned.upper().startswith("DECORATIVE") or len(cleaned) < 40:
            self.stats["declined"] += 1
            if digest:
                self._cache[digest] = None
            return decorative

        if digest:
            self._cache[digest] = cleaned
        return {"kind": entry["kind"], "description": cleaned}


fallback_describer = FallbackDescriber(book_model, stage3_cfg)
log3.info("Stage 3 fallback describer ready.")

In [ ]:
# ============================================================================
# Stage 3b — Figures stay figures: writing the document and the figure store
# ============================================================================
"""
    "A figure becomes a description, but the image is NOT thrown away."

Every figure produces two artifacts:

  * a line in the text        ->  [IMAGE fig_slug_003]
                                  <the description>
  * a record in figure_store  ->  {"id": ..., "kind": ..., "path": ...,
                                   "description": ..., "used_in_sections": []}

The inline marker is what makes the Layer 3 join possible: after Stage 4 chunks
the corpus, a chunk's figure list is exactly the marker ids inside its span.
Without the marker, the figure store is a folder of PNGs nothing points at.

Decorative items are recorded too, with `kind: "decorative"` and no description.
They never enter the corpus text, but they stay countable -- a filter you cannot
audit is a filter you cannot tune.
"""


def convert_document(parsed: Dict[str, Any], model: BookModel,
                     fallback: FallbackDescriber,
                     cfg: Stage3Config) -> Tuple[str, List[Dict], Dict[str, Any]]:
    """Turn one Stage 2 document into (text, figure_records, stats)."""
    stats = {
        "total_items": len(parsed["content_list"]),
        "text_blocks": 0,
        "figures_kept": 0, "tables_kept": 0, "equations_kept": 0,
        "decorative": 0, "dropped_layout": 0,
        "document_level": 0, "fallback_used": 0,
        "started": time.time(),
    }

    stats["dropped_layout"] = sum(
        1 for item in parsed["content_list"]
        if item.get("type") in cfg.drop_types)

    if cfg.describe_whole_document:
        described, marker_text, figures = describe_document(parsed, model, cfg)
        stats["document_level"] = sum(1 for v in described.values() if v.get("description"))
    else:
        marker_text, figures = build_document_view(parsed, cfg)
        described = {}

    # Any figure the document-level pass did not settle goes to the fallback.
    for entry in figures:
        if entry["id"] in described:
            continue
        if entry["prefilter"]:
            described[entry["id"]] = {"kind": "decorative", "description": "",
                                      "why": entry["prefilter"]}
            continue
        described[entry["id"]] = fallback.describe(entry, parsed["content_list"])
        stats["fallback_used"] += 1

    # ---- build the final text by substituting each marker ------------------
    figure_records: List[Dict[str, Any]] = []
    by_id = {e["id"]: e for e in figures}

    def render(match: re.Match) -> str:
        fig_id = match.group(1)
        entry = by_id.get(fig_id)
        outcome = described.get(fig_id, {"kind": "decorative", "description": ""})
        kind = outcome.get("kind", "figure")
        description = outcome.get("description", "")

        stored_path = None
        if description:
            # Keep the crop: the Writer in Pipeline C is multimodal and reads
            # the diagram itself, not somebody's summary of it.
            source = (entry or {}).get("item", {}).get("img_path")
            if source and Path(source).exists():
                dest = PATHS.figures / f"{fig_id}{Path(source).suffix or '.png'}"
                try:
                    shutil.copy2(source, dest)
                    stored_path = str(dest)
                except Exception as exc:
                    log3.warning(f"  could not store crop for {fig_id}: {exc}")

        figure_records.append({
            "id": fig_id,
            "kind": kind,
            "path": stored_path,
            "caption": (entry or {}).get("caption"),
            "description": description,
            "source_document": parsed["filename"],
            "source_type": parsed.get("source_type", "unknown"),
            "used_in_sections": [],
            "skipped_reason": outcome.get("why") if not description else None,
        })

        if not description:
            stats["decorative"] += 1
            return ""                       # "write nothing further"

        stats[{"figure": "figures_kept", "table": "tables_kept",
               "equation": "equations_kept"}.get(kind, "figures_kept")] += 1

        word = {"figure": "IMAGE", "table": "TABLE", "equation": "EQUATION"}.get(kind, "IMAGE")
        caption = (entry or {}).get("caption")
        header = f"[{word} {fig_id}: {caption}]" if caption else f"[{word} {fig_id}]"

        # Equations keep their LaTeX: it is ground truth, and the explanation is
        # a gloss on it rather than a replacement for it.
        payload = (entry or {}).get("payload")
        if kind == "equation" and payload:
            return f"{header}\n{payload}\n\n{description}"
        return f"{header}\n{description}"

    text = re.sub(r"<<(fig_[a-z0-9_]+)>>(?:  \(printed caption: [^\n]*\))?",
                  render, marker_text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    stats["text_blocks"] = marker_text.count("\n\n") + 1
    stats["elapsed"] = round(time.time() - stats.pop("started"), 2)
    stats["output_chars"] = len(text)
    stats["source_document"] = parsed["filename"]
    stats["source_type"] = parsed.get("source_type", "unknown")
    stats["figure_ids"] = [f["id"] for f in figure_records if f["description"]]

    return text, figure_records, stats


log3.info("Stage 3b conversion ready.")

In [ ]:
# ============================================================================
# Stage 3c — Conservative transcript cleanup
# ============================================================================
"""
    "The same model also cleans transcripts... Do this as a separate and
     explicitly conservative pass -- the instruction is FIX TERMS AND
     PUNCTUATION, CHANGE NOTHING ELSE -- and keep the raw transcript alongside
     the cleaned one. Cleaning is the step most likely to quietly delete
     content."

Two guards make that instruction enforceable rather than aspirational:

  1. The transcript is cleaned in windows, so no single generation holds an hour
     of speech, and no single bad generation can lose an hour of it.
  2. If a cleaned window's length differs from the raw window by more than
     `transcript_length_tolerance`, the RAW window is kept. A model that
     summarised instead of correcting fails this immediately.

The design also asks the pass to "mark where the speaker is reading a slide
versus explaining freely", which is why the prompt requests a [READING SLIDE]
tag -- a lecturer reading bullet points aloud is much weaker source material
than the same lecturer explaining why the bullet point is true.

Per the design's Files on Disk, both survive:
    data/transcripts/lecture_04.raw.json    <- Whisper, timestamped
    data/transcripts/lecture_04.clean.md    <- after this pass
"""

TRANSCRIPT_CLEAN_SYSTEM = (
    "You repair automatic speech transcripts of technical lectures.\n"
    "You may: fix mis-transcribed technical terms, fix punctuation and "
    "capitalisation, insert paragraph breaks at topic changes, and mark a "
    "passage where the speaker is plainly reading text off a slide by putting "
    "[READING SLIDE] on its own line before it.\n"
    "You may NOT: summarise, shorten, reword, reorder, add commentary, or remove "
    "anything the speaker said, including repetitions and false starts.\n"
    "Return only the corrected transcript text. No preamble, no explanation."
)


def clean_transcript_text(raw_text: str, vocabulary: List[str], model: BookModel,
                          cfg: Stage3Config) -> Tuple[str, Dict[str, int]]:
    """Clean window by window, falling back to raw on any doubt."""
    guard = {"windows": 0, "kept_clean": 0, "kept_raw": 0}

    paragraphs = raw_text.split("\n\n")
    windows, current = [], ""
    for para in paragraphs:
        if current and len(current) + len(para) > cfg.transcript_window_chars:
            windows.append(current)
            current = para
        else:
            current = f"{current}\n\n{para}" if current else para
    if current:
        windows.append(current)

    vocab_hint = ", ".join(vocabulary[:60])
    cleaned_windows: List[str] = []

    for window in tqdm(windows, desc="  cleaning transcript", leave=False):
        guard["windows"] += 1
        prompt = (f"KNOWN TECHNICAL TERMS IN THIS CORPUS: {vocab_hint}\n\n"
                  f"TRANSCRIPT SEGMENT:\n{window}")
        try:
            cleaned = model.generate(prompt=prompt, system=TRANSCRIPT_CLEAN_SYSTEM,
                                     max_new_tokens=int(len(window) / 2.5) + 128,
                                     temperature=0.0).strip()
        except Exception as exc:
            log3.warning(f"  transcript cleanup failed on a window: {exc}")
            cleaned = ""

        ratio = len(cleaned) / max(1, len(window))
        if cleaned and abs(1 - ratio) <= cfg.transcript_length_tolerance:
            cleaned_windows.append(cleaned)
            guard["kept_clean"] += 1
        else:
            if cleaned:
                log3.warning(f"  cleanup changed length by {abs(1 - ratio):.0%} "
                             f"- keeping raw")
            cleaned_windows.append(window)
            guard["kept_raw"] += 1

    return "\n\n".join(cleaned_windows), guard


def convert_transcripts(model: BookModel, cfg: Stage3Config) -> List[Dict[str, Any]]:
    """
    Turn every Stage 1 transcript into a converted text file with a
    character -> timestamp index attached.

    Transcripts skip Stage 2 entirely: a layout parser has nothing to say about
    speech. They join the corpus here.
    """
    raws = sorted(PATHS.transcripts.glob("*.raw.json"))
    if not raws:
        return []

    vocabulary = load_vocabulary_from_pipeline_b(PATHS.root / "normalized_tags.json")
    results = []

    for raw_path in raws:
        data = json.loads(raw_path.read_text(encoding="utf-8"))
        stem = raw_path.name.replace(".raw.json", "")
        out_txt = PATHS.converted / f"{stem}.txt"

        if cfg.skip_existing and not cfg.force_reconvert and out_txt.exists():
            log3.info(f"Transcript already converted: {stem}")
            continue

        # Normalise each segment BEFORE building the index. Normalising the
        # flattened text afterwards shifts character offsets ("…" -> "...",
        # collapsed runs of spaces), which would quietly falsify the
        # `timestamp_index_exact: True` claim for uncleaned transcripts.
        segments = [{**seg, "text": normalise_text(seg.get("text", ""))}
                    for seg in data["segments"]]
        flat, ts_index = segments_to_text(segments)

        guard = {"windows": 0, "kept_clean": 0, "kept_raw": 0}
        index_exact = True
        text = flat

        if cfg.clean_transcripts:
            log3.info(f"Cleaning transcript: {stem} ({len(flat):,} chars)")
            text, guard = clean_transcript_text(flat, vocabulary, model, cfg)
            # Cleaning shifts character positions, so the index built from the
            # raw text no longer lines up exactly. We keep it and mark it
            # approximate: minute-level provenance survives a few hundred
            # characters of drift, and that is all it is used for.
            index_exact = False
            (PATHS.transcripts / f"{stem}.clean.md").write_text(text, encoding="utf-8")

        out_txt.write_text(text, encoding="utf-8")

        stats = {
            "source_document": data["source_document"],
            "source_type": "transcript",
            "output_chars": len(text),
            "figures": [], "figure_ids": [],
            "duration_seconds": data.get("duration_seconds"),
            "language": data.get("language"),
            "timestamp_index": ts_index,
            "timestamp_index_exact": index_exact,
            "cleanup": guard,
        }
        (PATHS.converted / f"{stem}.txt.stats.json").write_text(
            json.dumps(stats, indent=2), encoding="utf-8")

        log3.info(f"  {stem}: {len(text):,} chars, {len(ts_index)} timestamp anchors, "
                  f"{guard['kept_raw']}/{guard['windows']} windows kept raw")
        results.append(stats)

    return results


log3.info("Stage 3c transcript cleanup ready.")

In [ ]:
# ============================================================================
# Stage 3 — Run
# ============================================================================

def run_stage3(model: BookModel, fallback: FallbackDescriber,
               cfg: Stage3Config) -> Dict[str, Any]:
    log3.info("=" * 70)
    log3.info("STAGE 3: DESCRIBING FIGURES IN CONTEXT")
    log3.info("=" * 70)

    parsed_files = sorted(PATHS.parsed.glob("*.json"))
    log3.info(f"Found {len(parsed_files)} parsed documents")

    all_figures: List[Dict[str, Any]] = []
    processed, skipped, failed = [], [], []
    totals = {"figures_kept": 0, "tables_kept": 0, "equations_kept": 0,
              "decorative": 0, "dropped_layout": 0,
              "document_level": 0, "fallback_used": 0}
    t0 = time.time()

    for json_path in tqdm(parsed_files, desc="Describing documents"):
        try:
            parsed = json.loads(json_path.read_text(encoding="utf-8"))
        except Exception as exc:
            log3.error(f"Unreadable {json_path.name}: {exc}")
            failed.append(json_path.name)
            continue

        if parsed.get("status") != "success":
            skipped.append(json_path.name)
            continue

        slug = parsed.get("doc_slug") or doc_slug(parsed["filename"])
        out_txt = PATHS.converted / f"{slug}.txt"
        sidecar = PATHS.converted / f"{slug}.txt.stats.json"

        if cfg.skip_existing and not cfg.force_reconvert and out_txt.exists():
            skipped.append(json_path.name)
            if sidecar.exists():     # keep figure_store complete across resumes
                all_figures.extend(
                    json.loads(sidecar.read_text(encoding="utf-8")).get("figures", []))
            continue

        try:
            text, figures, stats = convert_document(parsed, model, fallback, cfg)
            out_txt.write_text(text, encoding="utf-8")
            stats["figures"] = figures
            sidecar.write_text(json.dumps(stats, indent=2, ensure_ascii=False),
                               encoding="utf-8")

            all_figures.extend(figures)
            processed.append(json_path.name)
            for key in totals:
                totals[key] += stats.get(key, 0)

            log3.info(f"  {parsed['filename']}: {stats['output_chars']:,} chars | "
                      f"kept {stats['figures_kept']}f/{stats['tables_kept']}t/"
                      f"{stats['equations_kept']}e | {stats['decorative']} decorative | "
                      f"{stats['dropped_layout']} layout items dropped")
        except Exception as exc:
            log3.error(f"Conversion failed for {json_path.name}: {exc}")
            log3.debug(traceback.format_exc())
            failed.append(json_path.name)
            if not cfg.continue_on_error:
                raise

    transcript_stats = convert_transcripts(model, cfg)

    PATHS.figure_store.write_text(
        json.dumps({"model": model.model_id, "figures": all_figures},
                   indent=2, ensure_ascii=False), encoding="utf-8")

    summary = {
        "model": model.model_id,
        "documents_converted": len(processed),
        "documents_skipped": len(skipped),
        "documents_failed": len(failed),
        "transcripts_converted": len(transcript_stats),
        "figures_in_store": len(all_figures),
        "figures_described": sum(1 for f in all_figures if f["description"]),
        "totals": totals,
        "model_calls": model.calls,
        "structured_repairs": model.structured_repairs,
        "structured_failures": model.structured_failures,
        "fallback": dict(fallback.stats),
        "elapsed_minutes": round((time.time() - t0) / 60, 2),
    }

    log3.info("-" * 70)
    log3.info(f"  model       : {summary['model']}")
    log3.info(f"  converted   : {summary['documents_converted']} documents, "
              f"{summary['transcripts_converted']} transcripts")
    log3.info(f"  described   : {summary['figures_described']} of "
              f"{summary['figures_in_store']} items")
    log3.info(f"  model calls : {summary['model_calls']}")
    log3.info(f"  elapsed     : {summary['elapsed_minutes']} min")
    return summary


stage3_summary = run_stage3(book_model, fallback_describer, stage3_cfg)

In [ ]:
# ============================================================================
# Stage 3 — What the filters and the model actually did
# ============================================================================
"""
Read this before Stage 4. The numbers here are the only way to tell a working
filter from a destructive one:

  * `decorative` near zero on a corpus of web articles -> the model is not using
    its escape hatch, and you are back to paying 400 tokens for signup widgets.
  * `decorative` near 100% -> the pre-filter thresholds are eating real diagrams.
  * `fallback_used` high -> the document-level call is dropping ids. Lower
    `max_figures_per_call` so each call has fewer items to keep track of.
  * `structured_failures` above zero -> some documents lost their figures
    entirely. Those are logged by name in stage3_describe.log.
"""

t = stage3_summary["totals"]
kept = t["figures_kept"] + t["tables_kept"] + t["equations_kept"]
seen = kept + t["decorative"]

print("=" * 70)
print("STAGE 3 REPORT")
print("=" * 70)
print(f"Model actually used          : {stage3_summary['model']}")
if stage3_summary["model"] != BOOK_MODEL:
    print(f"  NOTE: design specifies      {BOOK_MODEL}")

print(f"\nLayout items dropped         : {t['dropped_layout']}"
      "   (headers, footers, page numbers)")
print(f"Multimodal items seen        : {seen}")
print(f"  described and kept         : {kept}"
      f"  ({t['figures_kept']} figures, {t['tables_kept']} tables, "
      f"{t['equations_kept']} equations)")
print(f"  decorative / declined      : {t['decorative']}")

print(f"\nDescribed at document level  : {t['document_level']}   "
      "(the design's path: whole document, one call)")
print(f"Fell back to per-figure      : {t['fallback_used']}")
print(f"  fallback cache hits        : {stage3_summary['fallback']['cache_hits']}")

print(f"\nModel calls total            : {stage3_summary['model_calls']}")
print(f"  JSON repairs needed        : {stage3_summary['structured_repairs']}")
print(f"  JSON failures (items lost) : {stage3_summary['structured_failures']}"
      f"{'   <-- check the log' if stage3_summary['structured_failures'] else ''}")

print(f"\nFigure crops on disk         : {len(list(PATHS.figures.glob('*')))}")

store = json.loads(PATHS.figure_store.read_text(encoding="utf-8"))["figures"]
described = [f for f in store if f["description"]]
if described:
    ex = described[0]
    print("\n" + "-" * 70)
    print(f"SAMPLE KEPT FIGURE  ({ex['id']}, kind={ex['kind']})")
    print(f"from : {ex['source_document']}")
    print(f"crop : {ex['path']}")
    print("-" * 70)
    print(ex["description"][:700])
print("=" * 70)

## Stage 4 — Consolidation and Chunking

> *"Output is `raw_consolidated_text.txt` with document boundaries marked, plus
> `chunk_metadata.json`."*

The previous version of this notebook produced the first file and stopped. The chunks and their
metadata were left to Pipeline B — but `chunk_metadata.json` is a **Layer 3** artifact, and Layer 3
is read-only: `SourceMemory` opens the file and the Chroma collection, and never writes to either.
If Pipeline A does not produce them, nothing does. So Stage 4 now does the whole job the design
gives it, and writes what the design's *Files on Disk* section lists:

```
data/
├── chunks/
│   ├── chunk_0001.txt
│   └── ...
├── figures/
├── transcripts/
├── chunk_metadata.json      # L3 — provenance, timestamps, tags, figures
├── document_index.json
└── source_index/            # Chroma: chunk embeddings
```

---

### The two chunking decisions, and why they are not arbitrary

**Chunks stay moderate: 1,500–2,000 tokens.**

> *"It is tempting to make them huge now that everything fits. Do not. Chunk size is the granularity
> of the coverage map — the thing that detects repetition — and coarse chunks make 'already used in
> section 4.1' a much blunter signal."*

This is the clearest example in the whole design of a preprocessing decision that only pays off two
pipelines later. The coverage map in the Book Ledger records which chunks a section consumed. If a
chunk is 8,000 tokens, then "already used" covers a quarter of a chapter's worth of material and
the repetition detector is useless. The chunk size *is* the resolution of failure #1.

**Retrieve neighbourhoods, not lone chunks.**

> *"When a search hits `chunk_211`, return `chunk_210` through `chunk_212`. A chunk boundary is an
> artifact of the chunker, not of the argument."*

Note what this replaces: **overlap**. The obvious way to avoid mid-thought fragments is to overlap
chunks by a few hundred tokens — but then the coverage map double-counts, because the same sentences
live in two chunks. The design solves it at *read* time instead: chunks are disjoint, and the
retriever widens the window. So `overlap_tokens` is zero here on purpose, and
`get_chunk_neighbourhood()` is provided for Layer 3 to call.

One refinement the design implies but does not spell out: **a chunk never straddles two documents,
and a neighbourhood never crosses a document boundary.** `chunk_0043` being the last chunk of a
lecture and `chunk_0044` the first page of an unrelated paper makes them neighbours by numbering
and strangers by meaning. Chunking runs per document, so every chunk has exactly one
`source_document` — which is what makes the provenance in `chunk_metadata.json` a fact rather than
a guess.

### The offsets

Every chunk records its exact character span in `raw_consolidated_text.txt`, and Stage 4 asserts
that `corpus[start:end]` really is the chunk it wrote. The old code predicted positions by hand
while a `"\n".join` built the string; the arithmetic and the string drifted apart and the index
pointed at the wrong text, silently. Positions now come from the string itself.

In [ ]:
# ============================================================================
# Stage 4 — Configuration
# ============================================================================

log4 = make_logger("stage4.consolidate", "stage4_consolidation.log")


@dataclass
class Stage4Config:
    """Configuration for Stage 4 - Consolidation and chunking."""

    # -- Consolidation -------------------------------------------------------
    sort_order: str = "alphabetical"        # "alphabetical" | "custom" | "type"
    custom_order_file: Optional[str] = None
    add_document_boundaries: bool = True
    boundary_rule: str = "=" * 78
    blank_lines_between_docs: int = 2
    create_backup: bool = True

    # -- Chunking ------------------------------------------------------------
    # "Keep chunks moderate: roughly 1,500-2,000 tokens." Chunk size is the
    # resolution of the coverage map, which is what detects repetition.
    target_chunk_tokens: int = 1_750
    min_chunk_tokens: int = 1_200
    max_chunk_tokens: int = 2_200

    # Zero on purpose. The design handles mid-thought fragments with
    # neighbourhood retrieval at read time, not with overlap at write time --
    # overlapping chunks would double-count in the coverage map.
    overlap_tokens: int = 0

    write_chunk_files: bool = True          # data/chunks/chunk_0001.txt
    verify_offsets: bool = True             # assert corpus[start:end] == chunk


stage4_cfg = Stage4Config()

log4.info("Stage 4 configuration:")
log4.info(f"  sort order   : {stage4_cfg.sort_order}")
log4.info(f"  chunk tokens : {stage4_cfg.min_chunk_tokens}-"
          f"{stage4_cfg.target_chunk_tokens}-{stage4_cfg.max_chunk_tokens}")
log4.info(f"  overlap      : {stage4_cfg.overlap_tokens} "
          f"(neighbourhood retrieval instead)")

In [ ]:
# ============================================================================
# Stage 4a — Consolidation with offsets that are measured, not predicted
# ============================================================================

def discover_converted() -> List[Path]:
    return sorted(p for p in PATHS.converted.glob("*.txt")
                  if not p.name.startswith("SAMPLE_"))


def load_sidecar(txt_path: Path) -> Dict[str, Any]:
    sidecar = txt_path.parent / f"{txt_path.name}.stats.json"
    if sidecar.exists():
        try:
            return json.loads(sidecar.read_text(encoding="utf-8"))
        except Exception as exc:
            log4.warning(f"Unreadable sidecar for {txt_path.name}: {exc}")
    return {}


def order_documents(paths: List[Path], cfg: Stage4Config) -> List[Path]:
    """
    Decide the order documents appear in the corpus.

    This is not the book's ordering -- Pipeline B's CurriculumOrderer decides
    that. It only has to be STABLE, so re-running the pipeline does not shuffle
    every character offset and invalidate a partially written book.
    """
    if cfg.sort_order == "custom" and cfg.custom_order_file:
        try:
            wanted = json.loads(Path(cfg.custom_order_file).read_text(encoding="utf-8"))
            rank = {name: i for i, name in enumerate(wanted)}
            return sorted(paths, key=lambda p: (rank.get(p.name, 10**9), p.name.lower()))
        except Exception as exc:
            log4.warning(f"Custom order unusable ({exc}); using alphabetical")

    if cfg.sort_order == "type":
        return sorted(paths, key=lambda p: (load_sidecar(p).get("source_type", "zz"),
                                            p.name.lower()))
    return sorted(paths, key=lambda p: p.name.lower())


def build_boundary(sidecar: Dict[str, Any], filename: str, cfg: Stage4Config) -> str:
    """Machine-readable, so the corpus is self-describing even without the index."""
    lines = [cfg.boundary_rule,
             f"DOCUMENT: {sidecar.get('source_document', filename)}",
             f"SOURCE_TYPE: {sidecar.get('source_type', 'unknown')}"]
    if sidecar.get("duration_seconds"):
        lines.append(f"DURATION_MINUTES: {sidecar['duration_seconds'] / 60:.1f}")
    if sidecar.get("figure_ids"):
        lines.append(f"FIGURES: {len(sidecar['figure_ids'])}")
    lines.append(cfg.boundary_rule)
    return "\n".join(lines)


def consolidate(paths: List[Path], cfg: Stage4Config) -> Tuple[str, List[Dict[str, Any]]]:
    """
    Build the corpus and its index in one pass.

    `emit()` is the only thing that touches the buffer and the only thing that
    moves the cursor, so there is no separate arithmetic that can drift out of
    step with the string being built.
    """
    buffer: List[str] = []
    cursor = 0
    index: List[Dict[str, Any]] = []

    def emit(chunk: str) -> None:
        nonlocal cursor
        if chunk:
            buffer.append(chunk)
            cursor += len(chunk)

    separator = "\n" * (cfg.blank_lines_between_docs + 1)

    for i, path in enumerate(tqdm(paths, desc="Consolidating")):
        try:
            content = path.read_text(encoding="utf-8")
        except Exception as exc:
            log4.error(f"Could not read {path.name}: {exc}")
            continue

        sidecar = load_sidecar(path)
        if i > 0:
            emit(separator)
        if cfg.add_document_boundaries:
            emit(build_boundary(sidecar, path.name, cfg))
            emit("\n\n")

        start = cursor
        emit(content)
        end = cursor

        index.append({
            "order": i + 1,
            "filename": path.name,
            "doc_slug": path.stem,
            "source_document": sidecar.get("source_document", path.name),
            "source_type": sidecar.get("source_type", "unknown"),
            "start": start, "end": end, "length": end - start,
            "figure_ids": sidecar.get("figure_ids", []),
            "timestamp_index": sidecar.get("timestamp_index", []),
            "timestamp_index_exact": sidecar.get("timestamp_index_exact", True),
        })

    text = "".join(buffer)

    if cfg.verify_offsets:
        for record, path in zip(index, paths):
            if text[record["start"]:record["end"]] != path.read_text(encoding="utf-8"):
                raise AssertionError(
                    f"Offset mismatch for {record['filename']}: recorded "
                    f"[{record['start']}:{record['end']}] does not match the file. "
                    f"Provenance would be wrong for the whole book.")
        log4.info(f"Document offsets verified for all {len(index)} documents")

    return text, index


def validate_corpus(text: str, index: List[Dict[str, Any]], cfg: Stage4Config) -> List[str]:
    issues: List[str] = []

    if len(text.strip()) < 100:
        issues.append("corpus is essentially empty")

    if cfg.add_document_boundaries:
        # Count real boundary lines. The old code used text.count("DOCUMENT:"),
        # which any document mentioning the word would inflate.
        found = len(re.findall(r"^DOCUMENT: ", text, flags=re.MULTILINE))
        if found != len(index):
            issues.append(f"expected {len(index)} boundaries, found {found}")

    if PATHS.figure_store.exists():
        stored = {f["id"] for f in
                  json.loads(PATHS.figure_store.read_text(encoding="utf-8"))["figures"]}
        referenced = {m.group(2) for m in FIGURE_MARKER_RE.finditer(text)}
        orphans = referenced - stored
        if orphans:
            issues.append(f"{len(orphans)} figure markers have no store entry "
                          f"(e.g. {sorted(orphans)[:3]})")

    junk = text.count("[UNKNOWN CONTENT TYPE:")
    if junk:
        issues.append(f"{junk} '[UNKNOWN CONTENT TYPE:]' markers - stale Stage 3 "
                      f"output, delete data/converted and re-run")

    return issues


log4.info("Stage 4a consolidation functions ready.")

In [ ]:
# ============================================================================
# Stage 4b — Chunking, and the Layer 3 metadata
# ============================================================================

def split_into_blocks(text: str) -> List[Tuple[int, int, str]]:
    """
    Split into paragraph-sized blocks, keeping exact offsets.

    A block is a run of lines with no blank line inside it -- which means a
    figure marker and its description, written as

        [IMAGE fig_x_003]
        A diagram of the encoder stack...

    stay together automatically. Splitting those apart would leave a chunk
    holding a description with no marker, and the Layer 3 figure join would miss
    it.
    """
    return [(m.start(), m.end(), m.group(0))
            for m in re.finditer(r"[^\n]+(?:\n(?!\s*\n)[^\n]*)*", text)]


def split_oversized_block(start: int, block: str,
                          cfg: Stage4Config) -> List[Tuple[int, int, str]]:
    """A single block larger than max_chunk_tokens, split on sentence ends."""
    pieces, current_start, current_end = [], None, None
    for m in re.finditer(r"[^.!?]*[.!?]+[\s]*|[^.!?]+$", block):
        if not m.group(0).strip():
            continue
        if current_start is None:
            current_start, current_end = m.start(), m.end()
        else:
            current_end = m.end()
        if count_tokens(block[current_start:current_end]) >= cfg.target_chunk_tokens:
            pieces.append((start + current_start, start + current_end,
                           block[current_start:current_end]))
            current_start = None
    if current_start is not None:
        pieces.append((start + current_start, start + current_end,
                       block[current_start:current_end]))
    return pieces or [(start, start + len(block), block)]


def chunk_document(doc_text: str, cfg: Stage4Config) -> List[Tuple[int, int]]:
    """
    Greedily pack blocks into chunks of roughly `target_chunk_tokens`.

    Returns (start, end) offsets RELATIVE to the document. Chunking runs per
    document, so no chunk ever straddles a document boundary and every chunk has
    exactly one source_document.
    """
    def _chunk_tokens(span: Tuple[int, int]) -> int:
        return count_tokens(doc_text[span[0]:span[1]])

    blocks: List[Tuple[int, int, str]] = []
    for start, end, block in split_into_blocks(doc_text):
        if count_tokens(block) > cfg.max_chunk_tokens:
            blocks.extend(split_oversized_block(start, block, cfg))
        else:
            blocks.append((start, end, block))

    chunks: List[Tuple[int, int]] = []
    current_start = current_end = None
    current_tokens = 0

    for start, end, block in blocks:
        block_tokens = count_tokens(block)
        combined = current_tokens + block_tokens

        # Two independent reasons to close the current chunk:
        #
        #   1. HARD CEILING. Adding this block would break max_chunk_tokens.
        #      This check must NOT also require that we have reached the
        #      minimum -- otherwise a chunk sitting just under min swallows an
        #      arbitrarily large next block. With min=1200 and a 2,100-token
        #      block, that produced a 3,875-token chunk: roughly twice the
        #      design's band, and therefore half the resolution the coverage
        #      map needs to detect repetition.
        #
        #   2. SOFT TARGET. We are past target and already large enough to be
        #      a useful chunk on its own.
        over_ceiling = current_start is not None and combined > cfg.max_chunk_tokens
        past_target = (current_start is not None
                       and combined > cfg.target_chunk_tokens
                       and current_tokens >= cfg.min_chunk_tokens)

        if over_ceiling or past_target:
            chunks.append((current_start, current_end))
            current_start, current_tokens = None, 0

        if current_start is None:
            current_start = start
        current_end = end
        current_tokens += block_tokens

    if current_start is not None:
        # A short tail is merged back rather than shipped as a stub -- a
        # 90-token chunk pollutes the coverage map without teaching anything.
        # But not if the merge would itself breach the ceiling: an undersized
        # tail is a smaller problem than an oversized chunk.
        merged_tokens = (count_tokens("") if not chunks else
                         current_tokens + _chunk_tokens(chunks[-1]))
        if (chunks and current_tokens < cfg.min_chunk_tokens // 2
                and merged_tokens <= cfg.max_chunk_tokens):
            chunks[-1] = (chunks[-1][0], current_end)
        else:
            chunks.append((current_start, current_end))

    return chunks


def timestamp_for(doc: Dict[str, Any], rel_start: int, rel_end: int) -> Optional[Dict]:
    """For a transcript chunk, the seconds range it covers."""
    anchors = doc.get("timestamp_index") or []
    hits = [a for a in anchors
            if a["char_end"] >= rel_start and a["char_start"] <= rel_end]
    if not hits:
        return None
    return {"start_seconds": round(min(h["t_start"] for h in hits), 1),
            "end_seconds": round(max(h["t_end"] for h in hits), 1),
            "approximate": not doc.get("timestamp_index_exact", True)}


def build_chunks_and_metadata(corpus: str, index: List[Dict[str, Any]],
                              cfg: Stage4Config) -> Dict[str, Any]:
    """
    Produce data/chunks/chunk_NNNN.txt and data/chunk_metadata.json.

    The metadata is exactly the Layer 3 shape from the design document:

        "chunk_0212": {
          "source_document": "lecture_04_slides.pptx",
          "source_type": "slides",
          "timestamp": null,
          "tags": [],
          "figures": [ {...} ]
        }

    `tags` is deliberately empty: Pipeline B's FineGrainedTagExtractor fills it.
    This function's job is the part only Pipeline A can know -- where the text
    came from.

    Figures are joined from the inline markers, which are inside the chunk text,
    so no position arithmetic is involved for them at all.
    """
    figures_by_id: Dict[str, Dict] = {}
    if PATHS.figure_store.exists():
        for figure in json.loads(
                PATHS.figure_store.read_text(encoding="utf-8"))["figures"]:
            figures_by_id[figure["id"]] = figure

    for stale in PATHS.chunks.glob("chunk_*.txt"):
        stale.unlink()

    metadata: Dict[str, Any] = {}
    counter = 0
    token_counts: List[int] = []
    oversized: List[Tuple[str, int]] = []

    for doc in tqdm(index, desc="Chunking"):
        doc_text = corpus[doc["start"]:doc["end"]]

        for rel_start, rel_end in chunk_document(doc_text, cfg):
            counter += 1
            chunk_id = f"chunk_{counter:04d}"

            abs_start = doc["start"] + rel_start
            abs_end = doc["start"] + rel_end
            chunk_text = corpus[abs_start:abs_end]

            if cfg.verify_offsets and chunk_text != doc_text[rel_start:rel_end]:
                raise AssertionError(f"{chunk_id}: chunk offsets do not resolve")

            if cfg.write_chunk_files:
                (PATHS.chunks / f"{chunk_id}.txt").write_text(chunk_text,
                                                              encoding="utf-8")

            figure_ids = [m.group(2) for m in FIGURE_MARKER_RE.finditer(chunk_text)]
            tokens = count_tokens(chunk_text)
            token_counts.append(tokens)

            # After the ceiling fix, the only way to land here is a single
            # block over max_chunk_tokens that split_oversized_block could not
            # divide -- i.e. a paragraph with no sentence-ending punctuation
            # anywhere in it. In a real corpus that means a base64 blob, a
            # dumped table, or ASCII art. Shipping it whole is the right call;
            # shipping it whole and silently is not, because an oversized chunk
            # blunts the coverage map exactly where the design says it must be
            # sharp.
            if tokens > cfg.max_chunk_tokens:
                oversized.append((chunk_id, tokens))
                log4.warning(f"  {chunk_id}: {tokens} tokens exceeds "
                             f"max_chunk_tokens={cfg.max_chunk_tokens} - one "
                             f"indivisible block (no sentence breaks)")

            metadata[chunk_id] = {
                "source_document": doc["source_document"],
                "source_type": doc["source_type"],
                "timestamp": timestamp_for(doc, rel_start, rel_end),
                "tags": [],                       # Pipeline B fills this in
                "figures": [figures_by_id[f] for f in figure_ids
                            if f in figures_by_id],
                # -- extras Layer 3 and the finishing passes make use of --
                "doc_slug": doc["doc_slug"],
                "char_span": [abs_start, abs_end],
                "token_count": tokens,
                "index_in_document": len([
                    c for c in metadata.values()
                    if c["source_document"] == doc["source_document"]]),
            }

    PATHS.chunk_metadata.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

    token_counts.sort()
    stats = {
        "chunks": len(metadata),
        "chunks_with_figures": sum(1 for c in metadata.values() if c["figures"]),
        "chunks_with_timestamps": sum(1 for c in metadata.values() if c["timestamp"]),
        "median_tokens": token_counts[len(token_counts) // 2] if token_counts else 0,
        "min_tokens": token_counts[0] if token_counts else 0,
        "max_tokens": token_counts[-1] if token_counts else 0,
        "in_target_band": sum(1 for t in token_counts if 1_500 <= t <= 2_000),
        "oversized": oversized,
    }
    log4.info(f"Chunked into {stats['chunks']} chunks "
              f"(median {stats['median_tokens']} tokens)")
    return stats


def get_chunk_neighbourhood(chunk_id: str, window: int = 1,
                            metadata: Optional[Dict] = None) -> str:
    """
    "When a search hits chunk_211, return chunk_210 through chunk_212."

    Layer 3's SourceMemory calls this instead of reading a lone chunk. The
    refinement the design implies: a neighbour from a DIFFERENT source document
    is not a neighbour, it is a stranger that happens to sit next in the
    numbering, so the window stops at document boundaries.
    """
    metadata = metadata or json.loads(
        PATHS.chunk_metadata.read_text(encoding="utf-8"))
    if chunk_id not in metadata:
        return ""

    home = metadata[chunk_id]["source_document"]
    number = int(chunk_id.split("_")[1])

    parts = []
    for n in range(number - window, number + window + 1):
        neighbour = f"chunk_{n:04d}"
        record = metadata.get(neighbour)
        if not record or record["source_document"] != home:
            continue
        path = PATHS.chunks / f"{neighbour}.txt"
        if path.exists():
            marker = " (requested)" if neighbour == chunk_id else ""
            parts.append(f"--- {neighbour}{marker} | {record['source_document']} ---\n"
                         f"{path.read_text(encoding='utf-8')}")
    return "\n\n".join(parts)


log4.info("Stage 4b chunking functions ready.")

In [ ]:
# ============================================================================
# Stage 4 — Run
# ============================================================================

def run_stage4(cfg: Stage4Config) -> Dict[str, Any]:
    log4.info("=" * 70)
    log4.info("STAGE 4: CONSOLIDATION AND CHUNKING")
    log4.info("=" * 70)

    paths = discover_converted()
    if not paths:
        log4.error("Nothing in data/converted - run Stage 3 first.")
        return {"error": "no converted documents"}

    ordered = order_documents(paths, cfg)
    log4.info(f"Consolidating {len(ordered)} documents ({cfg.sort_order} order)")
    for p in ordered[:10]:
        log4.info(f"  {p.name}")
    if len(ordered) > 10:
        log4.info(f"  ... and {len(ordered) - 10} more")

    corpus, index = consolidate(ordered, cfg)

    if cfg.create_backup and PATHS.consolidated_text.exists():
        backup = PATHS.consolidated_text.with_suffix(".txt.backup")
        shutil.copy2(PATHS.consolidated_text, backup)
        log4.info(f"Previous corpus backed up to {backup.name}")

    PATHS.consolidated_text.write_text(corpus, encoding="utf-8")
    PATHS.document_index.write_text(
        json.dumps({"documents": index}, indent=2, ensure_ascii=False),
        encoding="utf-8")

    issues = validate_corpus(corpus, index, cfg)
    for issue in issues:
        log4.warning(f"  validation: {issue}")
    if not issues:
        log4.info("  validation: clean")

    chunk_stats = build_chunks_and_metadata(corpus, index, cfg)

    words = len(corpus.split())
    summary = {
        "documents": len(index),
        "characters": len(corpus),
        "words": words,
        "corpus_tokens": count_tokens(corpus),
        "megabytes": round(len(corpus.encode("utf-8")) / (1024 * 1024), 2),
        "figures_referenced": len(set(m.group(2)
                                      for m in FIGURE_MARKER_RE.finditer(corpus))),
        "by_source_type": {},
        "chunking": chunk_stats,
        "issues": issues,
    }
    for record in index:
        st = record["source_type"]
        summary["by_source_type"][st] = summary["by_source_type"].get(st, 0) + 1

    log4.info("-" * 70)
    log4.info(f"  documents  : {summary['documents']}  {summary['by_source_type']}")
    log4.info(f"  tokens     : {summary['corpus_tokens']:,}")
    log4.info(f"  chunks     : {chunk_stats['chunks']} "
              f"(median {chunk_stats['median_tokens']} tokens)")
    return summary


stage4_summary = run_stage4(stage4_cfg)

c = stage4_summary.get("chunking", {})
print()
print("=" * 70)
print("STAGE 4 REPORT")
print("=" * 70)
print(f"Documents              : {stage4_summary['documents']}  "
      f"{stage4_summary['by_source_type']}")
print(f"Corpus                 : {stage4_summary['words']:,} words / "
      f"{stage4_summary['corpus_tokens']:,} tokens / "
      f"{stage4_summary['megabytes']} MB")
print(f"Figures referenced     : {stage4_summary['figures_referenced']}")
print()
print(f"Chunks                 : {c.get('chunks', 0)}")
print(f"  token range          : {c.get('min_tokens')} - {c.get('median_tokens')} "
      f"(median) - {c.get('max_tokens')}")
print(f"  inside 1500-2000     : {c.get('in_target_band', 0)} / {c.get('chunks', 0)}"
      "   <-- the design's band; the coverage map's resolution")
print(f"  carrying figures     : {c.get('chunks_with_figures', 0)}")
print(f"  carrying timestamps  : {c.get('chunks_with_timestamps', 0)}")
if c.get("oversized"):
    print(f"  OVER max_chunk_tokens: {len(c['oversized'])}   "
          f"(indivisible blocks: {c['oversized'][:3]})")
if stage4_summary["issues"]:
    print("\nValidation issues:")
    for issue in stage4_summary["issues"]:
        print(f"  - {issue}")
print("=" * 70)

# Show one chunk's Layer 3 record and one neighbourhood, so the output of this
# stage is something you have actually looked at rather than trusted.
_meta = json.loads(PATHS.chunk_metadata.read_text(encoding="utf-8"))
if _meta:
    _with_fig = next((k for k, v in _meta.items() if v["figures"]), None)
    _sample = _with_fig or next(iter(_meta))
    print(f"\nSAMPLE chunk_metadata['{_sample}']:")
    _record = dict(_meta[_sample])
    _record["figures"] = [{"id": f["id"], "kind": f["kind"], "path": f["path"]}
                          for f in _record["figures"]]
    print(json.dumps(_record, indent=2)[:900])
    print(f"\nNeighbourhood of {_sample} (what Layer 3 will retrieve):")
    print(get_chunk_neighbourhood(_sample, window=1, metadata=_meta)[:600])

## Stage 5 — The Source Index

The design's *Files on Disk* puts `data/source_index/` — a Chroma collection of chunk embeddings —
alongside the chunks. Layer 3 opens it:

```python
client = chromadb.PersistentClient(path=str(data_dir / "source_index"))
self.collection = client.get_or_create_collection(
    name="source_chunks",
    embedding_function=SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2"),
)
```

…and then only ever calls `.query()` on it. `SourceMemory.find_related()` is the Context
Assembler's way of pulling in relevant material the TOC never assigned — the design's *"semantic
source expansion"*, which closes coverage gaps. But nothing in Pipelines B or C ever calls `.add()`
or `.upsert()` on that collection.

So if Pipeline A does not populate it, `find_related()` returns nothing, forever, silently. Every
section would be written only from its assigned chunks, and the failure would look like a slightly
thin book rather than a broken index.

Building it here is cheap: one embedding pass over a few thousand chunks, done once, reused by all
150 sections.

In [ ]:
# ============================================================================
# Stage 5 — Embedding the chunks into data/source_index/
# ============================================================================

def build_source_index(model_name: str = "all-MiniLM-L6-v2",
                       batch_size: int = 256,
                       rebuild: bool = False) -> Dict[str, Any]:
    """
    Populate the Chroma collection Layer 3 reads from.

    Skips cleanly if chromadb is not installed, so a corpus can still be
    inspected without it -- but says so loudly, because a missing index is a
    silent quality loss in Pipeline C rather than an error.
    """
    try:
        import chromadb
        from chromadb.utils.embedding_functions import (
            SentenceTransformerEmbeddingFunction)
    except ImportError:
        log4.warning("chromadb / sentence-transformers not installed.")
        log4.warning("  pip install chromadb sentence-transformers")
        log4.warning("  Without data/source_index, SourceMemory.find_related() "
                     "returns nothing and every section is written only from "
                     "the chunks the TOC assigned it.")
        return {"status": "skipped", "reason": "chromadb not installed"}

    if not PATHS.chunk_metadata.exists():
        return {"status": "skipped", "reason": "no chunk_metadata.json - run Stage 4"}

    metadata = json.loads(PATHS.chunk_metadata.read_text(encoding="utf-8"))
    if rebuild and PATHS.source_index.exists():
        shutil.rmtree(PATHS.source_index)

    client = chromadb.PersistentClient(path=str(PATHS.source_index))
    collection = client.get_or_create_collection(
        name="source_chunks",
        embedding_function=SentenceTransformerEmbeddingFunction(model_name=model_name),
    )

    ids, documents, metadatas = [], [], []
    for chunk_id, record in metadata.items():
        path = PATHS.chunks / f"{chunk_id}.txt"
        if not path.exists():
            continue
        ids.append(chunk_id)
        documents.append(path.read_text(encoding="utf-8"))
        # Chroma metadata values must be scalars, so figures are reduced to a
        # count here. The full records stay in chunk_metadata.json.
        metadatas.append({
            "source_document": record["source_document"] or "unknown",
            "source_type": record["source_type"],
            "doc_slug": record.get("doc_slug", ""),
            "token_count": int(record.get("token_count", 0)),
            "figure_count": len(record.get("figures", [])),
        })

    log4.info(f"Embedding {len(ids)} chunks with {model_name} ...")
    t0 = time.time()
    for i in tqdm(range(0, len(ids), batch_size), desc="Embedding chunks"):
        collection.upsert(ids=ids[i:i + batch_size],
                          documents=documents[i:i + batch_size],
                          metadatas=metadatas[i:i + batch_size])

    result = {
        "status": "ok",
        "collection": "source_chunks",
        "path": str(PATHS.source_index),
        "chunks_indexed": len(ids),
        "embedding_model": model_name,
        "elapsed_seconds": round(time.time() - t0, 1),
    }
    log4.info(f"source_index built: {len(ids)} chunks in "
              f"{result['elapsed_seconds']}s")
    return result


source_index_summary = build_source_index()

print("=" * 70)
print("STAGE 5 — SOURCE INDEX")
print("=" * 70)
for k, v in source_index_summary.items():
    print(f"  {k:18s}: {v}")

# Prove the thing Layer 3 depends on actually works.
if source_index_summary.get("status") == "ok":
    import chromadb
    _client = chromadb.PersistentClient(path=str(PATHS.source_index))
    _coll = _client.get_collection("source_chunks")
    _probe = _coll.query(query_texts=["how does attention work"], n_results=3)
    print("\nfind_related('how does attention work') would return:")
    for _cid, _md in zip(_probe["ids"][0], _probe["metadatas"][0]):
        print(f"  {_cid}  from {_md['source_document']}  ({_md['source_type']})")
print("=" * 70)

print()
print("=" * 70)
print("PIPELINE A COMPLETE")
print("=" * 70)
print("Artifacts for the rest of the system:")
print(f"  corpus          -> {PATHS.consolidated_text}")
print(f"  chunks          -> {PATHS.chunks}/chunk_0001.txt ...")
print(f"  chunk metadata  -> {PATHS.chunk_metadata}          (Layer 3)")
print(f"  source index    -> {PATHS.source_index}/           (Layer 3, Chroma)")
print(f"  figure store    -> {PATHS.figure_store}")
print(f"  figure crops    -> {PATHS.figures}/                (passed to the Writer)")
print(f"  document index  -> {PATHS.document_index}")
print(f"  transcripts     -> {PATHS.transcripts}/            (raw + clean)")
print("=" * 70)

### Honest Caveats

Things worth knowing before you trust a run of this pipeline.

**1. The layout parser is the floor, and we cannot raise it.** MinerU produced lines like *"I did a
deep dive in 201 I have been a bit stale"* — a sentence cut in half by column detection. Everything
here is downstream of that. Read a few hundred lines of `raw_consolidated_text.txt` by eye before
building a book on it; if the text is mangled, try `parse_method="ocr"` or a different parser.

**2. Whole-document description is batched, not literally one call.** The design says pass the
entire document and all its figures at once. Where the figure count fits `max_figures_per_call`
that is exactly what happens. Above it, the images are split across calls while **every call still
receives the whole marker text**, so cross-figure reasoning survives — but a document with 95
figures gets 12 calls, not one, and figure 4 and figure 90 are never in the same forward pass.

**3. The decorative filters are heuristics with real thresholds.** `min_image_width=96` will discard
a small but genuine inline diagram and keep a large but useless hero image. The Stage 3 report
prints how many items each filter removed, but the only real test is opening `data/figures/` and
looking.

**4. `DECORATIVE` depends on the model using it.** Too eager and real figures are lost; too
reluctant and we are back to describing signup widgets. Both are visible in the counts and neither
is visible in the corpus text.

**5. Transcript cleanup shifts character positions.** The timestamp index is built from the raw
transcript, and cleaning changes lengths. Affected documents are marked
`timestamp_index_exact: false`. Minute-level provenance survives; second-level does not. Set
`clean_transcripts=False` if you need exact timestamps more than clean text.

**6. Chunking here and Pipeline B's `TextChunker` now overlap.** The design lists chunking under
both Stage 4 and Pipeline B step 1. Stage 4 owns it, because `chunk_metadata.json` is a Layer 3
artifact and Layer 3 only reads. **Point Pipeline B at `data/chunks/` instead of re-chunking**, or
its chunk ids will not match the ids in `chunk_metadata.json` and Layer 3's coverage map will index
chunks that no longer exist. This is the seam to resolve when we take the Pipeline B section.

**7. Nothing here deduplicates across documents.** The same paper under two filenames appears twice,
and Pipeline B will build two sections from it. The Book Ledger's coverage map catches the symptom
later. A near-duplicate check over `document_index.json` would be a sensible addition.

**8. The assertions protect the index, not the text.** They prove `document_index.json` and
`chunk_metadata.json` describe the corpus we wrote. They cannot prove the corpus is any good.

---

### Summary Table of Every Technique in Pipeline A

| Technique | What it does | What it prevents |
|---|---|---|
| **Whisper with vocabulary priming** | Conditions decoding on the corpus's real terms | "Katie cache", "pie torch" — lectures unusable as source |
| **`condition_on_previous_text=False`** | Removes cross-window conditioning | Repetition loops on hour-long recordings |
| **Timestamped segments preserved** | Keeps speech's only structural signal | A chunker guessing where a topic ended |
| **Per-document parse directory** | Image crops land somewhere known and stay | Silent `[IMAGE: unavailable]` for every figure |
| **`img_path` resolution + `img_missing`** | Verifies every crop, counts the ones that fail | A corpus with no diagrams and no warning |
| **Retry on returned status** | Retries fire on real failures | `max_retries=3` that never retried once |
| **Thread-local parsers** | One parser per worker thread | Undefined behaviour from shared model state |
| **Parser return normalisation** | Accepts list / tuple / dict | Breaking on a raganything upgrade |
| **Drop-list for `discarded`** | Layout furniture never enters the corpus | 48% of items becoming `[UNKNOWN CONTENT TYPE:]` |
| **Size pre-filter** | Icons and buttons never reach the vision encoder | GPU time spent on an avatar |
| **Content-hash cache** | Identical crops described once | Paying 40 times for the same clap icon |
| **`text_with_markers` document view** | The whole document as one string, figures by id | Descriptions that cannot see each other |
| **Whole-document description** | All figures described in one context | "A flowchart with boxes and arrows" |
| **`generate_structured` + repair loop** | JSON schema output, re-prompted with the parse error | Throwing away a good generation over a stray brace |
| **`kind: decorative`** | A legitimate answer, recorded not deleted | Earnest analysis of a newsletter signup box |
| **Figure store + inline ids** | Crops kept, described, referenceable | The Writer reading summaries instead of diagrams |
| **Conservative transcript cleanup** | Fix terms and punctuation, nothing else | The cleanup pass quietly deleting content |
| **`[READING SLIDE]` marking** | Flags recitation vs explanation | Slide text weighted like a real explanation |
| **Length-ratio guard on cleanup** | Rejects a window that changed size | A "cleanup" that was actually a summary |
| **Cursor-measured offsets + assertion** | Positions come from the string itself | An index that points at the wrong text |
| **1,500–2,000 token chunks** | Keeps the coverage map fine-grained | A blunt repetition detector two pipelines later |
| **Per-document chunking** | No chunk straddles two sources | Provenance that is a guess rather than a fact |
| **Zero overlap** | Chunks are disjoint | Double-counting in the coverage map |
| **`get_chunk_neighbourhood()`** | Widens at read time instead | Handing the Writer a paragraph starting mid-thought |
| **`chunk_metadata.json`** | The Layer 3 record: source, type, timestamp, figures | Pipeline C with no provenance to ground on |
| **`data/source_index/`** | Chroma over the chunks, built once | `find_related()` returning nothing, silently, forever |
| **One config object per stage** | No shared `config` name | Stage 2 silently running on Stage 4's settings |
| **One `BOOK_MODEL` constant** | The same model describes and later writes | Figure text that does not speak the book's vocabulary |

---

### One-Paragraph Takeaway

Pipeline A looks like the boring part of the system, and it is where the most damage gets done,
because every one of its failures is silent. The previous version raised no exceptions and produced
a corpus that was half layout furniture, contained no diagrams, described a newsletter signup widget
with more care than any real figure, carried an offset index that pointed at the wrong text, and
stopped before producing the two artifacts — `chunk_metadata.json` and `source_index/` — that Layer
3 can only read and never create. The corrected pipeline runs the design's four stages plus the
index build, and the difference is not that it is more careful in general: it is that it **counts
what it throws away, asserts what it claims, keeps the crops and the timestamps the design says to
keep, describes every figure in the context of its whole document, and refuses to describe things
that are not worth describing.** A book is grounded in its sources one chunk at a time, and a chunk
that cannot name where it came from is not a source — it is a rumour.

# Autonomous Table of Contents Generation

## Pipeline B — Deciding What the Book Is

Pipeline A produced a corpus. Pipeline B reads it and decides **what the book actually is**: how many
chapters, which sections, in what order, and which source chunks each section gets written from.

It is a chain of eight small LLM passes:

| Step | Name | What it does |
|---|---|---|
| 1 | **Load chunks** | Take Pipeline A's chunks — do **not** re-chunk |
| 2 | **FineGrainedTagExtractor** | Per chunk: atomic concept tags **plus the relationships between them** |
| 3 | **TagNormalizer** | Collapse thousands of raw tags into one canonical vocabulary, keeping every alias |
| 4 | **ThemeDiscovery** | Cluster canonical tags into chapter-sized themes |
| 5 | **TagAssigner** | Map every tag to exactly one chapter |
| 6 | **SectionFormer** | Break chapters into sections, each with a title, tags, chunk ids and a word budget |
| 7 | **CurriculumOrderer** | Order everything so concepts appear after their prerequisites |
| 8 | **build_final_toc** | Write `toc.json` |

---

### The output Pipeline C consumes

The Writer reads a flat list of sections in exactly this shape:

```jsonc
{
  "section_id": "sec_07_02",
  "chapter_id": "ch07",
  "title": "Handling Tool Failures",
  "tags": ["tool-calling", "error-handling", "retry"],
  "chunk_ids": ["chunk_0211", "chunk_0212", "chunk_0098"],
  "estimated_word_count": 800
}
```

Every field is load-bearing. `chapter_id` is how the Context Assembler finds open promises targeting
this chapter. `tags` is how the ledger slice picks which concepts to inject. `chunk_ids` is how
Layer 3 pulls source material. And `estimated_word_count` is what the Editor's guard measures
±15% against — with no value there, every section silently defaults to 800 words and the length
check is measured against a number nobody chose.

---

### The cheapest good idea in the whole design

Pipeline B produces far more than `toc.json`, and the writer should keep all of it:

> *"This is the cheapest good idea in the whole design: the LLM bill for this work has already been
> paid, so spend the results twice."*

| Artifact | What it really is | What it is worth to the writer |
|---|---|---|
| `normalized_tags.json` | Canonical concept → every alias the sources use | Seeds the concept registry on day one, no cold start |
| `tag_relationships.json` | Which concept must be taught before which | Prerequisite checking in Pipeline C (failure #5) |
| `chunk_tags.json` | chunk → canonical tags | A retrieval index over the sources |

Concretely: before section 1 is written, the Book Ledger is **pre-loaded** with every canonical
concept and every alias, so the terminology-drift detector works from the very first section
instead of slowly learning the vocabulary as it goes.

```python
def seed_from_toc_pipeline(self, normalized_tags: dict, tag_relationships: dict):
    for canonical, raw_variants in normalized_tags.items():
        self._cache["concepts"][key] = {
            "canonical_name": canonical,
            "aliases": list(raw_variants),                       # <- normalized_tags.json
            "prerequisites": tag_relationships.get(canonical, []),# <- tag_relationships.json
            ...
        }
```

Both files are written in exactly the shape that function reads.

---

### What was wrong before, and what changed

The clustering machinery here was sound. What was broken was the plumbing on both sides of it.

| # | What went wrong | Why it mattered | Fix |
|---|---|---|---|
| 1 | **Read `./raw_text.txt`, not Pipeline A's corpus** | The two pipelines were never connected | Reads `data/raw_consolidated_text.txt` |
| 2 | **Re-chunked from scratch** into its own `toc_output/chunks/` | Its `chunk_ids` could never match `chunk_metadata.json`, so Layer 3 would index chunks that do not exist | Loads `data/chunks/` — chunking happens once, in Stage 4 |
| 3 | **`overlap_tokens = 100`** in its chunker | Overlapping chunks double-count in the coverage map: "already used in sec_04_01" fires twice for one passage | Gone with the re-chunking |
| 4 | **No relationships extracted.** The extractor returned tags only | The Book Ledger's `prerequisites` field stayed empty, so Pipeline C's Continuity Gate check is dead code → failure #5 uncovered | Extractor returns `relationships` alongside `tags` |
| 5 | **`canonical_to_variations` computed, then dropped** | The alias list *is* the drift detector. It was calculated and thrown away | Written to `normalized_tags.json` |
| 6 | **`run_step3_normalization()` existed but nothing called it** | The function that saved the §7 artifacts was orphaned; the live path called the normalizer directly and checkpointed 2 of its 4 return values | One path, wired in |
| 7 | **Ordering could silently drop a chapter** | An id the model omitted from `ordered_ids` vanished from the book | The returned order is reconciled against the input |
| 8 | **No `estimated_word_count` anywhere** | The Editor's ±15% guard measured against a default nobody chose | Allocated from an explicit page target |
| 9 | **~24–48 sections** (6–12 chapters × 2–4 sections) | The design's arithmetic calls for **120–250** sections. This is a 30-page booklet | Section budget derived from target pages, allocated across chapters |
| 10 | **Three levels** (chapter → section → subsection) | Pipeline C reads `chapter_id` + `section_id`. There is no third level | Flattened: subsections become sections |
| 11 | **Nested `toc_final.json`** with `id`/`parent_id` | Not the shape the Writer, Assembler or Continuity Gate read | Flat `sections` list with the design's exact field names |
| 12 | **Different model** (`gpt-oss-20b`, 8K context) | Breaks *"the model that tags is the model that writes"* | `BOOK_MODEL`, one constant, shared with Stage 3 |
| 13 | **`quick_test_mode = True`** in the run cell | A "full run" quietly processed 20 chunks | Off by default, and the report prints how many chunks were used |
| 14 | **A section could inherit hundreds of chunks** | `list(set(...))` over every tag's chunks; the design's example has three | Chunks ranked by tag overlap, top-K kept, count reported |

## Step B0 — Setup

Two things get fixed before anything runs: **the model** and **the size of the book**.

### The model

The design specifies one model for the whole system, and Stage 3 already defined it as `BOOK_MODEL`.
Pipeline B does not load a second one — it **reuses the model Stage 3 already has in memory**:

```python
existing = globals().get("book_model")
if existing is not None and getattr(existing, "model", None) is not None:
    logB.info(f"Reusing the model Stage 3 already loaded: {existing.model_id}")
    return existing
```

That is the design's *"one model for the whole pipeline"* taken literally: the same weights that
described the diagrams are now reading the chunks and naming the chapters. It also saves a
multi-gigabyte reload. If Stage 3's cells were not run in this session, a fresh copy is loaded and a
warning says so.

The previous implementation hardcoded gpt-oss "harmony" prompt markup:

```python
prompt_parts = [
    f"<|start|>system<|message|>{system_content}<|end|>",
    f"<|start|>developer<|message|>{system_prompt}<|end|>",
]
```

Those control tokens are specific to one model family. Feed them to any other model and it does not
error — it reads them as literal text and answers a question with angle brackets in it. Prompts now
go through `BookModel._chat()`, which renders them with the tokenizer's own chat template, so
changing `BOOK_MODEL` changes the formatting with it.

**Generation is sequential.** One `model.generate()` per prompt, on the transformers runtime. That
is simple and it is the same code path Stage 3 uses, but it is not fast: tagging a few thousand
chunks is a few thousand forward passes. Two things follow from that, and both are handled below —
**incremental checkpointing** during extraction, so a crash at chunk 2,000 does not repay 2,000
calls, and a `quick_test_mode` that is honest about being a test.

### The size of the book

This is the part with no equivalent in the old code, and it decides the shape of everything
downstream. The design does the arithmetic in §1:

| Quantity | Value |
|---|---|
| Target book | 250–500 pages |
| Words per page (technical, with code) | ~350 |
| Total words | ~87,000 – 175,000 |
| Words per section | 600–800 |
| **Sections to write** | **~120 – 250** |

So the section count is not a style preference — it falls out of the page target. We declare the
target, derive the section budget, and then **allocate that budget across chapters in proportion to
how many concepts each chapter received**. A chapter with 90 tags earns more sections than one with
12.

In [ ]:
# ============================================================================
# Step B0.1 — Imports and configuration
# ============================================================================
"""
Pipeline B reads what Pipeline A produced. It does not re-chunk, it does not
invent its own directory layout, and it does not load its own model -- `PATHS`,
`BOOK_MODEL` and the loaded `book_model` from Stage 3 are all reused.
"""

import os
import re
import json
import time
import math
import logging
import traceback
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional, Any

from tqdm.auto import tqdm

if "PATHS" not in globals():
    raise RuntimeError(
        "Run the Document Preprocessing section first (Stage 0 defines PATHS, "
        "BOOK_MODEL and count_tokens)."
    )


def make_logger_b(name: str, logfile: str) -> logging.Logger:
    lg = logging.getLogger(name)
    lg.setLevel(logging.INFO)
    lg.handlers.clear()
    lg.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(name)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    for handler in (logging.FileHandler(logfile, encoding="utf-8"),
                    logging.StreamHandler()):
        handler.setFormatter(fmt)
        lg.addHandler(handler)
    return lg


logB = make_logger_b("pipelineB.toc", "pipelineB_toc.log")


@dataclass
class BookSizeConfig:
    """
    The design's arithmetic, made explicit.

        "A book of this size is not one piece of writing. Let us do the
         arithmetic, because the whole design falls out of it."

    Everything below -- how many chapters, how many sections per chapter, how
    many words each section is budgeted -- is derived from `target_pages`.
    Change that one number and the whole TOC resizes.
    """
    target_pages: int = 350            # design: 250-500
    words_per_page: int = 350          # technical prose with code
    words_per_section: int = 700       # design: 600-800

    min_section_words: int = 600
    max_section_words: int = 900

    target_chapters_min: int = 8
    target_chapters_max: int = 14
    min_sections_per_chapter: int = 4
    max_sections_per_chapter: int = 24

    @property
    def total_words(self) -> int:
        return self.target_pages * self.words_per_page

    @property
    def target_sections(self) -> int:
        return max(1, round(self.total_words / self.words_per_section))


@dataclass
class PipelineBConfig:
    """Configuration for Pipeline B - TOC generation."""

    # -- Model ---------------------------------------------------------------
    # No model settings beyond the id: Pipeline B reuses the transformers model
    # Stage 3 loaded, with its dtype and attention backend already resolved.
    model_name: str = BOOK_MODEL
    temperature: float = 0.3
    structured_max_attempts: int = 3

    # -- Extraction ----------------------------------------------------------
    min_tags_per_chunk: int = 10       # these are now actually USED in the prompt
    max_tags_per_chunk: int = 20
    max_relationships_per_chunk: int = 8
    max_tokens_extraction: int = 1_400

    # Generation is sequential, so extraction over a few thousand chunks is the
    # long pole of the whole pipeline. Save partial results this often, so a
    # crash costs minutes rather than the whole step.
    extraction_checkpoint_every: int = 50

    # -- Normalization -------------------------------------------------------
    normalization_batch_size: int = 100
    max_tokens_normalization: int = 2_400

    # -- Clustering ----------------------------------------------------------
    theme_sample_size: int = 200
    assignment_batch_size: int = 60
    max_tokens_structure: int = 3_000

    # -- Chunk mapping -------------------------------------------------------
    # The design's example section carries THREE chunk ids. Taking every chunk
    # that shares any tag would hand a section hundreds of them and blow the
    # Context Assembler's 60K source budget.
    max_chunks_per_section: int = 8
    min_chunks_per_section: int = 1

    # -- Book size -----------------------------------------------------------
    size: BookSizeConfig = field(default_factory=BookSizeConfig)

    # -- Housekeeping --------------------------------------------------------
    # Off by default. The old notebook shipped with this True, so a "full run"
    # silently processed 20 chunks.
    quick_test_mode: bool = False
    quick_test_chunks: int = 20
    resume: bool = True


bcfg = PipelineBConfig()

logB.info("Pipeline B configuration:")
logB.info(f"  model            : {bcfg.model_name}")
logB.info(f"  target pages     : {bcfg.size.target_pages}")
logB.info(f"  → total words    : {bcfg.size.total_words:,}")
logB.info(f"  → target sections: {bcfg.size.target_sections}"
          f"  (design band: 120-250)")
logB.info(f"  quick test mode  : {bcfg.quick_test_mode}")

if not 120 <= bcfg.size.target_sections <= 250:
    logB.warning(f"  target_sections={bcfg.size.target_sections} is outside the "
                 f"design's 120-250 band - check target_pages")

In [ ]:
# ============================================================================
# Step B0.2 — The model wrapper
# ============================================================================
"""
This is not a second model. It is a thin adapter that gives Pipeline B's steps
a `(system, user)` calling convention over the SAME transformers model Stage 3
loaded -- so the model that described the diagrams is the model that names the
chapters, exactly as the design intends.

    "The model that describes a diagram during preprocessing is the same model
     that later writes the chapter about it. That alone does more for
     consistency than any amount of prompt tuning."

It also inherits Stage 3's careful loading: the resolved dtype keyword, the
attention-backend ladder, and the audible fallback if BOOK_MODEL itself could
not be pulled.
"""


class BookLLM:
    """Pipeline B's interface to the book's model."""

    def __init__(self, cfg: PipelineBConfig):
        self.cfg = cfg
        self.model = self._acquire()
        self.model_id = self.model.model_id
        self.calls = 0
        self.structured_repairs = 0
        self.structured_failures = 0

    # ---------------------------------------------------------------- setup --
    def _acquire(self):
        """Reuse Stage 3's model if it is still in memory; otherwise load one."""
        existing = globals().get("book_model")
        if existing is not None and getattr(existing, "model", None) is not None:
            logB.info(f"Reusing the model Stage 3 already loaded: {existing.model_id}")
            return existing

        if "BookModel" not in globals() or "Stage3Config" not in globals():
            raise RuntimeError(
                "BookModel is not defined. Run the Document Preprocessing "
                "section (Stage 3) before Pipeline B.")

        logB.warning("Stage 3's model is not in memory - loading a fresh copy of "
                     f"{self.cfg.model_name}. Running the Stage 3 cells first "
                     "avoids a second multi-gigabyte load.")
        return BookModel(Stage3Config(model_name=self.cfg.model_name))

    # ------------------------------------------------------------ generation --
    def generate(self, system: str, user: str, max_tokens: int = 1_024,
                 temperature: Optional[float] = None, images=None) -> str:
        """
        One turn. Prompts are rendered by the model's own chat template.

        `images` (PIL objects) pass straight through to the multimodal model.
        Pipeline B never uses this; Pipeline C's Writer does, to attach the
        actual figure crops next to their descriptions.
        """
        temperature = self.cfg.temperature if temperature is None else temperature
        self.calls += 1
        return self.model.generate(prompt=user, system=system, images=images,
                                   max_new_tokens=max_tokens,
                                   temperature=temperature)

    # ------------------------------------------------------------ structured --
    def generate_structured(self, system: str, user: str, schema: Dict[str, Any],
                            max_tokens: int = 2_048,
                            temperature: float = 0.0) -> Optional[Dict]:
        """
        One JSON object matching `schema`, with the same repair loop Stage 3
        uses: on a parse failure, re-prompt with the parser's own error.

        "braces never balanced - the reply was probably truncated" is a far more
        useful instruction than "try again", and it usually succeeds on the
        second attempt. Throwing away a good generation over a stray brace is an
        expensive way to lose a chunk's tags.

        Returns None once the attempts are exhausted. Callers must handle None --
        a missing result is recoverable, a fabricated one is not.
        """
        instruction = ("\n\nReply with ONE JSON object and nothing else: no prose, "
                       "no markdown fence, no explanation.\nIt must match this shape "
                       f"exactly:\n{json.dumps(schema, indent=2)}")
        repair = None

        for attempt in range(1, self.cfg.structured_max_attempts + 1):
            message = user + instruction
            if repair:
                message += (f"\n\nYOUR PREVIOUS REPLY COULD NOT BE PARSED: {repair}\n"
                            f"Reply with only the JSON object.")

            raw = self.generate(system, message, max_tokens=max_tokens,
                                temperature=temperature)
            obj, reason = extract_json_object(raw)     # defined in Stage 3
            if obj is not None:
                missing = [k for k in schema if k not in obj]
                if not missing:
                    return obj
                reason = f"missing required key(s): {missing}"

            logB.warning(f"  structured attempt {attempt}: {reason}")
            self.structured_repairs += 1
            repair = reason

        self.structured_failures += 1
        return None


book_llm = BookLLM(bcfg)
print(f"Pipeline B model: {book_llm.model_id}")
if book_llm.model_id != BOOK_MODEL:
    print(f"  NOTE: the design specifies {BOOK_MODEL}")

In [ ]:
# ============================================================================
# Step B0.3 — Checkpoints
# ============================================================================
"""
Every step writes a checkpoint keyed by name, so a run that dies at step 6 does
not repay the LLM bill for steps 1-5. Kept deliberately simple: JSON on disk,
one file per step, under Pipeline A's checkpoint directory.
"""


class StepCheckpoints:
    def __init__(self, directory: Path, enabled: bool = True):
        self.dir = directory
        self.dir.mkdir(parents=True, exist_ok=True)
        self.enabled = enabled

    def _path(self, name: str) -> Path:
        return self.dir / f"pipelineB_{re.sub(r'[^A-Za-z0-9_]', '_', name)}.json"

    def load(self, name: str) -> Optional[Any]:
        if not self.enabled:
            return None
        path = self._path(name)
        if not path.exists():
            return None
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            logB.info(f"  resumed '{name}' from checkpoint")
            return data
        except Exception as exc:
            logB.warning(f"  checkpoint '{name}' unreadable ({exc}); recomputing")
            return None

    def save(self, name: str, data: Any) -> None:
        try:
            self._path(name).write_text(
                json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        except Exception as exc:
            logB.error(f"  could not save checkpoint '{name}': {exc}")


ckpt = StepCheckpoints(PATHS.checkpoints, enabled=bcfg.resume)
logB.info(f"Checkpoints: {PATHS.checkpoints} (resume={bcfg.resume})")

## Step 1 — Load Pipeline A's Chunks

There is no chunking step here, and that is the point.

The previous version called its own `TextChunker` on `./raw_text.txt`, producing chunks numbered
independently in `toc_output/chunks/`. Those ids then flowed into the TOC's `chunk_ids`. Meanwhile
Stage 4 had produced `data/chunks/chunk_0001.txt` and `chunk_metadata.json` with its own numbering.
The two never matched, so Layer 3's coverage map would have indexed chunk ids that did not exist on
disk, and every provenance lookup would have missed.

Chunking happens **once**, in Stage 4, where the design puts it — because `chunk_metadata.json` is a
Layer 3 artifact and Layer 3 only ever reads. This step just loads what is already there, and
asserts that the metadata and the files agree.

It also brings the **provenance along for free**. A chunk here is not just text; it knows its source
document, its type, its timestamp if it came from a lecture, and which figures it contains.

In [ ]:
# ============================================================================
# Step 1 — Load chunks (no re-chunking)
# ============================================================================

def load_chunks_from_pipeline_a(cfg: PipelineBConfig) -> List[Dict[str, Any]]:
    """
    Load Stage 4's chunks and their Layer 3 metadata.

    Fails loudly on a mismatch rather than skipping: a chunk in the metadata
    with no file on disk means the corpus was rebuilt without re-chunking, and
    every id downstream would be wrong.
    """
    if not PATHS.chunk_metadata.exists():
        raise FileNotFoundError(
            f"{PATHS.chunk_metadata} not found. Run Stage 4 of Document "
            f"Preprocessing first - Pipeline B does not chunk.")

    metadata = json.loads(PATHS.chunk_metadata.read_text(encoding="utf-8"))
    chunks: List[Dict[str, Any]] = []
    missing: List[str] = []

    for chunk_id, record in metadata.items():
        path = PATHS.chunks / f"{chunk_id}.txt"
        if not path.exists():
            missing.append(chunk_id)
            continue
        chunks.append({
            "chunk_id": chunk_id,
            "text": path.read_text(encoding="utf-8"),
            "source_document": record.get("source_document"),
            "source_type": record.get("source_type"),
            "timestamp": record.get("timestamp"),
            "figures": record.get("figures", []),
            "token_count": record.get("token_count", 0),
        })

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} chunks are in chunk_metadata.json but not on disk "
            f"(e.g. {missing[:3]}). The corpus and the chunks are out of step - "
            f"re-run Stage 4.")

    chunks.sort(key=lambda c: c["chunk_id"])

    if cfg.quick_test_mode:
        chunks = chunks[:cfg.quick_test_chunks]
        logB.warning(f"QUICK TEST MODE: using only {len(chunks)} chunks. "
                     f"The resulting TOC is not a real TOC.")

    by_type = Counter(c["source_type"] for c in chunks)
    total_tokens = sum(c["token_count"] for c in chunks)

    logB.info(f"Loaded {len(chunks)} chunks from {PATHS.chunks}")
    logB.info(f"  corpus tokens : {total_tokens:,}")
    logB.info(f"  by source type: {dict(by_type)}")
    logB.info(f"  with figures  : {sum(1 for c in chunks if c['figures'])}")
    logB.info(f"  with timestamps: {sum(1 for c in chunks if c['timestamp'])}")

    return chunks


chunks = load_chunks_from_pipeline_a(bcfg)
print(f"Step 1: {len(chunks)} chunks loaded (chunking happened once, in Stage 4)")

## Step 2 — Tags *and Relationships*

> *"**FineGrainedTagExtractor** reads each chunk and extracts fine-grained topic tags, plus the
> relationships between them ('attention is a prerequisite for transformers')."*

The tag half of this was already excellent. The extraction prompt is kept **verbatim** — it bans
broad tags explicitly (`"machine learning"`, `"CNN"`, `"optimization"`) and demands atomic ones
(`"max pooling"`, `"Xavier initialization"`), which is exactly what makes chunk retrieval precise
later.

The relationship half did not exist. The extractor returned `{"tags": [...], "chunk_summary": ...}`
and nothing else.

That absence is quiet but expensive, and the cost lands in **Pipeline C**, not here. Without
relationships, `tag_relationships.json` is never written, so the Book Ledger's `prerequisites` field
is empty for every concept — and the Continuity Gate's prerequisite check, *"does this section lean
on something not yet taught?"*, iterates an empty list forever and can never fire.

Pipeline B itself does nothing with these edges. It extracts them because it is already reading
every chunk, and because nothing downstream is in a position to.

So the extractor now returns three things per chunk:

```jsonc
{
  "tags": ["self-attention", "positional encoding", ...],
  "relationships": [
    {"from": "transformers", "to": "self-attention", "type": "requires"},
    {"from": "multi-head attention", "to": "self-attention", "type": "part_of"}
  ],
  "chunk_summary": "One sentence describing what this chunk teaches"
}
```

`requires` is the one that matters: **`from` cannot be understood without `to`.** That single edge
type is what lands in `tag_relationships.json` and becomes the Book Ledger's `prerequisites`.
`part_of` and `contrasts_with` are captured
because they are nearly free to ask for and useful to Pipeline B's clustering.

One more change: `min_tags_per_chunk` and `max_tags_per_chunk` were dead config — the prompt
hardcoded "10-20" and never read them. They are now interpolated into the prompt, so the knob
does what it says.

### Extraction is the slow step, so it checkpoints as it goes

Generation is sequential: one forward pass per chunk, a few thousand chunks. That is the long pole
of the entire pipeline, and it is the one place where losing progress genuinely hurts. So partial
results are written every `extraction_checkpoint_every` chunks, and a re-run skips whatever is
already done:

```python
todo = [c for c in chunks if c["chunk_id"] not in chunk_tags]
if len(todo) < len(chunks):
    logB.info(f"  resuming: {len(chunks) - len(todo)} chunks already done, "
              f"{len(todo)} remaining")
```

An interrupted job resumes where it stopped instead of re-paying for every call it already made.


In [ ]:
# ============================================================================
# Step 2 — Fine-grained tag AND relationship extraction
# ============================================================================

EXTRACTION_SCHEMA = {
    "tags": ["string - a specific, atomic concept"],
    "relationships": [{"from": "string - a tag", "to": "string - a tag",
                       "type": "requires|part_of|contrasts_with"}],
    "chunk_summary": "string - one sentence on what this chunk teaches",
}


class FineGrainedTagExtractor:
    """
    Extracts fine-grained atomic concepts AND the relationships between them.

    The tag half of the system prompt is preserved verbatim from the working
    implementation -- the bad/good examples are what keep the model off broad
    categories, and they were doing their job.
    """

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.stats = {"chunks": 0, "failed": 0, "tags": 0, "relationships": 0}

        self.system_prompt = f"""You are an expert knowledge engineer extracting SPECIFIC, ATOMIC concepts from educational content.

YOUR TASK: Extract fine-grained concepts that a student would need to learn, and the relationships between them. Think like an index at the back of a textbook.

CRITICAL RULES FOR GOOD TAGS:
1. Extract SPECIFIC concepts, not broad categories
2. Each tag should be a single learnable concept
3. Prefer concrete over abstract
4. Include techniques, algorithms, parameters, components
5. Extract {cfg.min_tags_per_chunk}-{cfg.max_tags_per_chunk} tags per chunk (more is better than fewer)

EXAMPLES OF BAD TAGS (too broad - NEVER use these):
- "machine learning" - too broad, extract specific ML concepts instead
- "deep learning" - too broad, extract specific DL concepts instead
- "neural networks" - too broad, extract specific NN concepts instead
- "CNN" - too broad, extract CNN components instead
- "NLP" - too broad, extract specific NLP techniques instead
- "optimization" - too broad, extract specific optimization methods instead

EXAMPLES OF GOOD TAGS (specific - USE these):
- "gradient descent", "learning rate", "batch size", "momentum"
- "convolutional layer", "max pooling", "stride", "padding", "kernel size"
- "backpropagation", "chain rule", "weight initialization", "Xavier initialization"
- "dropout regularization", "L2 regularization", "early stopping"
- "attention mechanism", "self-attention", "multi-head attention", "positional encoding"
- "LSTM cell", "forget gate", "input gate", "hidden state"

WHAT TO EXTRACT:
- Algorithms and methods (specific ones, not categories)
- Mathematical concepts (loss functions, activation functions, etc.)
- Architecture components (layers, gates, cells, etc.)
- Hyperparameters and settings (learning rate, batch size, etc.)
- Techniques and tricks (dropout, batch norm, residual connections, etc.)

WHAT NOT TO EXTRACT:
- Broad field names (ML, DL, AI, NLP, CV)
- Vague concepts ("training", "model", "data", "performance")
- Implementation details ("Python", "TensorFlow", "GPU")
- Meta-concepts ("introduction", "overview", "basics", "advanced")

RELATIONSHIPS -- this is the part that decides teaching order:
Report up to {cfg.max_relationships_per_chunk} relationships between the tags you extracted.
  "requires"        -- 'from' CANNOT be understood without 'to' first.
                       Example: {{"from": "transformers", "to": "self-attention", "type": "requires"}}
  "part_of"         -- 'from' is a component of 'to'.
                       Example: {{"from": "forget gate", "to": "LSTM cell", "type": "part_of"}}
  "contrasts_with"  -- the two are alternatives a reader may confuse.
                       Example: {{"from": "batch norm", "to": "layer norm", "type": "contrasts_with"}}

Only report a relationship if THIS TEXT supports it. Do not invent prerequisites from
general knowledge. Both 'from' and 'to' must be tags you extracted above.

Quality over quantity, but do not be too conservative."""

    def _user_message(self, chunk: Dict[str, Any]) -> str:
        return f"""Extract specific, atomic concepts and their relationships from this text chunk.

TEXT CHUNK (ID: {chunk['chunk_id']}):
\"\"\"
{chunk['text']}
\"\"\"

Remember:
- Extract {self.cfg.min_tags_per_chunk}-{self.cfg.max_tags_per_chunk} SPECIFIC concepts (not broad categories)
- Think: "What specific things would a student learn from this?"
- Then report which of those concepts REQUIRE which others."""

    def _absorb(self, chunk: Dict[str, Any], reply: Optional[Dict],
                chunk_tags: Dict, relationships: List, summaries: Dict) -> None:
        """Fold one model reply into the accumulating results."""
        self.stats["chunks"] += 1

        if not reply:
            self.stats["failed"] += 1
            chunk_tags[chunk["chunk_id"]] = []
            return

        tags = [str(t).strip().lower() for t in reply.get("tags", [])
                if isinstance(t, str) and t.strip()]
        tags = list(dict.fromkeys(tags))                  # dedupe, keep order
        chunk_tags[chunk["chunk_id"]] = tags
        summaries[chunk["chunk_id"]] = str(reply.get("chunk_summary", "")).strip()
        self.stats["tags"] += len(tags)

        # A relationship is only kept if BOTH endpoints are tags this chunk
        # actually extracted. Models will otherwise happily relate concepts that
        # appear nowhere in the text, and those invented prerequisites would
        # end up in the Book Ledger as facts about the material.
        tagset = set(tags)
        for rel in reply.get("relationships", []):
            if not isinstance(rel, dict):
                continue
            src = str(rel.get("from", "")).strip().lower()
            dst = str(rel.get("to", "")).strip().lower()
            kind = str(rel.get("type", "")).strip().lower()
            if (src in tagset and dst in tagset and src != dst
                    and kind in ("requires", "part_of", "contrasts_with")):
                relationships.append({"from": src, "to": dst, "type": kind,
                                      "chunk_id": chunk["chunk_id"]})
                self.stats["relationships"] += 1

    def extract_all(self, chunks: List[Dict[str, Any]],
                    ckpt: Optional["StepCheckpoints"] = None,
                    partial: Optional[Dict] = None
                    ) -> Tuple[Dict[str, List[str]], List[Dict], Dict[str, str]]:
        """
        Returns (chunk_tags, relationships, chunk_summaries).

        Generation is sequential -- one forward pass per chunk -- so this is the
        long pole of the whole pipeline. Partial results are written every
        `extraction_checkpoint_every` chunks and picked up on the next run, so an
        interrupted job resumes from where it stopped instead of re-paying for
        every call it already made.
        """
        chunk_tags: Dict[str, List[str]] = dict((partial or {}).get("chunk_tags", {}))
        relationships: List[Dict] = list((partial or {}).get("relationships", []))
        summaries: Dict[str, str] = dict((partial or {}).get("summaries", {}))

        todo = [c for c in chunks if c["chunk_id"] not in chunk_tags]
        if len(todo) < len(chunks):
            logB.info(f"  resuming: {len(chunks) - len(todo)} chunks already done, "
                      f"{len(todo)} remaining")

        for done, chunk in enumerate(tqdm(todo, desc="Extracting tags + relationships"), 1):
            reply = self.llm.generate_structured(
                self.system_prompt, self._user_message(chunk), EXTRACTION_SCHEMA,
                max_tokens=self.cfg.max_tokens_extraction, temperature=0.2)
            self._absorb(chunk, reply, chunk_tags, relationships, summaries)

            if ckpt and done % self.cfg.extraction_checkpoint_every == 0:
                ckpt.save("step2_extraction_partial",
                          {"chunk_tags": chunk_tags, "relationships": relationships,
                           "summaries": summaries})

        return chunk_tags, relationships, summaries


def run_step2_extraction(chunks, llm, cfg, ckpt):
    cached = ckpt.load("step2_extraction")
    if cached:
        return cached["chunk_tags"], cached["relationships"], cached["summaries"]

    logB.info(">>> STEP 2: TAG AND RELATIONSHIP EXTRACTION")
    logB.info(f"  {len(chunks)} chunks, one generation each - this is the slow step")

    partial = ckpt.load("step2_extraction_partial")
    extractor = FineGrainedTagExtractor(llm, cfg)
    chunk_tags, relationships, summaries = extractor.extract_all(chunks, ckpt, partial)

    s = extractor.stats
    logB.info(f"  chunks processed : {s['chunks']}  ({s['failed']} failed)")
    logB.info(f"  raw tags         : {s['tags']}  "
              f"({s['tags'] / max(1, s['chunks']):.1f} per chunk)")
    logB.info(f"  relationships    : {s['relationships']}")
    if s["failed"]:
        logB.warning(f"  {s['failed']} chunks produced no tags and will never be "
                     f"assigned to a section")

    ckpt.save("step2_extraction", {"chunk_tags": chunk_tags,
                                   "relationships": relationships,
                                   "summaries": summaries})
    return chunk_tags, relationships, summaries


print("Step 2 defined: tags AND relationships")

## Step 3 — Normalization, and the Three Artifacts Worth Keeping

The normalizer collapses thousands of raw tags into one canonical vocabulary: `"ReLU"`,
`"rectified linear unit"` and `"relu activation"` become one concept with three aliases.

That much already worked. What did not work is what happened to the results.

`normalize_in_batches()` returns **four** things. The live pipeline captured two:

```python
tag_to_canonical, canonical_to_variations = normalizer.normalize_in_batches(...)
...
checkpoint_mgr.save_checkpoint('step3_normalization_v2', {
    'normalized_chunk_tags': normalized_chunk_tags,
    'tag_to_canonical': tag_to_canonical,
})       # canonical_to_variations is never referenced again
```

`canonical_to_variations` **is the alias list**. It is the entire terminology-drift detector, it was
computed at real cost, and it was dropped on the floor. There was even a complete
`run_step3_normalization()` in the notebook that wrote it to disk properly — and nothing ever called
it.

So this step now writes the three files §7 depends on:

| File | Shape | Read by |
|---|---|---|
| `data/normalized_tags.json` | `{canonical: [alias, alias, …]}` | `BookLedger.seed_from_toc_pipeline()` → `aliases` |
| `data/tag_relationships.json` | `{canonical: [prerequisite, …]}` | `seed_from_toc_pipeline()` → `prerequisites` |
| `data/chunk_tags.json` | `{chunk_id: [canonical, …]}` | Layer 3's retrieval index |

Both of the first two are written in **exactly** the shape that function iterates:

```python
for canonical, raw_variants in normalized_tags.items():
    self._cache["concepts"][key] = {
        "aliases": list(raw_variants),
        "prerequisites": tag_relationships.get(canonical, []),
        ...
    }
```

Relationships are normalized too, which matters more than it sounds: an edge extracted as
*"transformers requires self-attention"* and another as *"transformer architecture requires
scaled dot-product attention"* collapse into one edge with a support count of 2. Without that
merge, the ledger would inherit near-duplicate prerequisites for what is really one concept.

In [ ]:
# ============================================================================
# Step 3 — Normalization + the artifacts that seed the Book Ledger
# ============================================================================

NORMALIZATION_SCHEMA = {
    "groups": [{"canonical": "string - the preferred form",
                "variants": ["string - every raw tag that means this"]}]
}


class TagNormalizer:
    """Collapses raw tags into a canonical vocabulary, keeping every alias."""

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.system_prompt = """You are an expert knowledge engineer building a controlled vocabulary.

YOUR TASK: Group tags that mean the SAME concept under one canonical name.

RULES:
1. Group only true synonyms and spelling/casing variants of the SAME concept
2. Do NOT group related-but-different concepts
   - "self-attention" and "multi-head attention" are DIFFERENT - do not merge
   - "relu" and "sigmoid" are DIFFERENT - do not merge
   - "ReLU", "relu activation", "rectified linear unit" are the SAME - merge these
3. The canonical name should be the clearest, most standard form, lowercase
4. Every input tag must appear in exactly one group
5. A tag with no synonyms forms a group of one

Be conservative. Wrongly merging two concepts is far worse than leaving them separate."""

    def collect_unique(self, chunk_tags: Dict[str, List[str]]) -> List[Tuple[str, int]]:
        counts = Counter(tag for tags in chunk_tags.values() for tag in tags)
        return counts.most_common()

    def normalize(self, ranked: List[Tuple[str, int]]
                  ) -> Tuple[Dict[str, str], Dict[str, List[str]]]:
        """
        Returns (tag_to_canonical, canonical_to_variants).

        Tags are batched most-frequent-first so the common vocabulary is settled
        before the long tail arrives, and later batches are shown the canonicals
        already chosen so they can join an existing group instead of starting a
        near-duplicate one.
        """
        tag_to_canonical: Dict[str, str] = {}
        canonical_to_variants: Dict[str, List[str]] = defaultdict(list)

        size = self.cfg.normalization_batch_size
        batches = [ranked[i:i + size] for i in range(0, len(ranked), size)]

        for batch in tqdm(batches, desc="Normalizing tags"):
            known = sorted(canonical_to_variants.keys())[:120]
            user = "TAGS TO GROUP (with corpus frequency):\n"
            user += "\n".join(f"  {tag}  ({count})" for tag, count in batch)
            if known:
                user += ("\n\nCANONICAL NAMES ALREADY CHOSEN — reuse one of these if a "
                         "tag below means the same thing:\n" + ", ".join(known))
            user += "\n\nGroup every tag above."

            reply = self.llm.generate_structured(
                self.system_prompt, user, NORMALIZATION_SCHEMA,
                max_tokens=self.cfg.max_tokens_normalization, temperature=0.0)

            handled = set()
            if reply:
                for group in reply.get("groups", []):
                    canonical = str(group.get("canonical", "")).strip().lower()
                    if not canonical:
                        continue
                    variants = [str(v).strip().lower()
                                for v in group.get("variants", []) if str(v).strip()]
                    for variant in variants:
                        tag_to_canonical[variant] = canonical
                        if variant != canonical and variant not in canonical_to_variants[canonical]:
                            canonical_to_variants[canonical].append(variant)
                        handled.add(variant)
                    canonical_to_variants[canonical]  # touch, so singletons exist

            # Any tag the model skipped becomes its own canonical. Silently
            # losing a tag here would silently lose whatever it points at.
            for tag, _ in batch:
                if tag not in handled:
                    tag_to_canonical[tag] = tag
                    canonical_to_variants[tag]

        return tag_to_canonical, dict(canonical_to_variants)

    @staticmethod
    def apply_to_chunks(chunk_tags: Dict[str, List[str]],
                        mapping: Dict[str, str]) -> Dict[str, List[str]]:
        return {cid: list(dict.fromkeys(mapping.get(t, t) for t in tags))
                for cid, tags in chunk_tags.items()}

    @staticmethod
    def apply_to_relationships(relationships: List[Dict],
                               mapping: Dict[str, str]) -> List[Dict]:
        """
        Canonicalise both endpoints, drop self-loops, and merge duplicates while
        counting support.

        Support records how many chunks independently stated the same
        dependency -- useful when reading the ledger, and a cheap signal for
        telling a well-attested prerequisite from a one-off mention.
        """
        merged: Dict[Tuple[str, str, str], Dict] = {}
        for rel in relationships:
            src = mapping.get(rel["from"], rel["from"])
            dst = mapping.get(rel["to"], rel["to"])
            if src == dst:
                continue                       # became a self-loop after merging
            key = (src, dst, rel["type"])
            entry = merged.setdefault(key, {"from": src, "to": dst,
                                            "type": rel["type"], "support": 0,
                                            "chunk_ids": []})
            entry["support"] += 1
            if len(entry["chunk_ids"]) < 5:
                entry["chunk_ids"].append(rel.get("chunk_id"))
        return list(merged.values())


def run_step3_normalization(chunk_tags, relationships, llm, cfg, ckpt):
    """
    Normalize, then WRITE THE THREE ARTIFACTS. The writing is the point.
    """
    cached = ckpt.load("step3_normalization")
    if cached:
        normalized_chunk_tags = cached["normalized_chunk_tags"]
        normalized_relationships = cached["normalized_relationships"]
        canonical_to_variants = cached["canonical_to_variants"]
    else:
        logB.info(">>> STEP 3: TAG NORMALIZATION")
        normalizer = TagNormalizer(llm, cfg)

        ranked = normalizer.collect_unique(chunk_tags)
        logB.info(f"  unique raw tags: {len(ranked)}")

        tag_to_canonical, canonical_to_variants = normalizer.normalize(ranked)
        normalized_chunk_tags = normalizer.apply_to_chunks(chunk_tags, tag_to_canonical)
        normalized_relationships = normalizer.apply_to_relationships(
            relationships, tag_to_canonical)

        logB.info(f"  canonical concepts: {len(canonical_to_variants)}  "
                  f"(from {len(ranked)} raw tags)")
        logB.info(f"  relationships: {len(relationships)} → "
                  f"{len(normalized_relationships)} after merging")

        ckpt.save("step3_normalization", {
            "normalized_chunk_tags": normalized_chunk_tags,
            "normalized_relationships": normalized_relationships,
            "canonical_to_variants": canonical_to_variants,
        })

    # ---- the three artifacts §7 depends on ---------------------------------
    # 1. normalized_tags.json -> BookLedger aliases (drift detection, day one)
    PATHS.normalized_tags.write_text(
        json.dumps(canonical_to_variants, indent=2, ensure_ascii=False),
        encoding="utf-8")

    # 2. tag_relationships.json -> BookLedger prerequisites (failure #5)
    #    seed_from_toc_pipeline reads `tag_relationships.get(canonical, [])`,
    #    so the shape is {concept: [things it requires]}.
    prerequisites: Dict[str, List[str]] = defaultdict(list)
    for rel in normalized_relationships:
        if rel["type"] == "requires" and rel["to"] not in prerequisites[rel["from"]]:
            prerequisites[rel["from"]].append(rel["to"])
    PATHS.tag_relationships.write_text(
        json.dumps(dict(prerequisites), indent=2, ensure_ascii=False),
        encoding="utf-8")

    # 3. chunk_tags.json -> Layer 3's retrieval index
    PATHS.chunk_tags.write_text(
        json.dumps(normalized_chunk_tags, indent=2, ensure_ascii=False),
        encoding="utf-8")

    logB.info(f"  wrote {PATHS.normalized_tags.name}: "
              f"{len(canonical_to_variants)} concepts, "
              f"{sum(len(v) for v in canonical_to_variants.values())} aliases")
    logB.info(f"  wrote {PATHS.tag_relationships.name}: "
              f"{len(prerequisites)} concepts with prerequisites")
    logB.info(f"  wrote {PATHS.chunk_tags.name}: {len(normalized_chunk_tags)} chunks")

    return normalized_chunk_tags, normalized_relationships, canonical_to_variants


print("Step 3 defined: normalization + normalized_tags / tag_relationships / chunk_tags")

## Steps 4–7 — Clustering into Chapters and Sections

Four LLM passes turn a flat vocabulary into a curriculum. All four are the model's judgement — there
is no clustering algorithm, no graph, and no scoring function anywhere in this stage.

**Theme discovery** samples the most frequent canonical tags and asks for 8–14 mutually exclusive,
collectively exhaustive chapter themes. **Tag assignment** then places every single tag into exactly
one of them, in batches. Both prompts are preserved verbatim — they were well written and they work.

**Section formation** is where the design's arithmetic finally bites. The old prompt said:

> *"Each chapter should have 2–4 SECTIONS. Each section should have 2–4 SUBSECTIONS."*

Two problems. First, the third level does not exist downstream — Pipeline C reads `chapter_id` and
`section_id`, and nothing else. Second, 6–12 chapters × 2–4 sections is **24–48 sections**, when
the design's page arithmetic calls for **120–250**. That is a 30-page booklet.

So the structure is flattened to two levels, and each chapter is given an explicit **section quota**
computed from its share of the vocabulary:

```
target_sections  = total_words / words_per_section          # e.g. 122,500 / 700 = 175
chapter quota    = target_sections × (chapter's tags / all tags)
                   clamped, then rebalanced so the totals add up
```

A chapter with 90 concepts earns more sections than one with 12, and the book comes out the size it
was asked to be. This is arithmetic on a budget, not an algorithm on the content.

**Ordering** is the model's job too. It is shown every chapter with its title and a sample of its
concepts, and asked to sequence them foundational-to-advanced:

```
PRINCIPLES:
1. Prerequisites come before dependent topics
2. Foundational concepts before advanced applications
3. Theory before implementation
4. General before specific
5. Build complexity gradually
```

The only thing the code does afterwards is **list hygiene** — check that every id came back exactly
once, restore any the model dropped, discard any it invented. That is bookkeeping, not a second
opinion about pedagogy. The model has read every textbook in this field; the ordering is its call.

The same pass runs twice: once over chapters, once over the sections inside each chapter.

In [ ]:
# ============================================================================
# Steps 4-5 — Theme discovery and tag assignment
# ============================================================================

THEME_SCHEMA = {"themes": [{"id": "string", "title": "string", "description": "string",
                            "example_concepts": ["string"],
                            "difficulty_level": "foundational|intermediate|advanced"}],
                "reasoning": "string"}

ASSIGNMENT_SCHEMA = {"assignments": {"concept": "theme_id"},
                     "uncategorized": ["string"], "reasoning": "string"}


class ThemeDiscovery:
    """Phase 1: identify the chapter-sized themes. Prompt preserved verbatim."""

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.system_prompt = f"""You are an expert curriculum designer analyzing concepts to identify major learning themes.

YOUR TASK: Given a sample of concepts from a technical curriculum, identify the {cfg.size.target_chapters_min}-{cfg.size.target_chapters_max} major themes/chapters that would organize ALL similar concepts.

PRINCIPLES FOR GOOD THEMES:
1. Themes should be MUTUALLY EXCLUSIVE - a concept should clearly belong to one theme
2. Themes should be COLLECTIVELY EXHAUSTIVE - all concepts should fit somewhere
3. Themes should follow PEDAGOGICAL PROGRESSION - foundational to advanced
4. Themes should be at the right GRANULARITY - not too broad, not too narrow

EXAMPLE FOR MACHINE LEARNING CURRICULUM:
Good themes:
- "Mathematical Foundations" (linear algebra, calculus, probability)
- "Optimization & Training" (gradient descent, learning rate, convergence)
- "Neural Network Basics" (perceptrons, activation functions, backpropagation)
- "Attention & Transformers" (self-attention, positional encoding, BERT)
- "Regularization & Generalization" (dropout, batch norm, overfitting)

Bad themes:
- "Deep Learning" (too broad - should be split)
- "Advanced Topics" (too vague)
- "Part 1" (not descriptive)

Identify {cfg.size.target_chapters_min}-{cfg.size.target_chapters_max} themes that would create a well-structured curriculum."""

    def discover(self, chunk_tags: Dict[str, List[str]]) -> List[Dict]:
        counts = Counter(t for tags in chunk_tags.values() for t in tags)
        sample = [t for t, _ in counts.most_common(self.cfg.theme_sample_size)]

        user = (f"CONCEPTS FROM THE CORPUS ({len(counts)} total, "
                f"{len(sample)} most frequent shown):\n{', '.join(sample)}\n\n"
                f"Identify the major themes.")

        reply = self.llm.generate_structured(
            self.system_prompt, user, THEME_SCHEMA,
            max_tokens=self.cfg.max_tokens_structure, temperature=0.3)

        themes = (reply or {}).get("themes", [])
        cleaned = []
        for i, theme in enumerate(themes, 1):
            cleaned.append({
                "id": str(theme.get("id") or f"theme_{i:02d}"),
                "title": str(theme.get("title", f"Theme {i}")).strip(),
                "description": str(theme.get("description", "")).strip(),
                "difficulty_level": theme.get("difficulty_level", "intermediate"),
            })
        if not cleaned:
            raise RuntimeError("Theme discovery returned nothing - cannot build a TOC")
        return cleaned


class TagAssigner:
    """Phase 2: every concept into exactly one chapter. Prompt preserved verbatim."""

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.system_prompt = """You are an expert curriculum designer assigning concepts to chapters.

YOUR TASK: Assign each concept to the SINGLE most appropriate theme/chapter.

RULES:
1. Every concept MUST be assigned to exactly ONE theme
2. Choose the MOST SPECIFIC theme that fits
3. If a concept could fit multiple themes, choose the one where it's MOST CENTRAL
4. Use "UNCATEGORIZED" only for concepts that truly don't fit anywhere

Assign EVERY concept to a theme. Minimize uncategorized."""

    def assign(self, themes: List[Dict],
               chunk_tags: Dict[str, List[str]]) -> Dict[str, List[str]]:
        counts = Counter(t for tags in chunk_tags.values() for t in tags)
        all_tags = [t for t, _ in counts.most_common()]

        theme_block = "\n".join(
            f"  {t['id']}: \"{t['title']}\" - {t['description']}" for t in themes)
        valid_ids = {t["id"] for t in themes}

        assignments: Dict[str, str] = {}
        size = self.cfg.assignment_batch_size
        batches = [all_tags[i:i + size] for i in range(0, len(all_tags), size)]

        for batch in tqdm(batches, desc="Assigning tags to chapters"):
            user = (f"THEMES:\n{theme_block}\n\n"
                    f"CONCEPTS TO ASSIGN ({len(batch)}):\n{', '.join(batch)}\n\n"
                    f"Assign every concept above to one theme id.")
            reply = self.llm.generate_structured(
                self.system_prompt, user, ASSIGNMENT_SCHEMA,
                max_tokens=self.cfg.max_tokens_structure, temperature=0.2)

            for tag, theme_id in (reply or {}).get("assignments", {}).items():
                tag = str(tag).strip().lower()
                if tag in counts and str(theme_id) in valid_ids:
                    assignments[tag] = str(theme_id)

        # An unassigned tag is not a rounding error -- it is source material
        # that can never reach a section. Park it in the nearest chapter by
        # co-occurrence rather than dropping it, and report the count.
        unassigned = [t for t in all_tags if t not in assignments]
        if unassigned:
            logB.warning(f"  {len(unassigned)} concepts unassigned by the model; "
                         f"placing them by co-occurrence")
            cooccurrence = defaultdict(Counter)
            for tags in chunk_tags.values():
                for tag in tags:
                    if tag in unassigned:
                        for other in tags:
                            if other in assignments:
                                cooccurrence[tag][assignments[other]] += 1
            fallback = themes[0]["id"]
            for tag in unassigned:
                best = cooccurrence[tag].most_common(1)
                assignments[tag] = best[0][0] if best else fallback

        by_chapter: Dict[str, List[str]] = defaultdict(list)
        for tag, theme_id in assignments.items():
            by_chapter[theme_id].append(tag)
        return dict(by_chapter)


print("Steps 4-5 defined: theme discovery, tag assignment")

In [ ]:
# ============================================================================
# Step 6 — Section formation, with a real section budget
# ============================================================================

SECTION_SCHEMA = {"sections": [{"title": "string", "concepts": ["string"]}],
                  "reasoning": "string"}


def allocate_section_quota(tags_per_chapter: Dict[str, int],
                           size: BookSizeConfig) -> Dict[str, int]:
    """
    Split the book's section budget across chapters, in proportion to how much
    vocabulary each one received.

    This is where the design's page arithmetic actually lands. Without it, the
    section count is whatever the model felt like, and the book comes out at a
    fifth of the requested length.
    """
    total_tags = sum(tags_per_chapter.values()) or 1
    target = size.target_sections
    n_chapters = len(tags_per_chapter) or 1

    # The configured floor and ceiling are sanity caps, not hard truths. If the
    # chapter count and the section target disagree with them, the caps must
    # give way -- otherwise the loop below runs out of chapters to adjust and
    # returns a total that quietly misses the target. With 4 chapters, a
    # target of 175 and a ceiling of 24, that silently produced 96.
    # The ceiling scales with the book: capping at a fixed 24 when the mean is
    # already 17.5 flattens the distribution so hard that a chapter with 300
    # concepts and one with 120 get the same number of sections. Allow roughly
    # twice the mean, which keeps the cap meaningful without erasing the shape.
    ceiling = max(size.max_sections_per_chapter,
                  math.ceil(2.0 * target / n_chapters))
    floor = min(size.min_sections_per_chapter, max(1, target // n_chapters))

    quota = {}
    for chapter_id, n_tags in tags_per_chapter.items():
        share = target * n_tags / total_tags
        quota[chapter_id] = int(max(floor, min(ceiling, round(share))))

    # Rebalance towards the target: rounding and clamping push the total off, so
    # give or take sections from the chapters with the most vocabulary first.
    order = sorted(quota, key=lambda c: -tags_per_chapter[c])
    guard = 0
    while sum(quota.values()) != target and guard < 100_000:
        guard += 1
        if sum(quota.values()) > target:
            movable = [c for c in reversed(order) if quota[c] > floor]
            if not movable:
                break
            quota[movable[0]] -= 1
        else:
            movable = [c for c in order if quota[c] < ceiling]
            if not movable:
                break
            quota[movable[0]] += 1

    # If the caps still could not be reconciled with the target, say so. A book
    # that comes out a third of its requested length is not a rounding error.
    achieved = sum(quota.values())
    if achieved != target:
        logB.warning(f"  section budget: asked for {target}, allocated {achieved} "
                     f"across {n_chapters} chapters (floor={floor}, ceiling={ceiling}) "
                     f"- adjust target_pages or the per-chapter limits")

    return quota


class SectionFormer:
    """
    Phase 3: break each chapter into sections.

    Two levels only. The old version produced chapter -> section -> subsection,
    but Pipeline C reads `chapter_id` and `section_id` and there is no third
    level anywhere in the design. Subsections became sections.
    """

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.system_prompt = """You are an expert curriculum designer organizing concepts into sections.

YOUR TASK: Given the concepts belonging to a chapter, organize them into a specific number of sections.

PRINCIPLES:
1. Group related concepts together
2. Order sections from foundational to advanced within the chapter
3. Every concept must appear in exactly ONE section
4. Every section must contain at least one concept
5. Section titles should be specific and teachable, not "Introduction" or "Advanced Topics"

A section becomes roughly 700 words of a book -- one focused idea, taught once.
If you are asked for 12 sections, return exactly 12."""

    def form(self, chapter: Dict, tags: List[str], quota: int) -> List[Dict]:
        user = f"""CHAPTER: "{chapter['title']}"
DESCRIPTION: {chapter.get('description', 'N/A')}

CONCEPTS TO ORGANIZE ({len(tags)}):
{', '.join(sorted(tags))}

Create EXACTLY {quota} sections, ordered foundational to advanced.
Every concept above must appear in exactly one section."""

        reply = self.llm.generate_structured(
            self.system_prompt, user, SECTION_SCHEMA,
            max_tokens=self.cfg.max_tokens_structure, temperature=0.3)

        sections = []
        assigned = set()
        for i, section in enumerate((reply or {}).get("sections", []), 1):
            concepts = [str(c).strip().lower() for c in section.get("concepts", [])
                        if str(c).strip().lower() in set(tags)]
            concepts = [c for c in concepts if c not in assigned]
            assigned.update(concepts)
            title = str(section.get("title", "")).strip() or f"Section {i}"
            if concepts:
                sections.append({"title": title, "tags": concepts})

        # Concepts the model forgot go to the section they best match by
        # co-membership; if there is no section at all, make one.
        leftover = [t for t in tags if t not in assigned]
        if leftover:
            if not sections:
                sections.append({"title": chapter["title"], "tags": []})
            logB.debug(f"    {len(leftover)} concepts unplaced; appending")
            for tag in leftover:
                sections[-1]["tags"].append(tag)

        if len(sections) != quota:
            logB.warning(f"  '{chapter['title']}': asked for {quota} sections, "
                         f"got {len(sections)}")
        return sections


print("Step 6 defined: section formation with an explicit section budget")

In [ ]:
# ============================================================================
# Step 7 — Curriculum ordering
# ============================================================================

ORDER_SCHEMA = {"ordered_ids": ["string"], "reasoning": "string"}


class CurriculumOrderer:
    """
    Phase 4: order chapters, and the sections inside each chapter.

    Pure LLM. The model sees each unit's title and a sample of its concepts and
    returns them sequenced foundational-to-advanced. Its domain knowledge is the
    whole mechanism here -- nothing recomputes or second-guesses the sequence.

    The code's only job afterwards is list hygiene: models occasionally drop an
    id or hallucinate one, and a missing chapter would silently vanish from the
    book. So the returned list is reconciled against the input.
    """

    def __init__(self, llm: BookLLM, cfg: PipelineBConfig):
        self.llm = llm
        self.cfg = cfg
        self.notes: List[str] = []
        self.system_prompt = """You are an expert curriculum designer ordering content for optimal learning.

YOUR TASK: Order the items given from foundational to advanced.

PRINCIPLES:
1. Prerequisites come before dependent topics
2. Foundational concepts before advanced applications
3. Theory before implementation
4. General before specific
5. Build complexity gradually

Order by learning progression, not alphabetically or by size.
Return every id you were given, exactly once."""

    def order(self, units: List[Dict], label: str) -> List[Dict]:
        """Return `units` in the order the model chose."""
        if len(units) <= 1:
            return units

        listing = "\n".join(
            f"- {u['id']}: \"{u['title']}\"\n  concepts: "
            f"{', '.join(u['tags'][:10])}" for u in units)
        user = (f"Order these {len(units)} {label} for optimal learning progression:\n\n"
                f"{listing}\n\nReturn the ids in pedagogically correct order.")

        reply = self.llm.generate_structured(
            self.system_prompt, user, ORDER_SCHEMA,
            max_tokens=1_500, temperature=0.2)

        proposed = [str(i) for i in (reply or {}).get("ordered_ids", [])]
        known = {u["id"] for u in units}

        # -- list hygiene, not a second opinion ------------------------------
        invented = [p for p in proposed if p not in known]
        proposed = [p for p in proposed if p in known]
        seen, deduped = set(), []
        for p in proposed:                       # a repeated id would duplicate a chapter
            if p not in seen:
                seen.add(p)
                deduped.append(p)
        dropped = [u["id"] for u in units if u["id"] not in seen]
        final = deduped + dropped                # anything forgotten keeps its old place

        if invented or dropped:
            note = (f"{label}: model returned {len(invented)} unknown id(s) and omitted "
                    f"{len(dropped)}; reconciled against the input")
            logB.warning(f"  {note}")
            self.notes.append(note)
        if not reply:
            note = f"{label}: ordering call failed, keeping the original order"
            logB.warning(f"  {note}")
            self.notes.append(note)

        by_id = {u["id"]: u for u in units}
        return [by_id[i] for i in final]


print("Step 7 defined: curriculum ordering (LLM)")

## Step 8 — Building `toc.json`

The last step turns the ordered chapters and sections into the flat structure Pipeline C reads, and
does three things the old builder did not.

**It uses the design's field names and ids.** `section_id` / `chapter_id`, formatted as
`sec_07_02` / `ch07` — the exact shapes that appear throughout the design's Pipeline C code, in
`ContinuityGate.check_plan`, in `EditorAgent.smooth_section`, in `open_promises_for(chapter_id)`.

**It budgets words.** Each section gets an `estimated_word_count` derived from the book's total word
budget and weighted by how many concepts it teaches, clamped to the design's 600–900 band. Without
it the Editor's guard measures ±15% against a hardcoded default.

**It ranks chunks instead of dumping them.** The old builder did:

```python
section_chunks = []
for tag in section_tags:
    section_chunks.extend(tag_to_chunks.get(tag.lower(), []))
section_chunks = list(set(section_chunks))
```

A section with 8 tags, each appearing in 40 chunks, receives up to 320 chunk ids. The design's
example section carries **three**, and the Context Assembler budgets 60K tokens for source — so
those 320 chunks would be truncated arbitrarily at load time, and which material the section
actually saw would depend on dictionary ordering.

Chunks are now scored by how many of the section's tags they contain, tie-broken by how
*concentrated* those tags are in the chunk, and the top K kept. The rest of the corpus is still
reachable — `SourceMemory.find_related()` searches the whole `source_index` — but what the TOC
*assigns* is a deliberate handful.

### The sanity check

Before the TOC goes anywhere near Pipeline C, `validate_toc()` looks for the four things that make a
curriculum unusable, none of which need a model or any analysis of meaning:

| Check | Why it matters |
|---|---|
| Sections with **no chunk ids** | They would be written from nothing at all |
| Sections with **no tags** | The ledger slice has nothing to select on |
| **Duplicate section titles** | Two sections teaching the same thing, twice |
| Chapters with **no sections** | A chapter heading with nothing under it |

It is pure bookkeeping over the finished structure, and it costs nothing. The design's warning is
worth keeping in mind when reading its output:

> *"Nothing here fixes a bad table of contents… That is a signal to fix the TOC, not to keep
> writing."*

In [ ]:
# ============================================================================
# Step 8 — Build toc.json
# ============================================================================

def rank_chunks_for_tags(tags: List[str], tag_to_chunks: Dict[str, List[str]],
                         chunk_tags: Dict[str, List[str]],
                         cfg: PipelineBConfig) -> List[str]:
    """
    Pick the chunks a section should be written from.

    Score = how many of the section's tags the chunk contains.
    Tie-break = how concentrated those tags are in the chunk, so a focused chunk
    beats a sprawling one that mentions the topic in passing.
    """
    overlap: Counter = Counter()
    for tag in tags:
        for chunk_id in tag_to_chunks.get(tag, []):
            overlap[chunk_id] += 1

    def sort_key(item):
        chunk_id, score = item
        density = score / max(1, len(chunk_tags.get(chunk_id, [])))
        return (-score, -density, chunk_id)

    ranked = [cid for cid, _ in sorted(overlap.items(), key=sort_key)]
    return ranked[:cfg.max_chunks_per_section]


def allocate_word_counts(sections: List[Dict], size: BookSizeConfig) -> None:
    """
    Give every section a word budget, in place.

    Weighted by how many concepts a section teaches, so a section covering seven
    concepts is allowed to be longer than one covering two -- then clamped to the
    design's 600-800 band (900 ceiling for the densest) so no section becomes an
    unreadable slab.
    """
    if not sections:
        return
    mean_tags = sum(len(s["tags"]) for s in sections) / len(sections) or 1
    base = size.total_words / len(sections)

    for section in sections:
        weight = len(section["tags"]) / mean_tags if mean_tags else 1.0
        words = base * (0.6 + 0.4 * weight)          # damped, not proportional
        words = max(size.min_section_words, min(size.max_section_words, words))
        section["estimated_word_count"] = int(round(words / 25) * 25)

    # The clamp keeps any single section readable, but it also means that if the
    # pipeline produced far fewer sections than the budget assumed, every one of
    # them pins to the maximum and the book quietly comes out short. Say so.
    allocated = sum(s["estimated_word_count"] for s in sections)
    drift = abs(allocated - size.total_words) / max(1, size.total_words)
    if drift > 0.10:
        logB.warning(
            f"  word budget: {len(sections)} sections x ~{base:.0f} words wanted "
            f"{size.total_words:,}, but clamping to "
            f"{size.min_section_words}-{size.max_section_words} allocated "
            f"{allocated:,} ({allocated / size.words_per_page:.0f} pages vs "
            f"{size.target_pages} requested)")


def build_final_toc(ordered_chapters: List[Dict], chunk_tags: Dict[str, List[str]],
                    llm: BookLLM, cfg: PipelineBConfig) -> Dict[str, Any]:
    """
    Produce toc.json in the exact shape Pipeline C reads.

        {"section_id": "sec_07_02", "chapter_id": "ch07", "title": "...",
         "tags": [...], "chunk_ids": [...], "estimated_word_count": 800}
    """
    tag_to_chunks: Dict[str, List[str]] = defaultdict(list)
    for chunk_id, tags in chunk_tags.items():
        for tag in tags:
            tag_to_chunks[tag].append(chunk_id)

    chapters_out: List[Dict[str, Any]] = []
    sections_out: List[Dict[str, Any]] = []

    for chapter_number, chapter in enumerate(ordered_chapters, 1):
        chapter_id = f"ch{chapter_number:02d}"
        chapters_out.append({
            "chapter_id": chapter_id,
            "title": chapter["title"],
            "description": chapter.get("description", ""),
            "order": chapter_number,
            "section_count": len(chapter["sections"]),
        })

        for section_number, section in enumerate(chapter["sections"], 1):
            section_id = f"sec_{chapter_number:02d}_{section_number:02d}"
            sections_out.append({
                "section_id": section_id,
                "chapter_id": chapter_id,
                "title": section["title"],
                "tags": section["tags"],
                "chunk_ids": rank_chunks_for_tags(section["tags"], tag_to_chunks,
                                                  chunk_tags, cfg),
                "estimated_word_count": 0,        # filled in below
                "order": section_number,
            })

    allocate_word_counts(sections_out, cfg.size)

    # ---- book title -------------------------------------------------------
    titles = "\n".join(f"- {c['title']}" for c in chapters_out)
    try:
        title = llm.generate(
            "You are an expert at creating book titles. Reply with the title only.",
            f"Based on these chapter titles, give this book a concise, professional "
            f"title:\n\n{titles}\n\nReply with just the title.",
            max_tokens=40, temperature=0.3).strip().strip('"\'').split("\n")[0]
    except Exception as exc:
        logB.warning(f"Title generation failed: {exc}")
        title = "Technical Curriculum"

    used_chunks = {cid for s in sections_out for cid in s["chunk_ids"]}
    total_words = sum(s["estimated_word_count"] for s in sections_out)

    toc = {
        "book_title": title or "Technical Curriculum",
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "model": llm.model_id,
        "chapters": chapters_out,
        "sections": sections_out,
        "stats": {
            "chapters": len(chapters_out),
            "sections": len(sections_out),
            "target_sections": cfg.size.target_sections,
            "total_estimated_words": total_words,
            "estimated_pages": round(total_words / cfg.size.words_per_page),
            "chunks_assigned": len(used_chunks),
            "chunks_total": len(chunk_tags),
            "chunks_unused": len(chunk_tags) - len(used_chunks),
            "avg_chunks_per_section": round(
                sum(len(s["chunk_ids"]) for s in sections_out) /
                max(1, len(sections_out)), 1),
            "sections_with_no_chunks": sum(1 for s in sections_out
                                           if not s["chunk_ids"]),
        },
    }
    return toc


def validate_toc(toc: Dict[str, Any]) -> Dict[str, Any]:
    """
    Four structural checks over the finished TOC. No model, no analysis of
    meaning -- just the things that make a curriculum mechanically unusable.

        "Nothing here fixes a bad table of contents... That is a signal to fix
         the TOC, not to keep writing."
    """
    sections = toc["sections"]

    no_source = [s["section_id"] for s in sections if not s["chunk_ids"]]
    no_tags = [s["section_id"] for s in sections if not s["tags"]]

    seen: Dict[str, str] = {}
    duplicate_titles = []
    for section in sections:
        key = section["title"].strip().lower()
        if key in seen:
            duplicate_titles.append({"title": section["title"],
                                     "sections": [seen[key], section["section_id"]]})
        else:
            seen[key] = section["section_id"]

    with_sections = {s["chapter_id"] for s in sections}
    empty_chapters = [c["chapter_id"] for c in toc["chapters"]
                      if c["chapter_id"] not in with_sections]

    return {
        "sections": len(sections),
        "sections_without_source": no_source,
        "sections_without_tags": no_tags,
        "duplicate_titles": duplicate_titles,
        "chapters_without_sections": empty_chapters,
        "passed": not (no_source or no_tags or duplicate_titles or empty_chapters),
    }


print("Step 8 defined: toc.json + structural validation")

In [ ]:
# ============================================================================
# Pipeline B — Run
# ============================================================================

def run_pipeline_b(chunks: List[Dict], llm: BookLLM, cfg: PipelineBConfig,
                   ckpt: StepCheckpoints) -> Tuple[Dict, Dict]:
    """The eight steps, in order."""
    logB.info("=" * 70)
    logB.info("PIPELINE B: TABLE OF CONTENTS GENERATION")
    logB.info("=" * 70)
    started = time.time()

    # -- Steps 2 & 3 --------------------------------------------------------
    chunk_tags, relationships, summaries = run_step2_extraction(chunks, llm, cfg, ckpt)
    normalized_chunk_tags, normalized_relationships, aliases = run_step3_normalization(
        chunk_tags, relationships, llm, cfg, ckpt)

    # -- Steps 4-7 ----------------------------------------------------------
    cached = ckpt.load("step4to7_clustering")
    if cached:
        ordered_chapters = cached["ordered_chapters"]
    else:
        logB.info(">>> STEP 4: THEME DISCOVERY")
        themes = ThemeDiscovery(llm, cfg).discover(normalized_chunk_tags)
        logB.info(f"  {len(themes)} themes: {[t['title'] for t in themes]}")

        logB.info(">>> STEP 5: TAG ASSIGNMENT")
        tags_by_theme = TagAssigner(llm, cfg).assign(themes, normalized_chunk_tags)
        for theme in themes:
            logB.info(f"  {theme['id']}: {len(tags_by_theme.get(theme['id'], []))} concepts")

        # Chapters that received nothing are not chapters.
        themes = [t for t in themes if tags_by_theme.get(t["id"])]

        logB.info(">>> STEP 6: SECTION FORMATION")
        quota = allocate_section_quota(
            {t["id"]: len(tags_by_theme[t["id"]]) for t in themes}, cfg.size)
        logB.info(f"  section budget {cfg.size.target_sections} allocated as: "
                  f"{ {t['title'][:24]: quota[t['id']] for t in themes} }")

        former = SectionFormer(llm, cfg)
        chapters: List[Dict] = []
        for theme in tqdm(themes, desc="Forming sections"):
            tags = tags_by_theme[theme["id"]]
            sections = former.form(theme, tags, quota[theme["id"]])
            for i, section in enumerate(sections, 1):
                section["id"] = f"{theme['id']}_s{i:02d}"
            chapters.append({**theme, "tags": tags, "sections": sections})

        logB.info(">>> STEP 7: CURRICULUM ORDERING")
        orderer = CurriculumOrderer(llm, cfg)
        ordered_chapters = orderer.order(chapters, "chapters")
        for chapter in ordered_chapters:
            chapter["sections"] = orderer.order(
                chapter["sections"], f"sections of {chapter['title'][:28]}")

        ckpt.save("step4to7_clustering", {"ordered_chapters": ordered_chapters})

    # -- Step 8 -------------------------------------------------------------
    logB.info(">>> STEP 8: BUILD toc.json")
    toc = build_final_toc(ordered_chapters, normalized_chunk_tags, llm, cfg)
    PATHS.toc.parent.mkdir(parents=True, exist_ok=True)
    PATHS.toc.write_text(json.dumps(toc, indent=2, ensure_ascii=False), encoding="utf-8")

    # A human-readable companion, for reading the curriculum by eye.
    lines = [f"# {toc['book_title']}", ""]
    for chapter in toc["chapters"]:
        lines.append(f"\n## {chapter['order']}. {chapter['title']}")
        for section in toc["sections"]:
            if section["chapter_id"] == chapter["chapter_id"]:
                lines.append(f"{chapter['order']}.{section['order']} {section['title']}"
                             f"  *({section['estimated_word_count']} words, "
                             f"{len(section['chunk_ids'])} chunks)*")
    (PATHS.toc.parent / "toc.md").write_text("\n".join(lines), encoding="utf-8")

    # -- Validation ---------------------------------------------------------
    logB.info(">>> VALIDATION")
    report = validate_toc(toc)
    (PATHS.toc.parent / "toc_validation.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8")

    logB.info("-" * 70)
    logB.info(f"  chapters : {toc['stats']['chapters']}")
    logB.info(f"  sections : {toc['stats']['sections']} "
              f"(target {toc['stats']['target_sections']})")
    logB.info(f"  words    : {toc['stats']['total_estimated_words']:,} "
              f"≈ {toc['stats']['estimated_pages']} pages")
    logB.info(f"  elapsed  : {(time.time() - started) / 60:.1f} min")

    return toc, report


toc, validation = run_pipeline_b(chunks, book_llm, bcfg, ckpt)

In [ ]:
# ============================================================================
# Pipeline B — Report
# ============================================================================
"""
Read this before running Pipeline C. Four numbers decide whether the TOC is
usable:

  * sections vs target      -- the book's actual length
  * chunks unused           -- source material no section will ever see
  * sections with no source -- sections that would be written from nothing
  * duplicate titles        -- the same thing taught twice

And then read `storage/toc.md`. Nothing in this pipeline can tell you that two
chapters overlap or that a chapter title is vague; two minutes of reading can.

    "Nothing here fixes a bad table of contents... That is a signal to fix the
     TOC, not to keep writing."
"""

s = toc["stats"]

print("=" * 72)
print(f"PIPELINE B COMPLETE — \"{toc['book_title']}\"")
print("=" * 72)
print(f"Model                    : {toc['model']}")
print(f"Chapters                 : {s['chapters']}")
print(f"Sections                 : {s['sections']}   (target {s['target_sections']}, "
      f"design band 120-250)")
print(f"Estimated length         : {s['total_estimated_words']:,} words "
      f"≈ {s['estimated_pages']} pages")
print()
print(f"Chunks assigned          : {s['chunks_assigned']} / {s['chunks_total']}")
print(f"  never assigned         : {s['chunks_unused']}"
      "   (still reachable via find_related)")
print(f"  avg per section        : {s['avg_chunks_per_section']}")

print()
print("Structural validation:")
print(f"  sections with NO source: {len(validation['sections_without_source'])}"
      f"{'   <-- these would be written from nothing' if validation['sections_without_source'] else ''}")
print(f"  sections with NO tags  : {len(validation['sections_without_tags'])}")
print(f"  duplicate titles       : {len(validation['duplicate_titles'])}")
for d in validation["duplicate_titles"][:3]:
    print(f"    \"{d['title']}\"  in {d['sections']}")
print(f"  empty chapters         : {len(validation['chapters_without_sections'])}")
print(f"  verdict                : {'PASS' if validation['passed'] else 'NEEDS ATTENTION'}")

print()
print("Artifacts for Pipeline C:")
for label, path in [("toc.json", PATHS.toc),
                    ("normalized_tags.json", PATHS.normalized_tags),
                    ("tag_relationships.json", PATHS.tag_relationships),
                    ("chunk_tags.json", PATHS.chunk_tags)]:
    exists = "ok" if Path(path).exists() else "MISSING"
    print(f"  {label:24s} -> {path}   [{exists}]")

_aliases = json.loads(PATHS.normalized_tags.read_text(encoding="utf-8"))
_prereqs = json.loads(PATHS.tag_relationships.read_text(encoding="utf-8"))
print(f"\n  concepts with aliases  : {sum(1 for v in _aliases.values() if v)} "
      f"/ {len(_aliases)}      (seeds drift detection)")
print(f"  concepts with prereqs  : {len(_prereqs)}"
      "                (seeds the Continuity Gate in Pipeline C)")

# One section, in full, so the output is something you have actually looked at.
_sample = max(toc["sections"], key=lambda x: len(x["chunk_ids"]))
print("\n" + "-" * 72)
print("SAMPLE SECTION (what Pipeline C receives per section)")
print("-" * 72)
print(json.dumps(_sample, indent=2)[:800])
print("=" * 72)

### Honest Caveats

Accurate observations about Pipeline B as it now stands.

**1. The curriculum is the model's judgement, end to end.** Themes, assignments, sections and
ordering are all single LLM calls with no algorithmic second opinion. That is the design's
"LLM-native" principle taken at its word, and it means the TOC is only as good as the model's grasp
of the subject. It also means the output is **non-deterministic** — two runs of the same corpus will
not produce the same table of contents.

**2. Ordering is judged on titles and a sample.** `CurriculumOrderer` sees each unit's title and its
first ten concepts. A chapter holding 200 concepts is sequenced on a ten-concept preview, so subtle
dependencies buried deeper in a chapter are invisible to it.

**3. The section quota is enforced by asking.** `SectionFormer` is told *"create EXACTLY 12
sections"* and usually complies, but a model that returns 9 is logged rather than overruled. The
book lands *near* the target, not on it. Watch `stats.sections` against `stats.target_sections`.

**4. Relationship extraction happens here but is not used here.** `tag_relationships.json` exists to
seed the Book Ledger's `prerequisites` field so Pipeline C's Continuity Gate can check them per
section. Pipeline B itself does nothing with it. If you do not intend to run the Continuity Gate,
the relationship half of the extraction prompt is dead weight and can be dropped.

**5. Chunk assignment is lexical, not semantic.** A section gets the chunks that literally share its
canonical tags. A chunk that discusses the same idea in different words scores zero here. That gap
is covered later — `SourceMemory.find_related()` searches the whole embedding index — but the
*assigned* set is lexical.

**6. `max_chunks_per_section = 8` is a judgement call.** Eight × ~1,750 tokens ≈ 14K tokens,
comfortably inside the Context Assembler's 60K source budget with room for neighbourhood expansion.
Raise it if sections come out thin.

**7. Uncategorized concepts are placed, not dropped.** A tag the model refuses to assign is put in
the chapter it most often co-occurs with. Better than losing source material, but it is a guess, and
the count is logged.

**8. Normalization is one-directional and final.** Once two raw tags are merged, nothing later can
separate them. The prompt is deliberately conservative for this reason, but a wrong merge is
invisible after this step — the second concept simply never appears again.

**9. Validation is structural only.** It catches sections with no source, no tags, duplicate titles
and empty chapters. It cannot tell you that two chapters overlap, that "Advanced Topics" is a bad
chapter title, or that chapter 9 uses a term chapter 12 defines. **Read `toc.md` before running
Pipeline C** — it takes two minutes and it is the cheapest quality gate in the system.

**10. The word budget is an estimate, twice over.** `estimated_word_count` derives from a page
target using a 350-words-per-page assumption, and the Editor then holds the section to ±15% of it.
Neither number is measured. Treat the page count in the report as an order of magnitude.

---

### Summary Table of Every Technique in Pipeline B

| Technique | What it does | What it prevents |
|---|---|---|
| **Loads Pipeline A's chunks** | No re-chunking; ids match `chunk_metadata.json` | Layer 3 indexing chunks that do not exist |
| **Hard failure on chunk/metadata mismatch** | Refuses to run on an out-of-step corpus | A TOC built against stale ids |
| **Provenance carried through** | Chunks arrive knowing document, type, timestamp, figures | Losing what Stage 4 established |
| **Reuses Stage 3's loaded model** | No second load; the tagger *is* the describer | A second multi-GB load, and two models drifting |
| **Tokenizer chat templates** | Prompts rendered by the model's own template | gpt-oss "harmony" markup fed to a different model |
| **Structured output + repair loop** | Re-prompts with the parser's own error | Losing a chunk's tags to a stray brace |
| **Incremental extraction checkpoints** | Partial results saved every 50 chunks | Re-paying thousands of calls after a crash |
| **Counter-example prompting** | Bans "machine learning", demands "max pooling" | Tags too broad to retrieve anything |
| **Live config values in prompts** | `min/max_tags_per_chunk` actually used | Knobs that lie about what the code does |
| **Relationship extraction** | `requires` / `part_of` / `contrasts_with` per chunk | Pipeline C's Continuity Gate having no prerequisites |
| **Both-endpoints-must-be-tags rule** | Drops edges the text did not support | Invented prerequisites reaching the ledger |
| **`normalized_tags.json`** | Canonical → aliases, written to disk | The drift detector being computed and dropped |
| **`tag_relationships.json`** | Canonical → prerequisites | Failure #5 going uncovered in Pipeline C |
| **`chunk_tags.json`** | Chunk → canonical tags | Layer 3 with no retrieval index |
| **Frequency-ordered normalization batches** | Common vocabulary settles first | Near-duplicate canonicals in the long tail |
| **Unhandled tags become their own canonical** | Nothing is silently lost in normalization | Source material vanishing between steps |
| **Conservative merge prompt** | "Wrongly merging is far worse than not merging" | Two concepts taught as one, one never taught |
| **Co-occurrence placement** | Unassigned concepts land somewhere sensible | Losing source material to a shrug |
| **Empty chapters dropped** | A chapter with no concepts is not a chapter | Phantom chapters in the TOC |
| **Section budget from page target** | 120–250 sections, derived not guessed | A 30-page booklet from a 350-page brief |
| **Quota allocated by vocabulary share** | Big chapters get more sections | Uniform chapters regardless of content |
| **Caps give way to the target** | Clamps cannot make the target unreachable | 96 sections returned for a 175-section book |
| **Budget shortfall reported** | Says so when clamping bites | A short book reported as a success |
| **Two levels, not three** | chapter → section | A subsection level Pipeline C cannot read |
| **LLM ordering** | The model sequences chapters and sections | Alphabetical or arbitrary order |
| **Order reconciliation** | Dropped ids restored, invented ids discarded | A chapter silently vanishing from the book |
| **Design-exact ids and fields** | `sec_07_02`, `ch07`, `estimated_word_count` | Pipeline C reading fields that are not there |
| **Damped word allocation** | Denser sections get more room, not proportionally | 1,400-word slabs |
| **Ranked chunk assignment** | Top-K by tag overlap and density | 320 chunk ids truncated arbitrarily at load |
| **Coverage counters** | Unused chunks, sections with no source | Sections written from nothing |
| **Structural validation** | No source, no tags, duplicate titles, empty chapters | A mechanically broken TOC reaching Pipeline C |
| **`quick_test_mode` off by default** | A full run is a full run | 20 chunks masquerading as a corpus |
| **Per-step checkpoints** | Resume from any step | Repaying the LLM bill for steps 1–5 |

---

### The One-Paragraph Takeaway

Pipeline B decides what the book *is*, and it decides it with the model: themes, assignments,
sections and ordering are all the LLM's judgement, exactly as the notebook's "LLM-native" principle
intends. The machinery for that was already good. What was missing sat on either side of it — the
pipeline read a file Pipeline A never wrote, re-chunked the corpus so its `chunk_ids` could never
match Layer 3's, computed the alias list and threw it away, extracted no relationships so Pipeline
C's Continuity Gate had no prerequisites to check, and emitted a three-level nested TOC of forty-odd
sections when the consumers read a flat list and the design's arithmetic calls for a hundred and
seventy-five. The corrected pipeline changes almost none of the clustering and almost all of the
plumbing: **it consumes what Pipeline A produced, writes the three tag artifacts the Book Ledger is
seeded from, sizes the book from an explicit page target, ranks the source each section is given,
and checks the finished structure for the four faults that would make it unusable.** A table of
contents is a promise about what a book will contain — and everything here exists so the next
pipeline can find out whether the promise can be kept.

# **Book Writer**

## Pipeline C — Writing the Book, One Section at a Time

Pipeline A produced a corpus. Pipeline B decided what the book is. Pipeline C writes it: roughly
175 sections, each generated in its own conversation, each by a model with **no memory of the
previous 174** unless we hand it that memory deliberately.

That is the actual engineering problem, and the design states it plainly:

> *"Not 'can the model write well' — a modern model writes one good section easily. The problem is
> that section 140 has to know what section 12 said."*

Everything in this section exists to close that gap.

---

### The eight defects this is built to prevent

"This book feels thrown together" is not a matter of taste. When a reader says it, they are almost
always reacting to one of eight specific, detectable defects — and **none of them is a
writing-quality problem.**

| # | Failure | What the reader sees | What prevents it |
|---|---|---|---|
| 1 | **Repetition** | Section 47 re-explains what section 12 explained | Coverage map: which chunks are used, by whom |
| 2 | **Terminology drift** | "agent loop", "control loop", "orchestration cycle" all mean one thing | Concept registry with canonical names + aliases |
| 3 | **Broken promises** | "We'll cover retries in Chapter 9." Chapter 9 never does | Promise ledger with open/fulfilled status |
| 4 | **Phantom callbacks** | "As we saw earlier…" — but we never saw it | The same ledger, checked in reverse |
| 5 | **Prerequisite violations** | Chapter 3 uses a term Chapter 6 defines | Each concept records where it was defined |
| 6 | **Contradiction** | Chapter 2 says never use globals; Chapter 8's example does | A log of claims made |
| 7 | **Orphaned examples** | A running example is introduced, then abandoned | Example registry with current state |
| 8 | **Tone and depth drift** | Chapter 1 is chatty; Chapter 9 is clipped | Fixed constitution + previous section's ending |

Every one is a **memory** problem — which is good news, because memory problems are fixed with
plumbing rather than with a better model.

---

### Seven layers

```
╔═══════════════════════════════════════════════════════════════════════════╗
║                        THE BOOK'S MEMORY                                  ║
╠═══════════════════════════════════════════════════════════════════════════╣
║  L0  CONSTITUTION      fixed · ~700 tokens · in every single prompt        ║
║  L1  BOOK LEDGER       the book's understanding of itself                 ║
║                        concepts · claims · promises · examples · coverage  ║
║  L2  DRAFT STORE       append-only · every finished section, embedded      ║
║  L3  SOURCE MEMORY     the ground truth · chunks + provenance + usage      ║
║  ══════════ everything above is BOOK-WIDE and persistent ════════════════ ║
║  ══════════ everything below is PER-SECTION and disposable ══════════════ ║
║  L4  CONTEXT ASSEMBLER the slice that actually ships to the model          ║
║  L5  CLASSROOM         this section's plan, drift, doubts                 ║
║  L6  CONVERSATION      writer ↔ reviewer ↔ student turns                  ║
╚═══════════════════════════════════════════════════════════════════════════╝
```

And the loop that connects them:

```
   for each section in the TOC:
        ├─ L4 assembles the prompt ─────────── reads L0, L1, L2, L3
        ├─ Writer plans ──→ Continuity Gate + Reviewer ──→ approved
        ├─ Writer writes ──→ Student evaluates ──→ doubts ──→ clarify   (L5, L6)
        ├─ Editor melts the steps into one smooth section
        ├─ section saved to disk
        └─ Archivist reads it ──→ writes back into L1 and L2 ──→ checkpoint
                                            │
                                            └─→ the next section is better informed
```

**That last arrow is the whole design.** Without it you have 175 independent essays. With it you
have a book.

---

### What the existing implementation had, and what was missing

The notebook already had a Writer/Reviewer/Student loop, a prompt assembler, and an orchestrator.
The skeleton was sound. What was absent is precisely the machinery that makes section 140 know what
section 12 said:

| Component | Status before | Why it matters |
|---|---|---|
| **L1 Book Ledger** | **absent** | Nothing recorded concepts, claims, promises, or coverage |
| **L2 Draft Store** | **absent** | No section was ever embedded, so the book could not search itself |
| **L4 Context Assembler** | **absent** | Prompts were assembled ad hoc, with no budget |
| **Editor agent** | **absent** | Steps were concatenated raw — four openings inside one section |
| **Archivist agent** | **absent** | **The write-back arrow did not exist** |
| **Continuity Gate** | **absent** | No pre-write checks at all |
| **Finishing passes** | **absent** | No glossary, no promise resolution, no coverage report |
| `Layer2Narrative` | a stub | `get_section_context()` loaded summaries then discarded them, returning a comma-joined list of term names |
| `Layer5Source` | `open()` in a loop | No retrieval, no provenance, no usage tracking |
| Writer / Reviewer / Student | working | Kept, and rewritten against the design's prompts |

Six of the seven memory layers either did not exist or did not do what their names claimed. This
section builds all of them.

---

### One model, one namespace

Pipeline C reuses `book_model` — the same weights that described the figures in Stage 3 and named
the chapters in Pipeline B. The design is explicit about why:

> *"Using one language model for the whole pipeline is a deliberate choice… That alone does more for
> consistency than any amount of prompt tuning."*

## Phase 1 — Foundations, the Constitution, and the Book Ledger

Two layers, and they are the two the whole system leans on.

**Layer 0, the Constitution**, is a small fixed JSON file loaded into *every single prompt*. It
never changes during a run. That is the point: it is the one fixed reference frame that 175
independent generations are all measured against.

**Layer 1, the Book Ledger**, is the heart of the design. Everything else is either feeding it or
reading from it. It is the book's memory of itself — and critically, it is **a queryable database of
small typed facts, not a prose summary**:

> *"You cannot ask a paragraph 'is chunk_114 already used?' The ledger is a database of small typed
> facts precisely so the Continuity Gate can check it mechanically and the Context Assembler can
> slice it cheaply."*

It answers five questions no single section can answer for itself:

1. What words have we defined, and what do they mean here?
2. What have we asserted?
3. What have we promised, and did we deliver?
4. Where do our running examples currently stand?
5. Which source material has been used, and how deeply?

### The ledger starts full, not empty

The cheapest good idea in the design: Pipeline B already paid the LLM bill for tagging, so the
ledger is **pre-loaded** before section 1 is written —

```python
for canonical, raw_variants in normalized_tags.items():
    self._cache["concepts"][key] = {
        "canonical_name": canonical,
        "aliases": list(raw_variants),                        # normalized_tags.json
        "prerequisites": tag_relationships.get(canonical, []),# tag_relationships.json
        "depth": "unwritten", ...
    }
```

— so the terminology-drift detector works from the very first section instead of slowly learning
the vocabulary as it goes.

### Two design choices worth calling out

**Enrich in place, never duplicate.** When a section defines a term the ledger already knows under
a different name, `resolve_alias` catches it and we *add an alias* rather than creating a second
entry. Without this, by section 90 you would have `react_loop`, `reason_act_loop` and
`reasoning_cycle` sitting side by side as three separate concepts, and the drift detector would be
blind.

**Every write is logged as a diff.** The design's first stated risk is that the Archivist can be
wrong and its errors compound:

> *"If it hallucinates a definition or misses a promise, that error enters the ledger and every
> later section inherits it… Log every ledger diff so a bad harvest is traceable rather than
> mysterious."*

So `apply_archivist_update` records what changed, every time, to `output/ledger_diffs.jsonl`.

In [ ]:
# ============================================================================
# Phase 1.1 — Paths, config, and the shared model
# ============================================================================
"""
Pipeline C writes into the tree the design specifies in `Files on Disk`:

    storage/
      schemas/constitution.json     L0 - fixed, hand-authored
      toc.json                      from Pipeline B
      book_ledger.json              L1 - the central file
      checkpoints/work_queue.json
    output/
      sections/sec_01_01.md         the book, one file per section
      draft_index/                  L2 - Chroma: section abstracts
      raw_steps/                    pre-Editor drafts, for raw->edited diffs
      sessions/                     L5 - per-section classroom state
      history/                      L6 - per-section conversations
      book/                         manuscript, glossary, index, report

Everything is plain JSON plus one Chroma collection. It survives a crash, it is
diffable in git, and you can open book_ledger.json and read exactly what the
book believes about itself.
"""

import os
import re
import json
import time
import shutil
import logging
import traceback
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime
from collections import defaultdict, Counter
from typing import Dict, List, Any, Optional, Tuple

from tqdm.auto import tqdm

for _need in ("PATHS", "BOOK_MODEL", "count_tokens", "extract_json_object"):
    if _need not in globals():
        raise RuntimeError(
            f"`{_need}` is missing. Run the Document Preprocessing section first — "
            f"Pipeline C builds on Stage 0's paths, model and helpers.")


@dataclass
class BookPaths:
    """The design's `Files on Disk`, for the writing side."""

    storage: Path = Path("./storage")
    output: Path = Path("./output")

    # -- storage ------------------------------------------------------------
    @property
    def constitution(self) -> Path:  return self.storage / "schemas" / "constitution.json"
    @property
    def toc(self) -> Path:           return self.storage / "toc.json"
    @property
    def ledger(self) -> Path:        return self.storage / "book_ledger.json"
    @property
    def checkpoints(self) -> Path:   return self.storage / "checkpoints"
    @property
    def work_queue(self) -> Path:    return self.checkpoints / "work_queue.json"

    # -- output -------------------------------------------------------------
    @property
    def sections(self) -> Path:      return self.output / "sections"
    @property
    def draft_index(self) -> Path:   return self.output / "draft_index"
    @property
    def raw_steps(self) -> Path:     return self.output / "raw_steps"
    @property
    def sessions(self) -> Path:      return self.output / "sessions"
    @property
    def history(self) -> Path:       return self.output / "history"
    @property
    def book(self) -> Path:          return self.output / "book"
    @property
    def ledger_diffs(self) -> Path:  return self.output / "ledger_diffs.jsonl"

    def mkdirs(self) -> None:
        for p in (self.storage / "schemas", self.checkpoints, self.sections,
                  self.raw_steps, self.sessions, self.history, self.book):
            p.mkdir(parents=True, exist_ok=True)


BOOK = BookPaths()
BOOK.mkdirs()


def make_logger_c(name: str, logfile: str) -> logging.Logger:
    lg = logging.getLogger(name)
    lg.setLevel(logging.INFO)
    lg.handlers.clear()
    lg.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(name)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    for h in (logging.FileHandler(logfile, encoding="utf-8"), logging.StreamHandler()):
        h.setFormatter(fmt)
        lg.addHandler(h)
    return lg


logC = make_logger_c("pipelineC.writer", "pipelineC_writer.log")


@dataclass
class WriterConfig:
    """Configuration for Pipeline C."""

    # -- loop limits ---------------------------------------------------------
    max_plan_revisions: int = 3
    max_doubt_rounds: int = 2          # per teaching step
    max_editor_retries: int = 1

    # -- generation ----------------------------------------------------------
    plan_max_tokens: int = 1_200
    step_max_tokens: int = 1_400
    review_max_tokens: int = 700
    student_max_tokens: int = 400
    editor_max_tokens: int = 2_500
    archivist_max_tokens: int = 1_800
    temperature_prose: float = 0.7     # writing wants some life
    temperature_structured: float = 0.0

    # -- section shape -------------------------------------------------------
    default_word_count: int = 700
    steps_per_section_min: int = 3
    steps_per_section_max: int = 5

    # -- memory --------------------------------------------------------------
    tail_chars: int = 800              # verbatim ending of the previous section
    compaction_threshold: int = 12     # exchanges before Layer 6 compacts
    repetition_threshold: float = 0.85

    # -- housekeeping --------------------------------------------------------
    resume: bool = True
    stop_on_error: bool = False
    save_raw_steps: bool = True


wcfg = WriterConfig()

# One model for the whole system. Pipeline B's adapter already wraps Stage 3's
# loaded weights with a (system, user) calling convention, so Pipeline C reuses
# it rather than loading anything.
if "book_llm" in globals():
    llm = book_llm
    logC.info(f"Reusing the Pipeline B adapter over {llm.model_id}")
elif "BookLLM" in globals() and "bcfg" in globals():
    llm = BookLLM(bcfg)
else:
    raise RuntimeError("Run the TOC Generation section first — Pipeline C reuses `book_llm`.")

logC.info("Pipeline C configuration:")
logC.info(f"  model        : {llm.model_id}")
logC.info(f"  storage      : {BOOK.storage.resolve()}")
logC.info(f"  output       : {BOOK.output.resolve()}")
logC.info(f"  plan revisions max {wcfg.max_plan_revisions}, "
          f"doubt rounds max {wcfg.max_doubt_rounds}")

In [ ]:
# ============================================================================
# Phase 1.2 — Layer 0: The Constitution
# ============================================================================
"""
A small, fixed JSON file loaded into every single prompt. It never changes
during a run -- that is the point.

Why each block earns its place (design section 9):

  canonical_examples  stops failure #7. If every prompt says "the book's running
                      example is SupportBot", sections stop inventing a fresh
                      FooBot each time.
  code_conventions    is what makes 175 code blocks look like one engineer wrote
                      them. Style drift in code is more visible to a reader than
                      style drift in prose.
  forbidden_patterns  costs about 40 tokens and removes the tics that make
                      generated text instantly recognisable.
"""

DEFAULT_CONSTITUTION = {
    "book_identity": {
        "title": "The Future of AI Agents",
        "subtitle": "A Production-Grade Implementation Guide",
        "target_audience": {
            "level": "Intermediate Developer",
            "prerequisites": ["Python", "Basic LLM knowledge"],
            "persona_description": "Software engineers building autonomous systems.",
        },
    },
    "style_guide": {
        "depth": "applied",
        "tone": "rigorous_accessible",
        "teaching_approach": "Code-First",
        "practicality_level": "Production-Grade",
        "engagement_style": "Socratic",
        "analogy_density": "Moderate",
    },
    # the running examples the whole book shares
    "canonical_examples": [
        {
            "name": "SupportBot",
            "description": "A customer-support agent we build across the book",
            "introduced_in_chapter": "ch02",
            "purpose": "The single thread readers follow end to end",
        }
    ],
    # mechanical conventions, so all the code looks like one author
    "code_conventions": {
        "language": "python",
        "style": "type-hinted, dataclasses for config, logging not print",
        "error_handling": "explicit try/except with logged context",
        "naming": "snake_case functions, PascalCase classes",
    },
    # house rules
    "forbidden_patterns": [
        "In today's fast-paced world",
        "It is important to note that",
        "Let's dive in",
        "In conclusion",
        "Furthermore,",
        "delve into",
    ],
}


class Constitution:
    """Layer 0. Fixed for the whole run; injected into every prompt."""

    def __init__(self, path: Path, default: Dict[str, Any]):
        self.path = path
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(default, indent=2), encoding="utf-8")
            logC.info(f"Constitution written to {path} (edit it before a real run)")
        self.data = json.loads(path.read_text(encoding="utf-8"))

    # ---------------------------------------------------------------- reads --
    @property
    def title(self) -> str:
        return self.data["book_identity"]["title"]

    @property
    def audience(self) -> str:
        return self.data["book_identity"]["target_audience"]["level"]

    def examples(self) -> List[Dict[str, Any]]:
        return self.data.get("canonical_examples", [])

    # ------------------------------------------------------------ injection --
    def get_prompt_injection(self) -> str:
        """The full constitution, for the Writer. Roughly 700 tokens."""
        identity = self.data["book_identity"]
        audience = identity["target_audience"]
        style = self.data["style_guide"]
        code = self.data["code_conventions"]

        lines = [
            "=== THE BOOK ===",
            f"Title: {identity['title']} — {identity.get('subtitle', '')}",
            f"Reader: {audience['level']}. {audience.get('persona_description', '')}",
            f"Assumed knowledge: {', '.join(audience.get('prerequisites', []))}",
            "",
            "=== HOW IT SOUNDS ===",
        ]
        lines += [f"{k}: {v}" for k, v in style.items()]

        if self.examples():
            lines += ["", "=== RUNNING EXAMPLES (use these, do not invent new ones) ==="]
            for ex in self.examples():
                lines.append(f"- {ex['name']}: {ex['description']} "
                             f"(introduced in {ex.get('introduced_in_chapter', '?')})")

        lines += ["", "=== CODE CONVENTIONS ==="]
        lines += [f"{k}: {v}" for k, v in code.items()]

        forbidden = self.data.get("forbidden_patterns", [])
        if forbidden:
            lines += ["", "=== NEVER WRITE THESE PHRASES ===",
                      "; ".join(f'"{p}"' for p in forbidden)]
        return "\n".join(lines)

    def get_style_injection(self) -> str:
        """The shorter form, for the Editor and Reviewer."""
        style = self.data["style_guide"]
        forbidden = self.data.get("forbidden_patterns", [])
        parts = ["=== STYLE ===",
                 "; ".join(f"{k}: {v}" for k, v in style.items())]
        if forbidden:
            parts.append("Never write: " + "; ".join(f'"{p}"' for p in forbidden))
        return "\n".join(parts)


constitution = Constitution(BOOK.constitution, DEFAULT_CONSTITUTION)
print(f"Layer 0 ready — \"{constitution.title}\" for {constitution.audience}")
print(f"  constitution: {BOOK.constitution}")
print(f"  injection is {count_tokens(constitution.get_prompt_injection())} tokens "
      f"(design budget: 700)")

In [ ]:
# ============================================================================
# Phase 1.3 — Layer 1: The Book Ledger
# ============================================================================
"""
The book's memory of itself: what it has defined, claimed, promised,
demonstrated, and consumed. Read by every section, written by the Archivist.

Not a prose summary -- a database of small typed facts, so the Continuity Gate
can check it mechanically and the Context Assembler can slice it cheaply.
"""

EMPTY_LEDGER = {
    "version": 0,
    "last_section": None,
    "concepts": {},           # prevents drift and redefinition (#2, #5)
    "claims": [],             # prevents contradiction (#6)
    "promises": [],           # prevents broken promises and phantoms (#3, #4)
    "examples": {},           # prevents orphaned examples (#7)
    "coverage": {},           # prevents repetition (#1)
    "section_summaries": {},  # the book's memory of itself
    "chapter_rollups": {},    # keeps the ledger small in the prompt
    "figures_used": {},       # figure_id -> sections that leaned on it
}

CONCEPT_DEPTHS = ("unwritten", "mentioned", "introduced", "explained")


class BookLedger:
    """Layer 1."""

    def __init__(self, path: Path):
        self.path = path
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(EMPTY_LEDGER, indent=2), encoding="utf-8")
        self._cache = json.loads(path.read_text(encoding="utf-8"))
        self.ledger_diff_log = BOOK.ledger_diffs
        # tolerate a ledger written by an older version of this notebook
        for key, blank in EMPTY_LEDGER.items():
            self._cache.setdefault(key, blank if not isinstance(blank, (dict, list))
                                   else type(blank)())

    # ---------------------------------------------------------------- writes --
    def _flush(self) -> None:
        tmp = self.path.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(self._cache, indent=2, ensure_ascii=False),
                       encoding="utf-8")
        tmp.replace(self.path)      # atomic: a crash mid-write cannot corrupt it

    @staticmethod
    def key_for(term: str) -> str:
        return re.sub(r"[^a-z0-9]+", "_", term.lower().strip()).strip("_") or "concept"

    # ----------------------------------------------------------------- reads --
    def resolve_alias(self, term: str) -> Optional[str]:
        """'thought-action cycle' -> 'react_loop'. This is the drift detector."""
        t = term.lower().strip()
        for key, c in self._cache["concepts"].items():
            if t == c["canonical_name"].lower() or t in [a.lower() for a in c["aliases"]]:
                return key
        return None

    def undefined_prerequisites(self, concept_keys: List[str]) -> List[str]:
        """Concepts this section needs that the book has not defined yet (#5)."""
        missing = []
        for term in concept_keys:
            key = self.resolve_alias(term) or self.key_for(term)
            c = self._cache["concepts"].get(key)
            if not c:
                continue
            for pre in c.get("prerequisites", []):
                pkey = self.resolve_alias(pre) or self.key_for(pre)
                p = self._cache["concepts"].get(pkey)
                if p and p["depth"] in ("unwritten", "mentioned"):
                    missing.append(p["canonical_name"])
        return sorted(set(missing))

    def defined_prerequisites(self, concept_keys: List[str]) -> List[Dict]:
        """
        Concepts this section depends on that the book HAS already defined.

        Tag overlap alone is not enough. A section tagged only `transformers`
        will certainly USE self-attention -- the prerequisite relation says so --
        but self-attention is not one of its tags, so the Writer would never be
        shown its agreed definition or its alias list, and could reintroduce it
        under a variant name. That is failure #2 arriving through the front door.
        """
        out: Dict[str, Dict] = {}
        for term in concept_keys:
            key = self.resolve_alias(term) or self.key_for(term)
            c = self._cache["concepts"].get(key)
            if not c:
                continue
            for pre in c.get("prerequisites", []):
                pkey = self.resolve_alias(pre) or self.key_for(pre)
                p = self._cache["concepts"].get(pkey)
                if p and p.get("definition"):
                    out[pkey] = p
        return list(out.values())

    def open_promises_for(self, chapter_id: str) -> List[Dict]:
        return [p for p in self._cache["promises"]
                if p["status"] == "open" and p.get("target_hint") == chapter_id]

    def all_open_promises(self) -> List[Dict]:
        return [p for p in self._cache["promises"] if p["status"] == "open"]

    def chunk_already_used(self, chunk_id: str) -> List[str]:
        return self._cache["coverage"].get(chunk_id, {}).get("used_by", [])

    def concepts_for_tags(self, tags: List[str]) -> List[Dict]:
        """The defined concepts whose source tags overlap this section's tags."""
        want = {t.lower() for t in tags}
        return [c for c in self._cache["concepts"].values()
                if c.get("definition") and (
                    {s.lower() for s in c.get("source_tags", [])} & want
                    or c["canonical_name"].lower() in want)]

    def stats(self) -> Dict[str, int]:
        concepts = self._cache["concepts"].values()
        return {
            "version": self._cache["version"],
            "concepts": len(self._cache["concepts"]),
            "concepts_defined": sum(1 for c in concepts if c.get("definition")),
            "claims": len(self._cache["claims"]),
            "promises_open": sum(1 for p in self._cache["promises"] if p["status"] == "open"),
            "promises_fulfilled": sum(1 for p in self._cache["promises"]
                                      if p["status"] == "fulfilled"),
            "examples": len(self._cache["examples"]),
            "chunks_used": len(self._cache["coverage"]),
            "sections_written": len(self._cache["section_summaries"]),
        }

    # ------------------------------------------------------------- seeding ---
    def seed_from_toc_pipeline(self, normalized_tags: Dict[str, List[str]],
                               tag_relationships: Dict[str, List[str]]) -> int:
        """
        Pre-populate the concept registry before any writing begins.

        "The drift detector works from the first section rather than slowly
         learning the vocabulary as it goes."

        Idempotent: re-seeding an existing ledger enriches aliases rather than
        wiping definitions the Archivist has already harvested.
        """
        added = 0
        for canonical, raw_variants in normalized_tags.items():
            key = self.key_for(canonical)
            entry = self._cache["concepts"].get(key)
            if entry is None:
                self._cache["concepts"][key] = {
                    "canonical_name": canonical,
                    "aliases": sorted(set(raw_variants)),
                    "definition": None,
                    "defined_in": None,
                    "depth": "unwritten",
                    "referenced_in": [],
                    "source_tags": [canonical],
                    "prerequisites": tag_relationships.get(canonical, []),
                }
                added += 1
            else:
                for alias in raw_variants:
                    if alias != entry["canonical_name"] and alias not in entry["aliases"]:
                        entry["aliases"].append(alias)
                for pre in tag_relationships.get(canonical, []):
                    if pre not in entry["prerequisites"]:
                        entry["prerequisites"].append(pre)
        self._flush()
        return added

    # ----------------------------- writes (the Archivist is the only caller) --
    def apply_archivist_update(self, update: Dict[str, Any], section_id: str) -> Dict[str, Any]:
        """
        Fold one harvested section into the ledger, and record the diff.

        The design's first stated risk is that the Archivist can be wrong and
        its errors compound. A diff log is what makes a bad harvest traceable
        rather than mysterious.
        """
        diff = {"section_id": section_id, "at": datetime.now().isoformat(timespec="seconds"),
                "concepts_new": [], "concepts_enriched": [], "aliases_added": [],
                "claims": 0, "promises_made": [], "promises_fulfilled": [],
                "examples_touched": [], "chunks": []}

        # ---- concepts defined ---------------------------------------------
        for c in update.get("concepts_defined", []):
            term = str(c.get("term", "")).strip()
            if not term:
                continue
            key = self.resolve_alias(term) or self.key_for(term)
            entry = self._cache["concepts"].get(key)
            if entry is None:
                entry = {"canonical_name": term, "aliases": [], "definition": None,
                         "defined_in": None, "depth": "unwritten", "referenced_in": [],
                         "source_tags": [], "prerequisites": []}
                self._cache["concepts"][key] = entry
                diff["concepts_new"].append(key)

            # enrich in place, never duplicate
            if entry["definition"] is None:
                entry["definition"] = str(c.get("definition", "")).strip() or None
                entry["defined_in"] = section_id
                entry["depth"] = "explained"
                diff["concepts_enriched"].append(key)
            if term != entry["canonical_name"] and term not in entry["aliases"]:
                entry["aliases"].append(term)
                diff["aliases_added"].append(f"{key}<-{term}")

        # ---- concepts referenced ------------------------------------------
        for term in update.get("concepts_referenced", []):
            key = self.resolve_alias(str(term))
            if not key:
                continue
            entry = self._cache["concepts"][key]
            if section_id not in entry["referenced_in"]:
                entry["referenced_in"].append(section_id)
            if entry["depth"] == "unwritten":
                entry["depth"] = "mentioned"

        # ---- claims --------------------------------------------------------
        for claim in update.get("claims", []):
            claim = dict(claim)
            claim.setdefault("claim_id", f"c_{len(self._cache['claims']) + 1:04d}")
            claim["section_id"] = section_id
            claim.setdefault("tags", update.get("summary", {}).get("teaches", []))
            self._cache["claims"].append(claim)
            diff["claims"] += 1

        # ---- promises ------------------------------------------------------
        for promise in update.get("promises_made", []):
            promise = dict(promise)
            promise.setdefault("promise_id",
                               f"p_{len(self._cache['promises']) + 1:04d}")
            promise["made_in"] = section_id
            promise["status"] = "open"
            self._cache["promises"].append(promise)
            diff["promises_made"].append(promise["promise_id"])

        for pid in update.get("promises_fulfilled", []):
            for p in self._cache["promises"]:
                if p["promise_id"] == pid and p["status"] == "open":
                    p["status"] = "fulfilled"
                    p["fulfilled_in"] = section_id
                    diff["promises_fulfilled"].append(pid)

        # ---- running examples ----------------------------------------------
        for name, state in update.get("example_states", {}).items():
            ex = self._cache["examples"].setdefault(
                name, {"introduced_in": section_id, "files_shown": []})
            ex["current_state"] = state
            ex["last_touched"] = section_id
            diff["examples_touched"].append(name)

        # ---- coverage ------------------------------------------------------
        for cid, depth in update.get("chunks_used", {}).items():
            cov = self._cache["coverage"].setdefault(cid, {"used_by": [], "depth": depth})
            if section_id not in cov["used_by"]:
                cov["used_by"].append(section_id)
            # a chunk used as primary anywhere is primary
            if depth == "primary":
                cov["depth"] = "primary"
            diff["chunks"].append(f"{cid}:{depth}")

        # ---- summary + bookkeeping -----------------------------------------
        summary = update.get("summary") or {}
        self._cache["section_summaries"][section_id] = summary
        self._cache["version"] += 1
        self._cache["last_section"] = section_id
        self._flush()

        with self.ledger_diff_log.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(diff, ensure_ascii=False) + "\n")

        return diff

    def figure_use_count(self, figure_id: str) -> int:
        return len(self._cache.get("figures_used", {}).get(figure_id, []))

    def record_figures_used(self, figure_ids: List[str], section_id: str) -> None:
        """The design budgets 2-3 figures per section and tracks usage so the
        same diagram is not leaned on twice. This is the tracking half."""
        if not figure_ids:
            return
        used = self._cache.setdefault("figures_used", {})
        for fid in figure_ids:
            entries = used.setdefault(fid, [])
            if section_id not in entries:
                entries.append(section_id)
        self._flush()

    def add_chapter_rollup(self, chapter_id: str, text: str) -> None:
        self._cache["chapter_rollups"][chapter_id] = text
        self._flush()


ledger = BookLedger(BOOK.ledger)
print(f"Layer 1 ready — {BOOK.ledger}")
print(f"  {ledger.stats()}")

In [ ]:
# ============================================================================
# Phase 1.4 — Seed the ledger from Pipeline B, and load the TOC
# ============================================================================
"""
    "Before section 1 is written, the Book Ledger is PRE-LOADED with every
     canonical concept and every alias the sources use for it."

This is the join between Pipeline B and Pipeline C. If it fails, the drift
detector cold-starts and failure #2 goes uncovered for the first several
chapters, so it fails loudly rather than quietly.
"""


def load_toc() -> Dict[str, Any]:
    if not BOOK.toc.exists():
        raise FileNotFoundError(
            f"{BOOK.toc} not found — run the TOC Generation section first.")
    toc = json.loads(BOOK.toc.read_text(encoding="utf-8"))

    required = {"section_id", "chapter_id", "title", "tags",
                "chunk_ids", "estimated_word_count"}
    for section in toc["sections"]:
        missing = required - set(section)
        if missing:
            raise ValueError(f"{section.get('section_id', '?')} is missing {missing} — "
                             f"toc.json does not match the design's contract.")
    return toc


def seed_ledger_from_pipeline_b(ledger: BookLedger) -> Dict[str, int]:
    tag_file, rel_file = PATHS.normalized_tags, PATHS.tag_relationships
    if not tag_file.exists():
        raise FileNotFoundError(
            f"{tag_file} not found. Pipeline B writes it, and the ledger is seeded "
            f"from it — without it the drift detector starts empty.")

    normalized_tags = json.loads(tag_file.read_text(encoding="utf-8"))
    relationships = (json.loads(rel_file.read_text(encoding="utf-8"))
                     if rel_file.exists() else {})
    if not rel_file.exists():
        logC.warning(f"{rel_file} not found — concepts will be seeded with no "
                     f"prerequisites, so the Continuity Gate's check #3 cannot fire.")

    added = ledger.seed_from_toc_pipeline(normalized_tags, relationships)
    return {"canonical_concepts": len(normalized_tags),
            "newly_added": added,
            "with_prerequisites": len(relationships),
            "total_aliases": sum(len(v) for v in normalized_tags.values())}


toc = load_toc()
seed_stats = seed_ledger_from_pipeline_b(ledger)

print("=" * 72)
print("PHASE 1 COMPLETE — Constitution and Book Ledger")
print("=" * 72)
print(f"Book                 : \"{toc['book_title']}\"")
print(f"TOC                  : {toc['stats']['chapters']} chapters, "
      f"{toc['stats']['sections']} sections")
print(f"Target length        : {toc['stats']['total_estimated_words']:,} words "
      f"≈ {toc['stats']['estimated_pages']} pages")
print()
print("Ledger seeded from Pipeline B:")
for k, v in seed_stats.items():
    print(f"  {k:22s}: {v}")
print()
print("Ledger state:")
for k, v in ledger.stats().items():
    print(f"  {k:22s}: {v}")

# Prove the drift detector is live before a single word is written.
_sample = next((c for c in ledger._cache["concepts"].values() if c["aliases"]), None)
if _sample:
    print(f"\nDrift detection is armed. Example:")
    print(f"  canonical : {_sample['canonical_name']}")
    print(f"  aliases   : {', '.join(_sample['aliases'][:5])}")
    _probe = _sample["aliases"][0]
    print(f"  resolve_alias({_probe!r}) -> {ledger.resolve_alias(_probe)!r}")
else:
    print("\nNOTE: no concept has aliases — drift detection will be weak. "
          "Check Pipeline B's normalization.")
print("=" * 72)

## Phase 2 — The Draft Store and Source Memory

Two more persistent layers, and they answer two different questions.

**The ledger stores what the book *means*. The draft store stores what the book *says*.** You need
both, because a 100-word abstract cannot tell you whether a new paragraph accidentally duplicates an
old one. Only the actual text can. So every finished section is stored twice: as markdown on disk,
and as an embedding so the book can search itself.

One method in Layer 2 deserves special mention:

> *"`get_tail()` is the cheapest trick in the entire design. Taking the previous section's last 800
> characters and injecting them verbatim, under the instruction 'here is how the previous section
> ended, continue from this voice', does more for perceived flow than any amount of style-guide
> text. It costs one file read."*

**Layer 3 is the grounding layer** — the chunks, their provenance, and an account of which ones have
already been used. That last part is fifteen characters of logic that closes failure #1 at its root:

```python
prior = self.ledger.chunk_already_used(cid)
if prior:
    header += f"[NOTE: already used in {', '.join(prior)} — do not re-explain]\n"
```

The TOC will assign the same chunk to two or three different sections, because a chunk carries
several tags. Without the note, the second and third sections receive it as fresh material and
dutifully re-explain it. With the note, they build on it instead.

Two things Layer 3 inherits from Pipeline A for free:

- **Neighbourhood retrieval.** A chunk arrives with its neighbours, because *"a chunk boundary is
  an artifact of the chunker, not of the argument"*.
- **Provenance in the header.** `from: lecture_04.mp4 | type: transcript | at 34:12` — so the Writer
  can say *"as the lecture demonstrated"* rather than *"as Vaswani et al. put it"*, and mean it.

In [ ]:
# ============================================================================
# Phase 2.1 — Layer 2: The Draft Store
# ============================================================================
"""
Every finished section, on disk as markdown and in a vector index as an
abstract. The index is what lets the Continuity Gate ask "is this plan too close
to something we already wrote?" before a word is generated.
"""


class DraftStore:
    """Layer 2. Append-only: the book so far."""

    def __init__(self, paths: BookPaths, model_name: str = "all-MiniLM-L6-v2"):
        self.sections_dir = paths.sections
        self.sections_dir.mkdir(parents=True, exist_ok=True)
        self.collection = None
        self._index_path = paths.draft_index

        try:
            import chromadb
            from chromadb.utils.embedding_functions import (
                SentenceTransformerEmbeddingFunction)
            client = chromadb.PersistentClient(path=str(self._index_path))
            self.collection = client.get_or_create_collection(
                name="written_sections",
                embedding_function=SentenceTransformerEmbeddingFunction(
                    model_name=model_name),
            )
            logC.info(f"Layer 2: draft index at {self._index_path}")
        except ImportError:
            # Not fatal, but it disables the repetition detector, so it is loud.
            logC.warning("chromadb/sentence-transformers not installed.")
            logC.warning("  Sections will still be written to disk, but the "
                         "Continuity Gate's repetition check and the Context "
                         "Assembler's 'nearest by meaning' neighbours are DISABLED.")

    # ----------------------------------------------------------------- write --
    @staticmethod
    def render_markdown(title: str, content: str) -> str:
        """The one canonical on-disk form of a section."""
        return f"## {title}\n\n{content.strip()}\n"

    def write_markdown(self, section_id: str, title: str, content: str) -> Path:
        """
        Put the section on disk.

        Called by the orchestrator BEFORE the Archivist runs, so a section
        survives even if cataloguing fails. `add_section` re-renders the same
        bytes, so the two writers can never disagree about what the book says --
        an earlier version wrote a heading here and stripped it there.
        """
        path = self.sections_dir / f"{section_id}.md"
        path.write_text(self.render_markdown(title, content), encoding="utf-8")
        return path

    def add_section(self, section_id: str, title: str, chapter_id: str,
                    content: str, abstract: str) -> None:
        self.write_markdown(section_id, title, content)
        if self.collection is None:
            return
        try:
            self.collection.upsert(
                ids=[section_id],
                documents=[abstract or title],
                metadatas=[{"section_id": section_id, "title": title,
                            "chapter_id": chapter_id,
                            "word_count": len(content.split())}],
            )
        except Exception as exc:
            logC.error(f"Could not index {section_id}: {exc}")

    # ------------------------------------------------------------------ read --
    def get_full(self, section_id: Optional[str]) -> str:
        if not section_id:
            return ""
        path = self.sections_dir / f"{section_id}.md"
        return path.read_text(encoding="utf-8") if path.exists() else ""

    def get_tail(self, section_id: Optional[str], n_chars: int = 800) -> str:
        """
        The previous section's last N characters, verbatim.

        The cheapest trick in the design: injecting this under "here is how the
        previous section ended, continue from this voice" does more for perceived
        flow than any amount of style-guide text.
        """
        return self.get_full(section_id)[-n_chars:]

    def written_ids(self) -> List[str]:
        return sorted(p.stem for p in self.sections_dir.glob("sec_*.md"))

    def find_similar(self, query: str, k: int = 3,
                     exclude: Optional[str] = None) -> List[Dict]:
        """Nearest written sections by meaning. Empty list if the index is off."""
        if self.collection is None or not query.strip():
            return []
        try:
            n_written = self.collection.count()
            if n_written == 0:
                return []
            res = self.collection.query(query_texts=[query],
                                        n_results=min(k + 1, n_written))
        except Exception as exc:
            logC.warning(f"draft index query failed: {exc}")
            return []

        out = []
        for i, sid in enumerate(res["ids"][0]):
            if sid == exclude:
                continue
            distance = res["distances"][0][i] if res.get("distances") else 0.0
            out.append({"section_id": sid,
                        "title": res["metadatas"][0][i].get("title", sid),
                        "abstract": res["documents"][0][i],
                        "similarity": 1 - distance})
        return out[:k]


drafts = DraftStore(BOOK)
print(f"Layer 2 ready — {len(drafts.written_ids())} sections already on disk, "
      f"index {'ON' if drafts.collection is not None else 'OFF'}")

In [ ]:
# ============================================================================
# Phase 2.2 — Layer 3: Source Memory
# ============================================================================
"""
The grounding layer: chunks, their provenance, and -- importantly -- an account
of which ones have already been used.

Everything here was produced by Pipeline A. Layer 3 is read-mostly: it opens
chunk_metadata.json and the source_index Chroma collection and never writes to
either.
"""


class SourceMemory:
    """Layer 3."""

    def __init__(self, ledger: BookLedger, neighbourhood: int = 1):
        self.ledger = ledger
        self.chunks_dir = PATHS.chunks
        self.neighbourhood = neighbourhood

        if not PATHS.chunk_metadata.exists():
            raise FileNotFoundError(
                f"{PATHS.chunk_metadata} not found — run Pipeline A's Stage 4.")
        self.meta: Dict[str, Any] = json.loads(
            PATHS.chunk_metadata.read_text(encoding="utf-8"))

        self.collection = None
        try:
            import chromadb
            from chromadb.utils.embedding_functions import (
                SentenceTransformerEmbeddingFunction)
            client = chromadb.PersistentClient(path=str(PATHS.source_index))
            self.collection = client.get_collection(
                name="source_chunks",
                embedding_function=SentenceTransformerEmbeddingFunction(
                    model_name="all-MiniLM-L6-v2"))
            logC.info(f"Layer 3: {len(self.meta)} chunks, source index ON")
        except Exception as exc:
            logC.warning(f"source_index unavailable ({type(exc).__name__}). "
                         f"Semantic source expansion is DISABLED — every section "
                         f"will be written only from the chunks the TOC assigned it. "
                         f"Run Pipeline A's Stage 5 to build it.")

    # ------------------------------------------------------------- provenance --
    def _header(self, chunk_id: str) -> str:
        m = self.meta.get(chunk_id, {})
        bits = [f"--- SOURCE {chunk_id}",
                f"from: {m.get('source_document', 'unknown')}",
                f"type: {m.get('source_type', 'text')}"]
        ts = m.get("timestamp")
        if ts:
            mins = int(ts["start_seconds"] // 60)
            secs = int(ts["start_seconds"] % 60)
            bits.append(f"at ~{mins:d}:{secs:02d}"
                        + (" (approx)" if ts.get("approximate") else ""))
        header = " | ".join(bits) + " ---\n"

        # The fifteen characters of logic that close failure #1 at its root.
        prior = self.ledger.chunk_already_used(chunk_id)
        if prior:
            header += (f"[NOTE: already used in {', '.join(prior)} — "
                       f"do not re-explain, build on it]\n")
        return header

    def _read(self, chunk_id: str) -> Optional[str]:
        path = self.chunks_dir / f"{chunk_id}.txt"
        return path.read_text(encoding="utf-8") if path.exists() else None

    # ------------------------------------------------------- 1. assigned source --
    def get_assigned(self, chunk_ids: List[str], expand: bool = True) -> str:
        """
        The chunks the TOC assigned, with provenance and usage notes.

        Chunks arrive with their neighbours: "a chunk boundary is an artifact of
        the chunker, not of the argument, and there is no reason to hand the
        Writer a paragraph that starts mid-thought."
        """
        parts: List[str] = []
        emitted: set = set()

        for cid in chunk_ids:
            body = self._read(cid)
            if body is None:
                parts.append(f"\n[MISSING SOURCE: {cid}]\n")
                continue
            parts.append("\n" + self._header(cid) + body)
            emitted.add(cid)

            if not expand:
                continue
            # neighbours, same document only, without repeating anything
            number = int(cid.split("_")[1])
            home = self.meta.get(cid, {}).get("source_document")
            for n in range(number - self.neighbourhood, number + self.neighbourhood + 1):
                nid = f"chunk_{n:04d}"
                if nid in emitted or nid == cid or nid not in self.meta:
                    continue
                if self.meta[nid].get("source_document") != home:
                    continue
                nbody = self._read(nid)
                if nbody:
                    parts.append(f"\n--- CONTEXT {nid} (neighbour of {cid}) ---\n{nbody}")
                    emitted.add(nid)

        return "".join(parts)

    # --------------------------------------------------- 2. semantic expansion --
    def find_related(self, query: str, exclude: List[str], k: int = 3) -> str:
        """Relevant material the TOC did not assign. Closes coverage gaps."""
        if self.collection is None or not query.strip():
            return ""
        try:
            res = self.collection.query(query_texts=[query],
                                        n_results=k + len(exclude))
        except Exception as exc:
            logC.warning(f"source index query failed: {exc}")
            return ""

        parts, taken = [], 0
        for i, cid in enumerate(res["ids"][0]):
            if cid in exclude or taken >= k:
                continue
            parts.append(f"\n--- RELATED SOURCE {cid} (supporting, not assigned) "
                         f"| from: {self.meta.get(cid, {}).get('source_document', '?')} ---\n"
                         f"{res['documents'][0][i]}")
            taken += 1
        return "".join(parts)

    # ------------------------------------------------------------- 3. figures --
    def get_figures(self, chunk_ids: List[str]) -> List[Dict]:
        """
        The figures that belong with these chunks.

        Because the Writer is a multimodal model, the Context Assembler can
        attach the image itself next to its description -- the model reads the
        diagram instead of reading somebody's summary of it.
        """
        figures = []
        for cid in chunk_ids:
            for f in self.meta.get(cid, {}).get("figures", []):
                if f.get("description"):
                    figures.append({**f, "chunk_id": cid})
        return figures

    def all_chunk_ids(self) -> List[str]:
        return sorted(self.meta.keys())


source = SourceMemory(ledger, neighbourhood=1)

print("=" * 72)
print("PHASE 2 COMPLETE — Draft Store and Source Memory")
print("=" * 72)
print(f"Sections on disk     : {len(drafts.written_ids())}")
print(f"Draft index          : {'ON' if drafts.collection is not None else 'OFF (repetition detection disabled)'}")
print(f"Source chunks        : {len(source.meta)}")
print(f"Source index         : {'ON' if source.collection is not None else 'OFF (semantic expansion disabled)'}")
_figs = sum(len(m.get('figures', [])) for m in source.meta.values())
print(f"Figures reachable    : {_figs}")

# Show what a Writer actually receives for one section's source, so the
# provenance and usage-note machinery is something you have looked at.
_probe = toc["sections"][0]
_block = source.get_assigned(_probe["chunk_ids"][:1])
print(f"\nSAMPLE assigned source for {_probe['section_id']}:")
print("-" * 72)
print(_block[:600] if _block else "(no chunks assigned)")
print("=" * 72)

## Phase 3 — The Context Assembler, and the Working Memory

Layers 0 to 3 are the library. **Layer 4 is the librarian who decides what goes in the bag.**

By section 120 the ledger holds hundreds of concepts, hundreds of claims and 119 summaries. Most of
it is irrelevant to the section being written. The assembler's job is **selection**: build one
prompt out of everything the book knows, containing only what this section actually needs.

### How much each section actually sees

| Distance from the current section | What it receives |
|---|---|
| The previous section | Full text, verbatim |
| 3 nearest by meaning, anywhere in the book | Full text |
| Same chapter | Every section's abstract |
| Every other chapter | Chapter rollup plus section titles |
| Everything else | Nothing as prose — but the **ledger** still carries its concepts, claims and promises |

That last row is the important one. A section from 200 pages ago contributes nothing as prose, but
its definitions, claims and promises are still live, because they were distilled into small typed
facts rather than left as text.

> **The ledger is what lets the system forget the prose without forgetting the book.**

### Three rules this layer encodes

1. **Relevance beats recency.** The three most *semantically related* written sections are more
   useful than the three most recent. Sections 12 and 88 can be neighbours in meaning while being
   200 pages apart.
2. **Definitions travel with their terms.** A bare list of term names is useless. The Writer needs
   the definition in order to reuse it consistently, and the alias list in order to know which
   variants to avoid.
3. **Budgets are declared, not discovered.** Every block gets an explicit allowance and overflow is
   a *logged truncation* rather than a mystery. Having a large window makes this **more** important,
   not less: at a small window an over-stuffed prompt failed loudly; at 256K it just gets slow and
   vague.

The design also flags the ledger slice as the weakest link — it selects concepts by tag overlap, and
thin or wrong tags mean the Writer gets the wrong fifteen concepts. So the stated fallback is
implemented: **if tag overlap yields fewer than five concepts, fall back to embedding similarity
against the section title.**

### Layers 5 and 6 are disposable

**Classroom** holds one section's working state — plan version, drift count, doubts resolved and
deferred — and is thrown away when the section ships. One thing it does first matters a great deal:

> *"A deferred doubt is a promise in disguise. Promote it to the ledger so the book actually owes
> the reader an answer later."*

A deferred doubt is the most honest signal available about a real gap: a simulated reader hit
something confusing and the Writer said "later". If "later" never arrives, that is failure #3.

**Conversation** holds the Writer/Reviewer/Student turns and compacts them when they grow, keeping
recent exchanges verbatim and summarising older ones into a **fixed template** — because a free-form
"summarize the key points" produces a different shape every time and downstream prompts cannot rely
on it. And the summary is **rewritten, not stacked**, or it grows without bound and turns into a
game of telephone.

In [ ]:
# ============================================================================
# Phase 3.1 — Layer 4: The Context Assembler
# ============================================================================


class ContextAssembler:
    """
    Turns the whole book memory into one prompt for ONE section.
    Budgets are explicit so nothing silently overflows.
    """

    BUDGETS = {                    # tokens
        "constitution":     700,
        "book_spine":     6_000,   # chapter rollups + all section titles
        "ledger_slice":  12_000,   # only the facts relevant to this section
        "neighbors":     14_000,   # previous section + 3 nearest by meaning
        "source":        60_000,   # assigned chunks + neighbourhoods + related
        "figures":        4_000,   # 2-3 actual images
        "conversation":  12_000,   # this section's dialogue so far
        "draft_so_far":   6_000,
    }

    MIN_LEDGER_CONCEPTS = 5        # below this, fall back to similarity
    MAX_SLICE_CONCEPTS = 15
    MAX_FIGURES = 3

    def __init__(self, constitution: Constitution, ledger: BookLedger,
                 drafts: DraftStore, source: SourceMemory, toc: Dict[str, Any]):
        self.constitution = constitution
        self.ledger = ledger
        self.drafts = drafts
        self.source = source
        self.toc = toc
        self.sections_by_id = {s["section_id"]: s for s in toc["sections"]}
        self.truncations: List[Dict] = []

    # ------------------------------------------------------------- budgeting --
    def _fits(self, text: str, block: str, margin: int = 0) -> bool:
        return count_tokens(text) + margin <= self.BUDGETS[block]

    def _truncate(self, text: str, block: str) -> str:
        """Overflow is a logged truncation, never a silent one."""
        budget = self.BUDGETS[block]
        if count_tokens(text) <= budget:
            return text
        # ~4 chars per token, trimmed conservatively then verified
        keep = budget * 4
        cut = text[:keep]
        while count_tokens(cut) > budget and len(cut) > 100:
            cut = cut[:int(len(cut) * 0.9)]
        self.truncations.append({"block": block, "budget": budget,
                                 "was": count_tokens(text)})
        logC.warning(f"  context: '{block}' truncated "
                     f"{count_tokens(text)} -> {budget} tokens")
        return cut + f"\n\n[... {block} truncated to fit its budget ...]"

    # ------------------------------------------------------------ 2. the spine --
    def _render_spine(self, chapter_id: str) -> str:
        """Where we are in the whole arc: rollups elsewhere, titles everywhere."""
        lines = [f"=== THE BOOK'S ARC ===",
                 f"\"{self.toc['book_title']}\" — {len(self.toc['chapters'])} chapters, "
                 f"{len(self.toc['sections'])} sections."]

        rollups = self.ledger._cache["chapter_rollups"]
        for chapter in self.toc["chapters"]:
            cid = chapter["chapter_id"]
            marker = "  <-- YOU ARE HERE" if cid == chapter_id else ""
            lines.append(f"\n{cid}. {chapter['title']}{marker}")
            if cid in rollups:
                lines.append(f"    {rollups[cid]}")
            titles = [s for s in self.toc["sections"] if s["chapter_id"] == cid]
            if cid == chapter_id:
                # our own chapter: titles plus abstracts of what is already written
                for s in titles:
                    summary = self.ledger._cache["section_summaries"].get(s["section_id"])
                    mark = "written" if summary else "not yet written"
                    lines.append(f"    {s['section_id']}: {s['title']}  [{mark}]")
                    if summary and summary.get("abstract"):
                        lines.append(f"        {summary['abstract']}")
            else:
                lines.append("    " + " · ".join(s["title"] for s in titles))
        return "\n".join(lines)

    # ------------------------------------------------- 3. the ledger slice ----
    def _render_ledger_slice(self, section: Dict, tags: List[str],
                             chapter_id: str) -> str:
        """
        The judgement call that makes or breaks this layer.

        Selects by tag overlap, with the design's stated fallback: if that
        yields too few concepts, fall back to embedding similarity against the
        section title, because thin or wrong tags would otherwise leave the
        Writer with nothing.
        """
        lines: List[str] = []
        relevant = self.ledger.concepts_for_tags(tags)
        selection = "tag overlap"

        # A section's own prerequisites are concepts it will certainly use, even
        # though they are not among its tags. Their definitions and aliases have
        # to travel with them or the Writer can reintroduce them under a variant.
        seen = {c["canonical_name"] for c in relevant}
        prereqs = [c for c in self.ledger.defined_prerequisites(tags)
                   if c["canonical_name"] not in seen]
        if prereqs:
            relevant = relevant + prereqs
            selection += f" + {len(prereqs)} defined prerequisite(s)"

        if len(relevant) < self.MIN_LEDGER_CONCEPTS:
            extra = self._concepts_by_similarity(section, exclude=relevant)
            if extra:
                relevant = relevant + extra
                selection = f"tag overlap + similarity fallback ({len(extra)} added)"

        # (a) concepts this section will use, with their agreed definitions
        if relevant:
            lines.append("ALREADY DEFINED — use these exact terms, do not redefine:")
            for c in relevant[:self.MAX_SLICE_CONCEPTS]:
                lines.append(f"  • {c['canonical_name']} "
                             f"(defined in {c['defined_in']}): {c['definition']}")
                if c["aliases"]:
                    lines.append(f"      avoid the variants: "
                                 f"{', '.join(c['aliases'][:4])}")

        # (b) concepts this section needs that nothing has defined yet
        missing = self.ledger.undefined_prerequisites(tags)
        if missing:
            lines.append("\nNOT YET DEFINED ANYWHERE — define briefly if you lean on them:")
            lines.append("  " + ", ".join(missing[:10]))

        # (c) promises this section is expected to keep
        for p in self.ledger.open_promises_for(chapter_id):
            lines.append(f"\nOPEN PROMISE from {p['made_in']}: \"{p['text']}\" — "
                         f"fulfil it here if it fits.")

        # (d) claims that constrain what we may now assert
        want = {t.lower() for t in tags}
        claims = [c for c in self.ledger._cache["claims"][-40:]
                  if {t.lower() for t in c.get("tags", [])} & want]
        if claims:
            lines.append("\nTHE BOOK HAS ALREADY ASSERTED — do not contradict:")
            for c in claims[:8]:
                lines.append(f"  • {c['text']}  ({c['section_id']})")

        # (e) where the running examples stand right now
        for name, ex in self.ledger._cache["examples"].items():
            lines.append(f"\nRUNNING EXAMPLE '{name}' currently: {ex.get('current_state', '?')} "
                         f"(last touched {ex.get('last_touched', '?')})")

        if not lines:
            return "(The book has not yet defined anything relevant to this section.)"
        return f"[ledger slice selected by: {selection}]\n" + "\n".join(lines)

    def _concepts_by_similarity(self, section: Dict, exclude: List[Dict]) -> List[Dict]:
        """Fallback when tags are thin: cheap lexical overlap on the title."""
        seen = {c["canonical_name"] for c in exclude}
        words = {w for w in re.findall(r"[a-z]{4,}", section["title"].lower())}
        scored = []
        for c in self.ledger._cache["concepts"].values():
            if not c.get("definition") or c["canonical_name"] in seen:
                continue
            name_words = set(re.findall(r"[a-z]{4,}", c["canonical_name"].lower()))
            overlap = len(words & name_words)
            if overlap:
                scored.append((overlap, c))
        scored.sort(key=lambda kv: -kv[0])
        return [c for _, c in scored[:self.MIN_LEDGER_CONCEPTS]]

    # -------------------------------------------------------- 4. neighbours ---
    def _render_neighbors(self, section: Dict, prev_section_id: Optional[str]) -> str:
        """Previous section in full, plus the nearest written sections by meaning."""
        parts: List[str] = []

        if prev_section_id:
            prev_text = self.drafts.get_full(prev_section_id)
            if prev_text:
                prev_title = self.sections_by_id.get(prev_section_id, {}).get("title", "")
                parts.append(f"=== THE PREVIOUS SECTION ({prev_section_id}: "
                             f"{prev_title}) — IN FULL ===\n{prev_text}")

        query = f"{section['title']} {' '.join(section.get('tags', []))}"
        for sim in self.drafts.find_similar(query, k=3, exclude=section["section_id"]):
            if sim["section_id"] == prev_section_id:
                continue
            body = self.drafts.get_full(sim["section_id"])
            if body:
                parts.append(f"=== NEAREST BY MEANING: {sim['section_id']} "
                             f"({sim['title']}, {sim['similarity']:.0%} similar) ===\n{body}")

        return "\n\n".join(parts) if parts else "(Nothing written yet — this is the opening.)"

    # ----------------------------------------------------------- 6. figures ---
    def _select_figures(self, chunk_ids: List[str], limit: int = 3) -> List[Dict]:
        """
        Two or three actual images, preferring ones no section has leaned on
        yet. Usage lives in the LEDGER (book state), not in Pipeline A's
        read-only chunk metadata -- the orchestrator records it after each
        section ships.
        """
        figures = self.source.get_figures(chunk_ids)
        figures.sort(key=lambda f: (self.ledger.figure_use_count(f["id"]), f["id"]))
        return figures[:limit]

    def _render_figures_text(self, figures: List[Dict]) -> str:
        """
        The textual half of figure delivery: id, kind and description, so the
        Writer can reference a figure by id even when the image itself cannot
        be attached. The image half rides along in write_step.
        """
        if not figures:
            return ""
        lines = ["These figures from the source are available. Reference one in "
                 "prose as (see Figure: <id>). The image itself is attached when "
                 "the crop exists on disk."]
        for f in figures:
            note = (" [already shown earlier in the book — do not re-explain it]"
                    if self.ledger.figure_use_count(f["id"]) else "")
            lines.append(f"- {f['id']} ({f['kind']}, from {f['chunk_id']}): "
                         f"{(f.get('description') or '')[:300]}{note}")
        return "\n".join(lines)

    # -------------------------------------------------------------- assemble --
    def assemble(self, section: Dict, prev_section_id: Optional[str]) -> Dict[str, Any]:
        chapter_id = section["chapter_id"]
        tags = section.get("tags", [])
        blocks: Dict[str, Any] = {}

        # 1. Constitution, always, in full.
        blocks["constitution"] = self.constitution.get_prompt_injection()

        # 2. Where we are in the whole arc.
        blocks["book_spine"] = self._truncate(self._render_spine(chapter_id), "book_spine")

        # 3. Only the ledger facts that touch this section.
        blocks["ledger_slice"] = self._truncate(
            self._render_ledger_slice(section, tags, chapter_id), "ledger_slice")

        # 4. The previous section, plus the nearest written sections by meaning.
        blocks["neighbors"] = self._truncate(
            self._render_neighbors(section, prev_section_id), "neighbors")

        # 5. Source: assigned first, related material to fill what is left.
        assigned = self.source.get_assigned(section["chunk_ids"])
        if self._fits(assigned, "source", margin=2_000):
            assigned += self.source.find_related(
                f"{section['title']} {' '.join(tags)}",
                exclude=section["chunk_ids"], k=2)
        blocks["source"] = self._truncate(assigned, "source")

        # 6. Two or three actual images, plus their textual half.
        blocks["figures"] = self._select_figures(section["chunk_ids"], self.MAX_FIGURES)
        blocks["figures_text"] = self._render_figures_text(blocks["figures"])

        blocks["_tokens"] = {k: count_tokens(v) for k, v in blocks.items()
                             if isinstance(v, str)}
        blocks["_tokens"]["TOTAL"] = sum(blocks["_tokens"].values())
        return blocks


assembler = ContextAssembler(constitution, ledger, drafts, source, toc)
print(f"Layer 4 ready — budgets: {assembler.BUDGETS}")

In [ ]:
# ============================================================================
# Phase 3.2 — Layer 5: Classroom Memory
# ============================================================================
"""
One section's working state: which version of the teaching plan is current, how
many times the Writer drifted from it, which student doubts were resolved and
which were deferred. Thrown away when the section ships.
"""


class ClassroomMemory:
    """Layer 5. Per-section, disposable -- but not before close_session runs."""

    def __init__(self, paths: BookPaths):
        self.dir = paths.sessions
        self.dir.mkdir(parents=True, exist_ok=True)
        self.sessions: Dict[str, Dict[str, Any]] = {}

    def start_session(self, section_id: str, section: Dict) -> Dict[str, Any]:
        self.sessions[section_id] = {
            "section_id": section_id,
            "title": section["title"],
            "started_at": datetime.now().isoformat(timespec="seconds"),
            "plan": None,
            "plan_version": 0,
            "plan_rejections": [],
            "drift_count": 0,
            "steps_completed": 0,
            "student_interactions": {"doubts_resolved": [], "doubts_deferred": []},
        }
        return self.sessions[section_id]

    def get_session(self, section_id: str) -> Dict[str, Any]:
        return self.sessions.setdefault(section_id, {
            "section_id": section_id, "plan": None, "plan_version": 0,
            "plan_rejections": [], "drift_count": 0, "steps_completed": 0,
            "student_interactions": {"doubts_resolved": [], "doubts_deferred": []}})

    def set_plan(self, section_id: str, plan: Dict, rejected_feedback: str = "") -> None:
        s = self.get_session(section_id)
        s["plan"] = plan
        s["plan_version"] += 1
        if rejected_feedback:
            s["plan_rejections"].append(rejected_feedback)

    def record_doubt(self, section_id: str, question: str, resolution: str,
                     deferred: bool = False, suggested_chapter: Optional[str] = None) -> None:
        s = self.get_session(section_id)
        bucket = "doubts_deferred" if deferred else "doubts_resolved"
        s["student_interactions"][bucket].append({
            "question": question, "resolution": resolution,
            "suggested_chapter": suggested_chapter})

    def close_session(self, section_id: str, ledger: BookLedger) -> List[Dict]:
        """
        A deferred doubt is a promise in disguise. Promote it to the ledger so
        the book actually owes the reader an answer later.

        "A deferred doubt is the most honest signal available about a real gap
         in the book: a simulated reader hit something confusing and the Writer
         said 'later'. If 'later' never arrives, that is failure #3."
        """
        session = self.get_session(section_id)
        promoted = []
        for doubt in session["student_interactions"]["doubts_deferred"]:
            promise = {
                "promise_id": f"p_{ledger._cache['version']}_{len(promoted)}",
                "text": f"Address the open question: {doubt['question']}",
                "made_in": section_id,
                "target_hint": doubt.get("suggested_chapter"),
                "status": "open",
                "origin": "deferred_doubt",
            }
            ledger._cache["promises"].append(promise)
            promoted.append(promise)
        if promoted:
            ledger._flush()
            logC.info(f"  [{section_id}] promoted {len(promoted)} deferred doubt(s) "
                      f"to open promises")

        # archive, then discard
        session["closed_at"] = datetime.now().isoformat(timespec="seconds")
        (self.dir / f"{section_id}.json").write_text(
            json.dumps(session, indent=2, ensure_ascii=False), encoding="utf-8")
        self.sessions.pop(section_id, None)
        return promoted


classroom = ClassroomMemory(BOOK)
print("Layer 5 ready — per-section sessions, deferred doubts become promises")

In [ ]:
# ============================================================================
# Phase 3.3 — Layer 6: Conversation Memory
# ============================================================================
"""
The Writer, Reviewer and Student exchange a lot of turns while a section is
drafted. This layer holds them, and compacts them when they get long.

Anchored compaction: keep the most recent exchanges word for word, and
summarize everything older into one structured block.
"""

COMPACTION_TEMPLATE = """Summarize this teaching conversation using EXACTLY this structure.
Be terse. Omit any section that has no content.

## Decisions Made
- (structural or pedagogical choices that must persist)

## Definitions Given
- term: definition

## Student Doubts — Resolved
- doubt → how it was answered

## Student Doubts — Deferred
- doubt (and where it should be answered)

## Open Threads
- anything the next step must pick up

CONVERSATION:
{conversation_text}
"""


class ConversationMemory:
    """Layer 6. Per-section dialogue, with anchored compaction."""

    def __init__(self, paths: BookPaths, llm, threshold: int = 12, keep_recent: int = 6):
        self.dir = paths.history
        self.dir.mkdir(parents=True, exist_ok=True)
        self.llm = llm
        self.threshold = threshold
        self.keep_recent = keep_recent
        self.histories: Dict[str, Dict[str, Any]] = {}
        self.compactions = 0

    def _history(self, section_id: str) -> Dict[str, Any]:
        return self.histories.setdefault(
            section_id, {"exchanges": [], "compressed_context": ""})

    def add(self, section_id: str, role: str, content: str) -> None:
        h = self._history(section_id)
        h["exchanges"].append({"role": role, "content": content,
                               "at": datetime.now().isoformat(timespec="seconds")})
        if len(h["exchanges"]) >= self.threshold:
            self._compact(section_id)

    def _compact(self, section_id: str) -> None:
        """
        Rewrite the summary, do not stack it.

        The naive approach appends each new summary to the old one, which grows
        without bound and turns into a game of telephone. Instead, feed the old
        summary AND the new exchanges to the model and ask for one merged
        summary in the same template. Fixed size, no drift.
        """
        h = self._history(section_id)
        older = h["exchanges"][:-self.keep_recent]
        if not older:
            return

        text = "\n\n".join(f"[{e['role'].upper()}]\n{e['content']}" for e in older)
        prompt = COMPACTION_TEMPLATE.format(
            conversation_text=(f"[EXISTING SUMMARY]\n{h['compressed_context']}\n\n"
                               f"[NEW EXCHANGES]\n{text}")
            if h["compressed_context"] else text)

        try:
            h["compressed_context"] = self.llm.generate(
                "You compress teaching conversations into a fixed structure. "
                "Return only the structured summary.",
                prompt, max_tokens=600, temperature=0.0)
            h["exchanges"] = h["exchanges"][-self.keep_recent:]
            self.compactions += 1
            logC.info(f"  [{section_id}] conversation compacted "
                      f"({len(older)} exchanges -> summary)")
        except Exception as exc:
            # Losing recent turns is worse than a long prompt: keep them.
            logC.warning(f"  [{section_id}] compaction failed ({exc}); keeping raw")

    def render(self, section_id: str) -> str:
        h = self._history(section_id)
        parts = []
        if h["compressed_context"]:
            parts.append(f"=== EARLIER IN THIS SECTION (summarized) ===\n"
                         f"{h['compressed_context']}")
        if h["exchanges"]:
            parts.append("=== RECENT EXCHANGES ===\n" + "\n\n".join(
                f"[{e['role'].upper()}]\n{e['content']}" for e in h["exchanges"]))
        return "\n\n".join(parts)

    def close(self, section_id: str) -> None:
        """Write the dialogue to disk for debugging, then clear it."""
        h = self.histories.pop(section_id, None)
        if h:
            (self.dir / f"{section_id}_chat.json").write_text(
                json.dumps(h, indent=2, ensure_ascii=False), encoding="utf-8")


conversation = ConversationMemory(BOOK, llm, threshold=wcfg.compaction_threshold)
print("Layer 6 ready — anchored compaction at "
      f"{wcfg.compaction_threshold} exchanges, template-fixed")

In [ ]:
# ============================================================================
# Phase 3.4 — The MemoryManager facade
# ============================================================================
"""
The single object the orchestrator talks to. Everything above stays addressable
(`memory.ledger`, `memory.drafts`, ...) because the agents genuinely need
different layers -- but the orchestrator should not have to wire seven of them
together on every call.
"""


class MemoryManager:
    def __init__(self, constitution, ledger, drafts, source, assembler,
                 classroom, conversation):
        self.constitution = constitution   # L0
        self.ledger = ledger               # L1
        self.drafts = drafts               # L2
        self.source = source               # L3
        self.assembler = assembler         # L4
        self.classroom = classroom         # L5
        self.conversation = conversation   # L6

    def build_context(self, section: Dict, prev_section_id: Optional[str]) -> Dict:
        return self.assembler.assemble(section, prev_section_id)

    def previous_tail(self, prev_section_id: Optional[str], n: int = 800) -> str:
        return self.drafts.get_tail(prev_section_id, n)

    def snapshot(self) -> Dict[str, Any]:
        return {"ledger": self.ledger.stats(),
                "sections_on_disk": len(self.drafts.written_ids()),
                "compactions": self.conversation.compactions,
                "truncations": len(self.assembler.truncations)}


memory = MemoryManager(constitution, ledger, drafts, source, assembler,
                       classroom, conversation)

print("=" * 72)
print("PHASE 3 COMPLETE — Context Assembler and working memory")
print("=" * 72)

# Assemble a real prompt for the first section and show the token budget in use,
# so the assembler is something you have measured rather than trusted.
_first = toc["sections"][0]
_ctx = memory.build_context(_first, prev_section_id=None)
print(f"Assembled context for {_first['section_id']} — \"{_first['title']}\"")
print(f"{'block':<16}{'tokens':>10}   {'budget':>8}")
print("-" * 40)
for block, tokens in _ctx["_tokens"].items():
    if block == "TOTAL":
        continue
    budget = ContextAssembler.BUDGETS.get(block, "-")
    flag = "  OVER" if isinstance(budget, int) and tokens > budget else ""
    print(f"{block:<16}{tokens:>10}   {budget:>8}{flag}")
print("-" * 40)
print(f"{'TOTAL':<16}{_ctx['_tokens']['TOTAL']:>10}")
print(f"\nFigures attached     : {len(_ctx['figures'])}")
print(f"Truncations so far   : {len(assembler.truncations)}")
print("=" * 72)

## Phase 4 — The Five Agents

All five are the same model with different system prompts and radically different context.

| Agent | Runs | Job | Sees |
|---|---|---|---|
| **Writer** | many times per section | Plans the section, writes each step, clarifies when the Student is confused | The full assembled context |
| **Reviewer** | once or twice per section | Approves or rejects the teaching plan on pedagogy and fit with the source | The plan, the source, the style guide |
| **Student** | once per step | Reads the drafted prose as a target reader would, returns `UNDERSTOOD` or `DOUBT` | The step's prose only |
| **Editor** | once per section | Melts the drafted steps and their clarifications into one continuous section | The raw steps, the style guide, the previous section's ending |
| **Archivist** | once per section | Reads the finished section and catalogues it into the ledger | The finished text only |

### The Writer and the Student are the drafting loop

The Writer does **not** "chat about" the topic and then convert the chat into a chapter — that
approach produces mush. The Writer drafts **actual book prose** for each step of the plan. The
Student then reads that prose as a target reader would and either accepts it or raises a specific
doubt. If a doubt is raised, the Writer clarifies, and the loop continues.

So by the end of the teaching loop you have finalized prose for every step. What you do not have is
a *section*.

### Why the Editor exists

Look at what a four-step teaching loop actually leaves behind:

```
[step 1 prose]                  ← written cold, opens with its own mini-introduction
[step 1 clarification]          ← appended after the Student raised a doubt
[step 2 prose]                  ← opens cold again, may re-state step 1's setup
[step 3 prose]
[step 3 clarification]
[step 3 second clarification]   ← the Student pushed twice
[step 4 prose]                  ← ends abruptly, nothing was written to close the section
```

Concatenated, that is not a chapter. Three problems are structural:

1. **Seams.** Each step was generated as a standalone unit, so each opens by orienting the reader
   and closes by wrapping up. Four openings and four closings inside one 800-word section.
2. **Bolted-on clarifications.** A doubt resolved after step 2 lands at the *end* of step 2, in a
   question-and-answer voice, instead of being folded into the explanation where it belonged. *The
   Student's question is a signal that the prose was unclear. The right response is to fix the
   prose, not to staple an answer to it.*
3. **No section-level arc.** Nobody wrote the first sentence of the section or the last. Step 1
   opened it by accident and step 4 closed it by running out.

**Code blocks never reach the Editor.** The biggest risk in letting a model "improve flow" is that
it silently edits a code sample and breaks it. Do not rely on an instruction for this — make it
mechanically impossible by masking fenced blocks to `[[CODE_BLOCK_0]]` placeholders and restoring
them afterwards.

There is a second benefit that costs nothing: because the Editor writes `closing_line` on purpose
rather than letting step 4 trail off, the ending the *next* section inherits is a deliberate
handoff.

### Why the Archivist is separate, and runs after the fact

You could ask the Writer to emit this metadata alongside its prose. Do not:

1. **Writers are unreliable narrators.** Asked to report what it defined, a writer reports what it
   *intended* to define. The Archivist reads the actual output — which is the thing readers read.
2. **Separation of concerns.** The Writer's prompt already carries a constitution, a ledger slice,
   neighbouring sections and tens of thousands of tokens of source. Adding a cataloguing task
   degrades the prose.
3. **It is cheap.** One structured call per section, low temperature, no source material.

**The Archivist is the arrow that points from a finished section back into memory.** Everything in
Phases 1–3 is inert without it.

In [ ]:
# ============================================================================
# Phase 4.1 — BaseAgent
# ============================================================================
"""
All five agents share one execution path: build a prompt, call the model, get
either free text or a validated JSON object back.

Structured calls go through `generate_structured`, which re-prompts with the
parser's own error on a malformed reply -- so a stray brace costs a retry, not
the whole generation.
"""


class BaseAgent:
    """Common execution for every agent."""

    name = "BaseAgent"

    def __init__(self, llm, cfg: WriterConfig):
        self.llm = llm
        self.cfg = cfg
        self.calls = 0
        self.failures = 0

    def _execute_step(self, prompt_text: str, system_prompt: str,
                      structured_schema: Optional[Dict] = None,
                      max_tokens: int = 1_200,
                      temperature: Optional[float] = None,
                      images: Optional[List[Any]] = None) -> Any:
        self.calls += 1
        tokens_in = count_tokens(system_prompt) + count_tokens(prompt_text)
        if tokens_in > 200_000:
            logC.warning(f"[{self.name}] prompt is {tokens_in:,} tokens — "
                         f"the assembler's budgets are being exceeded upstream")

        try:
            if structured_schema is not None:
                result = self.llm.generate_structured(
                    system_prompt, prompt_text, structured_schema,
                    max_tokens=max_tokens,
                    temperature=self.cfg.temperature_structured
                    if temperature is None else temperature)
                if result is None:
                    self.failures += 1
                return result
            kwargs: Dict[str, Any] = {}
            if images:
                # the Writer is a multimodal model: attach the actual figure
                # crops so it reads the diagram, not a summary of the diagram
                kwargs["images"] = images
            return self.llm.generate(
                system_prompt, prompt_text, max_tokens=max_tokens,
                temperature=self.cfg.temperature_prose
                if temperature is None else temperature, **kwargs)
        except Exception as exc:
            self.failures += 1
            logC.error(f"[{self.name}] step failed: {exc}")
            raise


def render_context(blocks: Dict[str, Any], include: Tuple[str, ...]) -> str:
    """Stitch selected assembler blocks into one prompt body, in a fixed order."""
    titles = {
        "constitution": "",                       # already self-labelled
        "book_spine": "",
        "ledger_slice": "=== WHAT THE BOOK ALREADY KNOWS ===",
        "neighbors": "=== SURROUNDING SECTIONS ===",
        "source": "=== SOURCE MATERIAL ===",
        "figures_text": "=== FIGURES AVAILABLE ===",
    }
    parts = []
    for key in include:
        body = blocks.get(key)
        if isinstance(body, str) and body.strip():
            head = titles.get(key, f"=== {key.upper()} ===")
            parts.append(f"{head}\n{body}".strip())
    return "\n\n".join(parts)


print("Phase 4.1 — BaseAgent ready")

In [ ]:
# ============================================================================
# Phase 4.2 — The Writer
# ============================================================================
"""
The Writer plans the section, drafts each step as ACTUAL BOOK PROSE, and
clarifies when the Student is confused.

It never "chats about" the topic: every generation is text that could appear in
the book unchanged.
"""

PLAN_SCHEMA = {
    "steps": [{"title": "string — the teaching move, not a heading",
               "topic": "string — what this step establishes, one sentence",
               "uses_chunks": ["string — chunk ids this step draws on"]}],
    "teaches": ["string — the concepts this section will define or explain"],
    "assumes": ["string — concepts used but NOT defined here"],
    "opening_promise": "string — what the reader will be able to do afterwards",
}

WRITER_SYSTEM = """You are the Writer of a technical book. You produce finished book prose, not notes and not chat.

RULES THAT OVERRIDE EVERYTHING ELSE:
1. Use the book's canonical terms exactly as the ledger gives them. If a term is listed with
   variants to avoid, never use a variant.
2. Never redefine a concept the ledger says is already defined. Reference it and build on it.
3. Never re-explain source material marked "already used in <section>". Build on it instead.
4. Honour the running examples. Do not invent a new one.
5. Write in the book's voice, for the book's stated reader.
6. Never write a phrase on the forbidden list."""


class WriterAgent(BaseAgent):
    name = "Writer"

    def create_teaching_plan(self, section: Dict, context: Dict,
                             feedback: str = "") -> Optional[Dict]:
        """Plan the section as a sequence of teaching moves."""
        body = render_context(context, ("constitution", "book_spine",
                                        "ledger_slice", "neighbors", "source",
                                        "figures_text"))
        target = section.get("estimated_word_count", self.cfg.default_word_count)
        n_min, n_max = self.cfg.steps_per_section_min, self.cfg.steps_per_section_max

        task = f"""{body}

=== YOUR TASK ===
Plan the section "{section['title']}" ({section['section_id']}, chapter {section['chapter_id']}).
It should teach: {', '.join(section.get('tags', [])) or '(see the source)'}
Target length: {target} words total, so plan {n_min}-{n_max} steps of roughly
{target // max(1, n_min):d}-{target // max(1, n_max):d} words each.

A step is a teaching MOVE, not a heading: "show why the naive loop fails" beats "Overview".
Plan only what the source material can actually support.
"""
        if feedback:
            task += f"\n=== YOUR PREVIOUS PLAN WAS REJECTED ===\n{feedback}\nAddress this.\n"

        return self._execute_step(task, WRITER_SYSTEM, PLAN_SCHEMA,
                                  max_tokens=self.cfg.plan_max_tokens)

    def write_step(self, section: Dict, plan: Dict, step_index: int,
                   context: Dict, drafted_so_far: str, prev_tail: str,
                   conversation: str = "") -> str:
        """Draft one step as finished book prose, with the figures attached."""
        step = plan["steps"][step_index]
        body = render_context(context, ("constitution", "ledger_slice", "source",
                                        "figures_text"))

        # The image half of figure delivery: load the crops that exist and pass
        # them to the multimodal model alongside the prompt.
        images: List[Any] = []
        for fig in (context.get("figures") or []):
            path = fig.get("path")
            if path and Path(path).exists():
                try:
                    from PIL import Image
                    img = Image.open(path)
                    img.load()
                    images.append(img)
                except Exception:
                    pass

        dialogue = ""
        if conversation:
            dialogue = ("\n=== THE TEACHING DIALOGUE SO FAR (doubts already "
                        "resolved — do not re-trigger them) ===\n"
                        f"{conversation[-6000:]}\n")
        target = section.get("estimated_word_count", self.cfg.default_word_count)
        per_step = max(120, target // max(1, len(plan["steps"])))

        continuity = ""
        if step_index == 0 and prev_tail:
            continuity = (f"\n=== HOW THE PREVIOUS SECTION ENDED (continue from this "
                          f"voice; do not restate it) ===\n{prev_tail}\n")
        elif drafted_so_far:
            continuity = (f"\n=== WHAT YOU HAVE WRITTEN IN THIS SECTION SO FAR ===\n"
                          f"{drafted_so_far[-2500:]}\n")

        task = f"""{body}
{continuity}{dialogue}
=== THE PLAN FOR THIS SECTION ===
{chr(10).join(f"{i + 1}. {s['title']} — {s.get('topic', '')}"
              for i, s in enumerate(plan['steps']))}

=== YOUR TASK ===
Write step {step_index + 1} of {len(plan['steps'])}: "{step['title']}"
What it must establish: {step.get('topic', '')}
Length: about {per_step} words.

Write finished book prose. No headings, no "In this step", no meta-commentary about
what you are about to do. Code goes in fenced blocks and must follow the book's
code conventions. Do not summarise or conclude — later steps continue from here.
"""
        return self._execute_step(task, WRITER_SYSTEM,
                                  max_tokens=self.cfg.step_max_tokens,
                                  images=images or None)

    def clarify(self, section: Dict, step_prose: str, doubt: str,
                context: Dict) -> str:
        """Answer a Student doubt — in book prose, not in a Q&A voice."""
        task = f"""{render_context(context, ("constitution", "ledger_slice"))}

=== THE PROSE A READER FOUND UNCLEAR ===
{step_prose}

=== WHAT CONFUSED THEM ===
{doubt}

=== YOUR TASK ===
Write the additional book prose that resolves this. One or two short paragraphs.
Write it as it would appear in the book — do NOT address the reader's question
directly, do not write "You might wonder", and do not repeat what is already above.
The Editor will fold this into the passage where the confusion arises.
"""
        return self._execute_step(task, WRITER_SYSTEM,
                                  max_tokens=self.cfg.step_max_tokens // 2)


writer = WriterAgent(llm, wcfg)
print("Phase 4.2 — Writer ready (plan · write_step · clarify)")

In [ ]:
# ============================================================================
# Phase 4.3 — The Reviewer and the Student
# ============================================================================
"""
The Reviewer judges the PLAN, before any prose exists -- catching a problem here
costs one cheap call instead of one expensive generation plus a rewrite.

The Student judges the PROSE, and sees nothing else: no plan, no source, no
ledger. It has to react the way a real reader would, and a real reader does not
have the source material open beside them.
"""

REVIEW_SCHEMA = {
    "approved": "boolean",
    "feedback": "string — what to change, concrete, empty if approved",
    "concerns": ["string — specific pedagogical or grounding problems"],
}

REVIEWER_SYSTEM = """You are the Reviewer. You judge a teaching plan before any prose is written.

Approve or reject on three grounds only:
1. PEDAGOGY — do the steps build in an order a reader can follow? Is anything assumed too early?
2. GROUNDING — can the assigned source material actually support every step, or is the plan
   inventing content the sources do not contain?
3. FIT — does the plan match the section's stated scope and length, without sprawling into
   what neighbouring sections cover?

You are not judging prose quality; none exists yet. Be decisive: approve a workable plan
rather than holding out for a perfect one. Reject only for a concrete, fixable problem."""

STUDENT_SCHEMA = {
    "verdict": "UNDERSTOOD|DOUBT",
    "doubt": "string — the single most blocking question, empty if UNDERSTOOD",
    "can_wait": "boolean — true if this is better answered in a later section",
}

STUDENT_SYSTEM = """You are a target reader of this book, reading a passage for the first time.

You have the book's assumed prerequisites and nothing else. You cannot see the source
material, the plan, or any other section.

Read the passage and answer honestly:
- UNDERSTOOD — you followed it and could explain it back.
- DOUBT — something specific blocks you. State the ONE most blocking question.

Raise a doubt only for something genuinely unclear or unexplained: a term used but never
defined, a leap in reasoning, a claim with no support. Do NOT raise a doubt because the
passage is incomplete — passages continue in later steps. Do not ask for more examples
out of politeness. Most well-written passages are UNDERSTOOD."""


class ReviewerAgent(BaseAgent):
    name = "Reviewer"

    def review_plan(self, section: Dict, plan: Dict, context: Dict) -> Dict:
        source_excerpt = (context.get("source") or "")[:12_000]
        task = f"""{constitution.get_style_injection()}

=== THE SECTION ===
{section['section_id']}: "{section['title']}" (chapter {section['chapter_id']})
Concepts it should teach: {', '.join(section.get('tags', []))}
Target length: {section.get('estimated_word_count', 700)} words

=== THE PROPOSED PLAN ===
{json.dumps(plan, indent=2)}

=== THE SOURCE MATERIAL AVAILABLE (excerpt) ===
{source_excerpt}

=== YOUR TASK ===
Approve or reject this plan.
"""
        result = self._execute_step(task, REVIEWER_SYSTEM, REVIEW_SCHEMA,
                                    max_tokens=self.cfg.review_max_tokens)
        # A failed review must not silently block the book.
        if result is None:
            logC.warning("  reviewer returned nothing — approving by default")
            return {"approved": True, "feedback": "", "concerns": ["reviewer call failed"]}
        return result


class StudentAgent(BaseAgent):
    name = "Student"

    def evaluate(self, prose: str) -> Dict:
        task = f"""=== THE PASSAGE ===
{prose}

=== YOUR TASK ===
Did you follow it? Answer UNDERSTOOD, or state the one thing that blocks you.
"""
        result = self._execute_step(task, STUDENT_SYSTEM, STUDENT_SCHEMA,
                                    max_tokens=self.cfg.student_max_tokens)
        if result is None:
            return {"verdict": "UNDERSTOOD", "doubt": "", "can_wait": False}
        # normalise a chatty verdict
        verdict = str(result.get("verdict", "")).strip().upper()
        result["verdict"] = "DOUBT" if verdict.startswith("DOUBT") else "UNDERSTOOD"
        if result["verdict"] == "DOUBT" and not str(result.get("doubt", "")).strip():
            result["verdict"] = "UNDERSTOOD"      # a doubt with no question is noise
        return result


reviewer = ReviewerAgent(llm, wcfg)
student = StudentAgent(llm, wcfg)
print("Phase 4.3 — Reviewer and Student ready")

In [ ]:
# ============================================================================
# Phase 4.4 — Code masking, and the Editor
# ============================================================================
"""
    "The biggest risk in letting a model 'improve flow' is that it silently
     edits a code sample and breaks it. Do not rely on an instruction for this
     -- make it mechanically impossible."

The Editor sees `[[CODE_BLOCK_0]]`. It can move it, keep it, or write a better
sentence leading into it -- but it cannot touch a character inside it.
"""

FENCE = re.compile(r"```[\s\S]*?```", re.MULTILINE)


def mask_code(text: str) -> Tuple[str, List[str]]:
    blocks = FENCE.findall(text)
    for i, _ in enumerate(blocks):
        text = FENCE.sub(f"[[CODE_BLOCK_{i}]]", text, count=1)
    return text, blocks


def unmask_code(text: str, blocks: List[str]) -> str:
    for i, block in enumerate(blocks):
        # a function replacement keeps backslashes in the code literal
        text = re.sub(re.escape(f"[[CODE_BLOCK_{i}]]"), lambda _m, b=block: b, text)
    return text


EDITOR_SCHEMA = {
    "content": "string — the finished section as markdown",
    "opening_line": "string — the section's first sentence, verbatim",
    "closing_line": "string — the section's final sentence, verbatim",
    "doubts_woven": ["string — each Student doubt folded into the prose"],
    "notes": "string — anything the Editor could not resolve, for the log",
}

EDITOR_SYSTEM = (
    "You are the Editor. You receive the raw output of a teaching session — "
    "several drafted steps plus clarifications the writer added when a reader "
    "got confused — and you turn it into one continuous section of a book.\n\n"
    "YOU MAY: reorder sentences, rewrite transitions, merge or split paragraphs, "
    "fold a clarification into the passage it clarifies, write a real opening "
    "and closing sentence, cut redundancy across steps.\n\n"
    "YOU MAY NOT: add a technical fact, definition, claim, statistic, or forward "
    "reference that is not already present in the input. You may not alter code "
    "blocks in any way — they arrive as placeholders and must be returned "
    "unchanged, in their original order. You may not remove a concept the input "
    "teaches. If something reads badly and you cannot fix it without inventing "
    "content, leave it and say so in `notes`."
)


class EditorAgent(BaseAgent):
    """Melts the seams between teaching steps. Rewrites prose. Never invents content."""

    name = "Editor"

    @staticmethod
    def _render_steps(raw_steps: List[Dict]) -> str:
        parts = []
        for i, step in enumerate(raw_steps, 1):
            parts.append(f"--- STEP {i}: {step['title']} ---\n{step['prose']}")
            for doubt in step.get("clarifications", []):
                parts.append(f"--- CLARIFICATION added after step {i} "
                             f"(reader asked: \"{doubt['question']}\") ---\n"
                             f"{doubt['answer']}")
        return "\n\n".join(parts)

    def smooth_section(self, section: Dict, raw_steps: List[Dict],
                       constitution: Constitution, prev_tail: str = "") -> Optional[Dict]:
        task = f"""SECTION: {section['title']}  ({section['section_id']})
TARGET LENGTH: {section.get('estimated_word_count', 800)} words (±15%)

{constitution.get_style_injection()}

HOW THE PREVIOUS SECTION ENDED (open so this follows on, do not restate it):
{prev_tail or '(this is the opening section of the book)'}

RAW TEACHING OUTPUT — steps in order, with the doubts each one triggered:
{self._render_steps(raw_steps)}

Produce the finished section.
"""
        return self._execute_step(task, EDITOR_SYSTEM, EDITOR_SCHEMA,
                                  max_tokens=self.cfg.editor_max_tokens)


editor = EditorAgent(llm, wcfg)
print("Phase 4.4 — Editor ready; code blocks masked before it ever sees them")

In [ ]:
# ============================================================================
# Phase 4.5 — The Archivist: the write-back loop
# ============================================================================
"""
Everything in Phases 1-3 is inert without this. The Archivist is the arrow that
points from a finished section back into memory.

It reads the EDITED text, not the raw steps, because that is what readers will
read -- and it is told to be literal: "Record what the text actually says, not
what it should have said."
"""

ARCHIVIST_SCHEMA = {
    "summary": {
        "abstract": "string — 60-100 words, what this section actually taught",
        "teaches": ["string — concepts introduced or explained here"],
        "assumes": ["string — concepts used but not defined here"],
        "closing_line": "string — the section's final sentence, verbatim",
    },
    "concepts_defined": [{"term": "string",
                          "definition": "string — one sentence, as written"}],
    "concepts_referenced": ["string"],
    "claims": [{"claim_id": "string", "text": "string",
                "confidence": "strong|moderate|tentative"}],
    "promises_made": [{"promise_id": "string", "text": "string",
                       "target_hint": "string chapter id"}],
    "promises_fulfilled": ["string — promise_ids this section delivered on"],
    "example_states": {"ExampleName": "string — its state after this section"},
    "chunks_used": {"chunk_id": "primary|supporting|mentioned"},
}

ARCHIVIST_SYSTEM = (
    "You are the Archivist. You do not write or edit — you catalogue.\n"
    "You read a finished book section and extract a precise, structured record "
    "of what it defined, asserted, promised, demonstrated, and consumed.\n"
    "Be literal. Record what the text actually says, not what it should have said."
)


class ArchivistAgent(BaseAgent):
    """Reads a finished section and harvests it into the Book Ledger."""

    name = "Archivist"

    def harvest(self, section: Dict, content: str,
                open_promises: List[Dict]) -> Optional[Dict]:
        promise_list = "\n".join(f"  {p['promise_id']}: {p['text']}"
                                 for p in open_promises[:30]) or "  (none)"
        task = f"""SECTION: {section['title']}  ({section['section_id']})
SOURCE CHUNKS THIS SECTION WAS GIVEN: {', '.join(section['chunk_ids']) or '(none)'}

PROMISES CURRENTLY OPEN IN THE BOOK:
{promise_list}

FINISHED SECTION TEXT:
{content}

Catalogue this section:
1. A 60-100 word abstract of what it TAUGHT (not what it was about).
2. Every term it DEFINED, with the definition as written.
3. Every previously-defined term it USED.
4. Every claim a later section could contradict.
5. Every FORWARD PROMISE it made ("we'll cover X later").
6. Any promise from the list above that this section FULFILLED (give its promise_id).
7. The state of any running example after this section.
8. Which given chunks it actually used, and how deeply.
"""
        return self._execute_step(task, ARCHIVIST_SYSTEM, ARCHIVIST_SCHEMA,
                                  max_tokens=self.cfg.archivist_max_tokens)

    @staticmethod
    def fallback_update(section: Dict, content: str) -> Dict:
        """
        What to write into the ledger when the Archivist call fails outright.

        A missing harvest is recoverable; an empty one that looks complete is
        not. So the fallback records the mechanical facts we can be certain of
        -- the section exists, it is this long, it consumed these chunks -- and
        deliberately claims no concepts, no definitions and no promises.
        """
        words = content.split()
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", content.strip()) if s.strip()]
        return {
            "summary": {
                "abstract": f"[ARCHIVIST FAILED] {' '.join(words[:60])}…",
                "teaches": [], "assumes": [],
                "closing_line": sentences[-1] if sentences else "",
                "word_count": len(words),
            },
            "concepts_defined": [], "concepts_referenced": [],
            "claims": [], "promises_made": [], "promises_fulfilled": [],
            "example_states": {},
            "chunks_used": {cid: "mentioned" for cid in section["chunk_ids"]},
            "_degraded": True,
        }


archivist = ArchivistAgent(llm, wcfg)

print("=" * 72)
print("PHASE 4 COMPLETE — five agents, one model")
print("=" * 72)
for agent in (writer, reviewer, student, editor, archivist):
    print(f"  {agent.name:<10} -> {type(agent).__name__}")
print(f"\nModel: {llm.model_id}")

# Prove the code mask is airtight before the Editor ever runs unattended.
_sample = ("Intro prose.\n\n```python\ndef f(x):\n    return x ** 2  # keep \\n literal\n```\n\n"
           "Middle prose.\n\n```\nplain block\n```\n\nEnd.")
_masked, _blocks = mask_code(_sample)
assert "def f" not in _masked and _masked.count("[[CODE_BLOCK_") == 2
assert unmask_code(_masked, _blocks) == _sample
print(f"\nCode masking verified: {len(_blocks)} blocks masked and restored byte-exact")
print("=" * 72)

## Phase 5 — The Continuity Gate, the Edit Guard, and the Orchestrator

### The Continuity Gate

The Reviewer checks a plan against the source material and the style guide. **It has no idea the
rest of the book exists.** The Continuity Gate is what gives that check a memory.

It runs **on the plan, before any prose is generated** — catching a problem costs one cheap call
instead of one expensive generation plus a rewrite. And it makes **zero LLM calls**: five checks
built from embedding similarity, dictionary lookups and set arithmetic.

| # | Check | Failure it prevents |
|---|---|---|
| 1 | **Repetition** — is this plan too close to something already written? | #1 |
| 2 | **Redefinition** — is it planning to define an already-defined term? | #2 |
| 3 | **Prerequisites** — does it lean on something not yet taught? | #5 |
| 4 | **Stale sources** — is every assigned chunk already fully mined? | #1 |
| 5 | **Unkept promises** — is this the chapter that owed the reader something? | #3 |

The LLM is reserved for the judgements code cannot make — whether an analogy lands, whether a claim
genuinely contradicts an earlier one.

### The Edit Guard

The Editor is the last thing to touch the text before it becomes the book, and **nobody reviews the
Editor.** So it gets checked mechanically — again with no LLM call:

```python
if any(f"[[CODE_BLOCK_{i}]]" not in edited for i in range(len(masked_blocks))):
    issues.append("dropped or renamed a code block placeholder")
ratio = len(edited.split()) / max(len(raw.split()), 1)
if not 0.75 <= ratio <= 1.15:
    issues.append(f"length changed by {abs(1-ratio):.0%} — expected ±15%")
for term in taught_terms:
    if term.lower() not in edited.lower():
        issues.append(f"concept '{term}' present in draft, absent after edit")
for n in set(re.findall(r"\b\d[\d,.]*\b", raw)) - set(re.findall(r"\b\d[\d,.]*\b", edited)):
    issues.append(f"numeric value {n} disappeared")
```

On failure, retry once with the issues appended. On a second failure, **ship the concatenated raw
draft and log it.**

> *"A slightly bumpy section is a much better outcome than a smooth one that lost a definition.
> Write that fallback — it is what makes the Editor safe to run unattended across 150 sections."*

### The Orchestrator

The whole per-section flow, and one ordering rule that matters:

> **Everything downstream reads the edited text, not the raw steps.**

| Consumer | Reads | Why |
|---|---|---|
| The saved markdown | edited | It is the book |
| The Archivist | edited | It catalogues what the *reader* will see |
| The draft store | edited | `get_tail()` must return the real closing line |
| `close_session` | raw | Deferred doubts are a property of the lesson, not the prose |

The queue checkpoints after every section, so a run that dies at section 90 resumes at 90 — with
the ledger, the draft store and the coverage map all exactly as section 89 left them.

In [ ]:
# ============================================================================
# Phase 5.1 — The Continuity Gate
# ============================================================================


class ContinuityGate:
    """Five deterministic checks on the teaching plan. Zero LLM calls."""

    def __init__(self, ledger: BookLedger, drafts: DraftStore,
                 threshold: float = 0.85):
        self.ledger = ledger
        self.drafts = drafts
        self.REPETITION_THRESHOLD = threshold

    def check_plan(self, section: Dict, plan: Dict) -> Dict:
        issues: List[Dict] = []
        plan_text = " ".join(f"{s.get('title', '')} {s.get('topic', '')}"
                             for s in plan.get("steps", []))

        # 1. REPETITION — is this plan too close to something already written?
        for sim in self.drafts.find_similar(plan_text, k=3,
                                            exclude=section["section_id"]):
            if sim["similarity"] > self.REPETITION_THRESHOLD:
                issues.append({
                    "type": "repetition", "severity": "high",
                    "message": (f"This plan is {sim['similarity']:.0%} similar to "
                                f"'{sim['title']}' ({sim['section_id']}). Reference it "
                                f"and go deeper, or narrow this section's scope."),
                })

        # 2. REDEFINITION — is it planning to define an already-defined term?
        for step in plan.get("steps", []):
            key = self.ledger.resolve_alias(step.get("title", ""))
            if key:
                c = self.ledger._cache["concepts"][key]
                if c["definition"] and c["defined_in"] != section["section_id"]:
                    issues.append({
                        "type": "redefinition", "severity": "medium",
                        "message": (f"'{c['canonical_name']}' was already defined in "
                                    f"{c['defined_in']}. Build on it, do not restate it."),
                    })

        # 3. PREREQUISITES — does it lean on something not yet taught?
        missing = self.ledger.undefined_prerequisites(section.get("tags", []))
        if missing:
            issues.append({
                "type": "prerequisite_gap", "severity": "high",
                "message": (f"Uses concepts the book has not defined yet: "
                            f"{', '.join(missing)}. Define them briefly here, or the "
                            f"reader is lost."),
            })

        # 4. STALE SOURCES — is every assigned chunk already fully mined?
        fresh = [c for c in section["chunk_ids"]
                 if not self.ledger.chunk_already_used(c)]
        if not fresh and section["chunk_ids"]:
            issues.append({
                "type": "no_fresh_source", "severity": "medium",
                "message": ("Every assigned chunk has been used elsewhere. This section "
                            "needs a genuinely new angle or it will read as filler."),
            })

        # 5. UNKEPT PROMISES — is this the chapter that owed the reader something?
        for p in self.ledger.open_promises_for(section["chapter_id"]):
            if p["text"].lower() not in plan_text.lower():
                issues.append({
                    "type": "unfulfilled_promise", "severity": "medium",
                    "message": (f"{p['made_in']} promised: \"{p['text']}\" — this chapter "
                                f"is where it was due. Cover it or move the promise."),
                })

        return {"passed": not any(i["severity"] == "high" for i in issues),
                "issues": issues}


gate = ContinuityGate(ledger, drafts, wcfg.repetition_threshold)
print("Phase 5.1 — Continuity Gate ready (5 checks, 0 LLM calls)")

In [ ]:
# ============================================================================
# Phase 5.2 — The Edit Guard
# ============================================================================
"""
The Editor is the last thing to touch the text before it becomes the book, and
nobody reviews the Editor. These four checks are what make it safe to run
unattended across 150 sections. No LLM call.
"""


def verify_edit(raw: str, edited: str, masked_blocks: List[str],
                taught_terms: List[str]) -> List[str]:
    issues: List[str] = []

    if any(f"[[CODE_BLOCK_{i}]]" not in edited for i in range(len(masked_blocks))):
        issues.append("dropped or renamed a code block placeholder")

    ratio = len(edited.split()) / max(len(raw.split()), 1)
    if not 0.75 <= ratio <= 1.15:
        issues.append(f"length changed by {abs(1 - ratio):.0%} — expected ±15%")

    for term in taught_terms:
        if term and term.lower() not in edited.lower():
            issues.append(f"concept '{term}' present in draft, absent after edit")

    for n in set(re.findall(r"\b\d[\d,.]*\b", raw)) - set(re.findall(r"\b\d[\d,.]*\b", edited)):
        issues.append(f"numeric value {n} disappeared")

    return issues


@dataclass
class WorkQueue:
    """The sections still to write, and what happened to the ones that are done."""

    pending: List[str] = field(default_factory=list)
    completed: List[str] = field(default_factory=list)
    failed: List[Dict] = field(default_factory=list)
    prev_section_id: Optional[str] = None
    started_at: str = ""

    @classmethod
    def build(cls, toc: Dict, resume: bool) -> "WorkQueue":
        path = BOOK.work_queue
        order = [s["section_id"] for s in toc["sections"]]

        if resume and path.exists():
            try:
                q = cls(**json.loads(path.read_text(encoding="utf-8")))
                # trust the disk for what is DONE, the TOC for what remains
                done = set(q.completed)
                q.pending = [sid for sid in order if sid not in done]
                logC.info(f"Resuming: {len(q.completed)} written, "
                          f"{len(q.pending)} remaining")
                return q
            except Exception as exc:
                logC.warning(f"Unreadable work queue ({exc}); starting fresh")

        return cls(pending=order,
                   started_at=datetime.now().isoformat(timespec="seconds"))

    def save(self) -> None:
        BOOK.work_queue.parent.mkdir(parents=True, exist_ok=True)
        BOOK.work_queue.write_text(json.dumps(asdict(self), indent=2),
                                   encoding="utf-8")


print("Phase 5.2 — Edit guard and work queue ready")

In [ ]:
# ============================================================================
# Phase 5.3 — The Orchestrator
# ============================================================================
"""
The per-section flow, in one class.

One ordering rule matters above all: everything downstream reads the EDITED
text, not the raw steps. The saved markdown is the book; the Archivist
catalogues what the reader will see; the draft store's get_tail() must return
the real closing line.

The exception is close_session, which reads the RAW session -- deferred doubts
are a property of the lesson, not of the prose.
"""


class BookOrchestrator:
    def __init__(self, toc: Dict, memory: MemoryManager, agents: Dict[str, BaseAgent],
                 gate: ContinuityGate, cfg: WriterConfig):
        self.toc = toc
        self.sections = {s["section_id"]: s for s in toc["sections"]}
        self.memory = memory
        self.writer = agents["writer"]
        self.reviewer = agents["reviewer"]
        self.student = agents["student"]
        self.editor = agents["editor"]
        self.archivist = agents["archivist"]
        self.gate = gate
        self.cfg = cfg
        self.queue = WorkQueue.build(toc, cfg.resume)
        self.stats = Counter()

    # ------------------------------------------------------------ planning ---
    def _run_planning_loop(self, section: Dict, context: Dict) -> Dict:
        """Plan, gate, review. The gate has memory; the reviewer has judgement."""
        sid = section["section_id"]
        plan, feedback = None, ""

        for attempt in range(1, self.cfg.max_plan_revisions + 1):
            plan = self.writer.create_teaching_plan(section, context, feedback)
            if not plan or not plan.get("steps"):
                feedback = "You returned no steps. Return a plan with 3-5 concrete steps."
                self.stats["plan_empty"] += 1
                continue

            gate_result = self.gate.check_plan(section, plan)
            review = self.reviewer.review_plan(section, plan, context)

            for issue in gate_result["issues"]:
                logC.info(f"  [{sid}] gate {issue['severity']}: {issue['message']}")
                self.stats[f"gate_{issue['type']}"] += 1

            if gate_result["passed"] and review.get("approved"):
                self.memory.classroom.set_plan(sid, plan)
                logC.info(f"  [{sid}] plan approved on attempt {attempt} "
                          f"({len(plan['steps'])} steps)")
                return plan

            feedback = (str(review.get("feedback", "")) + "\n\nCONTINUITY ISSUES:\n"
                        + "\n".join(f"- [{i['severity']}] {i['message']}"
                                    for i in gate_result["issues"]))
            self.memory.classroom.set_plan(sid, plan, feedback)
            self.memory.conversation.add(sid, "reviewer", feedback)
            self.stats["plan_rejected"] += 1
            logC.warning(f"  [{sid}] plan rejected on attempt {attempt}")

        # Out of revisions: proceed with the last plan rather than losing the
        # section. The gate's issues are already logged and counted.
        logC.warning(f"  [{sid}] proceeding with an unapproved plan")
        self.stats["plan_forced"] += 1
        return plan or {"steps": [{"title": section["title"], "topic": ""}],
                        "teaches": section.get("tags", []), "assumes": []}

    # ------------------------------------------------------------ teaching ---
    def _run_teaching_loop(self, section: Dict, plan: Dict,
                           context: Dict, prev_tail: str) -> List[Dict]:
        sid = section["section_id"]
        raw_steps: List[Dict] = []
        drafted = ""

        for index, step in enumerate(plan["steps"]):
            prose = self.writer.write_step(
                section, plan, index, context, drafted, prev_tail,
                conversation=self.memory.conversation.render(sid))
            self.memory.conversation.add(sid, "writer", prose)
            record = {"title": step.get("title", f"Step {index + 1}"),
                      "prose": prose, "clarifications": []}

            # The Student reads the prose as a target reader would.
            for round_no in range(self.cfg.max_doubt_rounds):
                verdict = self.student.evaluate(prose)
                self.memory.conversation.add(sid, "student", json.dumps(verdict))
                if verdict["verdict"] == "UNDERSTOOD":
                    break

                doubt = verdict["doubt"]
                self.stats["doubts_raised"] += 1
                logC.info(f"  [{sid}] step {index + 1} doubt: {doubt[:80]}")

                if verdict.get("can_wait"):
                    # A deferred doubt is a promise in disguise; Layer 5 promotes
                    # it to the ledger when the session closes.
                    self.memory.classroom.record_doubt(
                        sid, doubt, "deferred to a later section", deferred=True,
                        suggested_chapter=None)
                    self.stats["doubts_deferred"] += 1
                    break

                answer = self.writer.clarify(section, prose, doubt, context)
                self.memory.conversation.add(sid, "writer", answer)
                record["clarifications"].append({"question": doubt, "answer": answer})
                self.memory.classroom.record_doubt(sid, doubt, answer, deferred=False)
                self.stats["doubts_resolved"] += 1
                # re-read the prose WITH the clarification appended
                prose = f"{prose}\n\n{answer}"

            raw_steps.append(record)
            drafted += ("\n\n" + record["prose"] + "".join(
                "\n\n" + c["answer"] for c in record["clarifications"]))
            self.memory.classroom.get_session(sid)["steps_completed"] = index + 1

        return raw_steps

    # -------------------------------------------------------------- editing --
    @staticmethod
    def _concat_steps(raw_steps: List[Dict]) -> str:
        parts = []
        for step in raw_steps:
            parts.append(step["prose"])
            parts.extend(c["answer"] for c in step["clarifications"])
        return "\n\n".join(p for p in parts if p and p.strip())

    def _edit(self, section: Dict, raw_steps: List[Dict], plan: Dict,
              prev_tail: str) -> str:
        """
        Melt the steps into one section, then verify mechanically.

        On a second failure we ship the concatenated raw draft: a slightly bumpy
        section is a much better outcome than a smooth one that lost a definition.
        """
        sid = section["section_id"]
        raw_text = self._concat_steps(raw_steps)
        if not raw_text.strip():
            return ""

        # Code blocks never reach the Editor. Each step is masked locally,
        # then its placeholders are renumbered into one global sequence in a
        # SINGLE regex pass. Sequential .replace() calls collide: with offset 1,
        # renaming 0->1 and then 1->2 hits the placeholder just created, block 1
        # loses its marker, and the guard forces a raw fallback on every section
        # with more than one code block -- silently disabling the Editor exactly
        # where it matters most.
        def _renumber(text: str, offset: int) -> str:
            return re.sub(r"\[\[CODE_BLOCK_(\d+)\]\]",
                          lambda m: f"[[CODE_BLOCK_{offset + int(m.group(1))}]]",
                          text)

        masked_steps = []
        all_blocks: List[str] = []
        for step in raw_steps:
            masked_prose, blocks = mask_code(step["prose"])
            masked_prose = _renumber(masked_prose, len(all_blocks))
            all_blocks.extend(blocks)
            masked_clarifications = []
            for c in step["clarifications"]:
                m, b = mask_code(c["answer"])
                m = _renumber(m, len(all_blocks))
                all_blocks.extend(b)
                masked_clarifications.append({"question": c["question"], "answer": m})
            masked_steps.append({"title": step["title"], "prose": masked_prose,
                                 "clarifications": masked_clarifications})

        masked_raw = self._concat_steps(masked_steps)
        taught = plan.get("teaches", []) or section.get("tags", [])
        feedback = ""

        for attempt in range(self.cfg.max_editor_retries + 1):
            result = self.editor.smooth_section(
                section, masked_steps, self.memory.constitution,
                prev_tail + (f"\n\n[PREVIOUS EDIT REJECTED: {feedback}]" if feedback else ""))
            if not result or not str(result.get("content", "")).strip():
                feedback = "you returned no content"
                continue

            issues = verify_edit(masked_raw, result["content"], all_blocks, taught)
            if not issues:
                self.stats["edited"] += 1
                return unmask_code(result["content"], all_blocks)

            logC.warning(f"  [{sid}] edit guard: {issues}")
            feedback = "; ".join(issues)
            self.stats["edit_retry"] += 1

        logC.warning(f"  [{sid}] editor failed twice — shipping the raw draft")
        self.stats["edit_fallback"] += 1
        return raw_text

    # -------------------------------------------------------- one section ----
    def process_section(self, sid: str) -> Dict[str, Any]:
        section = self.sections[sid]
        started = time.time()
        logC.info(f"[{sid}] {section['title']}  "
                  f"(target {section.get('estimated_word_count', 700)} words)")

        self.memory.classroom.start_session(sid, section)
        prev_id = self.queue.prev_section_id

        # 1. Build the prompt from all memory layers.
        context = self.memory.build_context(section, prev_id)
        prev_tail = self.memory.previous_tail(prev_id, self.cfg.tail_chars)

        # 2. Plan, gate, review.
        plan = self._run_planning_loop(section, context)

        # 3. Draft, step by step, with the Student.
        raw_steps = self._run_teaching_loop(section, plan, context, prev_tail)
        if self.cfg.save_raw_steps:
            (BOOK.raw_steps / f"{sid}.json").write_text(
                json.dumps(raw_steps, indent=2, ensure_ascii=False), encoding="utf-8")

        # 4. Edit into one continuous section.
        final_content = self._edit(section, raw_steps, plan, prev_tail)
        if not final_content.strip():
            raise RuntimeError("section produced no content")

        # 5. Save. This file is the book, and it is written before the
        #    Archivist runs so a cataloguing failure cannot lose the prose.
        self.memory.drafts.write_markdown(sid, section["title"], final_content)

        # 6. Write back into memory. The Archivist reads the EDITED text.
        update = self.archivist.harvest(section, final_content,
                                        self.memory.ledger.all_open_promises())
        if update is None:
            logC.error(f"  [{sid}] archivist failed — recording a degraded entry")
            update = ArchivistAgent.fallback_update(section, final_content)
            self.stats["archivist_failed"] += 1

        summary = update.setdefault("summary", {})
        summary.setdefault("title", section["title"])
        summary["word_count"] = len(final_content.split())

        self.memory.ledger.apply_archivist_update(update, sid)
        # Track which figures this section leaned on, so _select_figures can
        # prefer fresh diagrams next time ("the same diagram is not leaned on
        # twice").
        self.memory.ledger.record_figures_used(
            [f["id"] for f in context.get("figures", [])], sid)
        self.memory.drafts.add_section(sid, section["title"], section["chapter_id"],
                                       final_content, summary.get("abstract", ""))
        self.memory.classroom.close_session(sid, self.memory.ledger)
        self.memory.conversation.close(sid)

        elapsed = time.time() - started
        words = len(final_content.split())
        self.stats["sections"] += 1
        self.stats["words"] += words
        logC.info(f"  [{sid}] done: {words} words in {elapsed / 60:.1f} min "
                  f"(ledger v{self.memory.ledger._cache['version']})")
        return {"section_id": sid, "words": words, "seconds": round(elapsed, 1),
                "degraded": bool(update.get("_degraded"))}

    # ------------------------------------------------- chapter boundaries ----
    def _maybe_roll_up(self, finished_sid: str) -> None:
        """
        Chapter rollups keep the spine small as the book grows.

        Written once per chapter, at the boundary, by the Archivist -- so that
        by section 120 the other eleven chapters cost a paragraph each rather
        than 119 abstracts competing for the model's attention.
        """
        chapter_id = self.sections[finished_sid]["chapter_id"]
        siblings = [s["section_id"] for s in self.toc["sections"]
                    if s["chapter_id"] == chapter_id]
        written = set(self.memory.ledger._cache["section_summaries"])
        if not set(siblings) <= written:
            return
        if chapter_id in self.memory.ledger._cache["chapter_rollups"]:
            return

        abstracts = "\n".join(
            f"- {sid}: {self.memory.ledger._cache['section_summaries'][sid].get('abstract', '')}"
            for sid in siblings)
        title = next(c["title"] for c in self.toc["chapters"]
                     if c["chapter_id"] == chapter_id)
        try:
            rollup = self.archivist._execute_step(
                f"CHAPTER {chapter_id}: \"{title}\"\n\nSECTION ABSTRACTS:\n{abstracts}\n\n"
                f"Write ONE paragraph (60-80 words) summarising what this chapter built "
                f"and what the reader can do at the end of it. Prose only.",
                "You are the Archivist. You summarise finished chapters factually.",
                max_tokens=250)
            if rollup:
                self.memory.ledger.add_chapter_rollup(chapter_id, rollup.strip())
                logC.info(f"  chapter rollup written for {chapter_id}")
        except Exception as exc:
            logC.warning(f"  rollup for {chapter_id} failed: {exc}")

    # ----------------------------------------------------------------- run ---
    def run(self, limit: Optional[int] = None) -> Dict[str, Any]:
        logC.info("=" * 70)
        logC.info(f"WRITING: \"{self.toc['book_title']}\" — "
                  f"{len(self.queue.pending)} sections to go")
        logC.info("=" * 70)
        started = time.time()
        done = 0

        while self.queue.pending and (limit is None or done < limit):
            sid = self.queue.pending[0]
            try:
                result = self.process_section(sid)
                self.queue.pending.pop(0)
                self.queue.completed.append(sid)
                self.queue.prev_section_id = sid
                self._maybe_roll_up(sid)
            except Exception as exc:
                logC.error(f"[{sid}] FAILED: {exc}")
                logC.debug(traceback.format_exc())
                self.queue.pending.pop(0)
                self.queue.failed.append({"section_id": sid, "error": str(exc)})
                self.stats["failed"] += 1
                if self.cfg.stop_on_error:
                    self.queue.save()
                    raise
            # Checkpoint after every section: a crash at 90 resumes at 90.
            self.queue.save()
            done += 1

        elapsed = time.time() - started
        return {
            "sections_written": self.stats["sections"],
            "words": self.stats["words"],
            "failed": self.stats["failed"],
            "remaining": len(self.queue.pending),
            "minutes": round(elapsed / 60, 1),
            "stats": dict(self.stats),
            "memory": self.memory.snapshot(),
        }


orchestrator = BookOrchestrator(
    toc, memory,
    {"writer": writer, "reviewer": reviewer, "student": student,
     "editor": editor, "archivist": archivist},
    gate, wcfg)

print("=" * 72)
print("PHASE 5 COMPLETE — gate, guard, orchestrator")
print("=" * 72)
print(f"Book        : \"{toc['book_title']}\"")
print(f"Queue       : {len(orchestrator.queue.pending)} pending, "
      f"{len(orchestrator.queue.completed)} already written")
print(f"Checkpoint  : {BOOK.work_queue}")
print("\nPer-section flow:")
print("  assemble → plan → [gate + reviewer] → teach ⇄ student → edit → guard")
print("  → save → ARCHIVIST → ledger + draft store → close session → checkpoint")
print("=" * 72)

In [ ]:
# ============================================================================
# Phase 5.4 — Write the book
# ============================================================================
"""
Start with `limit=1`. Read output/sections/<first>.md, read the ledger diff it
produced, and only then let it run unattended.

    "150 small generations with saved state after each one can be stopped,
     inspected, fixed, and resumed."

Set limit=None for the full run.
"""

RUN_LIMIT = 1        # <-- raise this, or set None, once you have read one section

run_report = orchestrator.run(limit=RUN_LIMIT)

print("=" * 72)
print("RUN REPORT")
print("=" * 72)
for key in ("sections_written", "words", "failed", "remaining", "minutes"):
    print(f"  {key:20s}: {run_report[key]}")

print("\nAgent activity:")
for agent in (writer, reviewer, student, editor, archivist):
    print(f"  {agent.name:<10} calls={agent.calls:<5} failures={agent.failures}")

print("\nContinuity gate + guard:")
for key, value in sorted(run_report["stats"].items()):
    if key.startswith(("gate_", "plan_", "edit", "doubts", "archivist")):
        print(f"  {key:24s}: {value}")

print("\nMemory after the run:")
for key, value in run_report["memory"].items():
    print(f"  {key:20s}: {value}")

if run_report["sections_written"]:
    _sid = orchestrator.queue.completed[-1]
    print("\n" + "-" * 72)
    print(f"LAST SECTION WRITTEN — {_sid}")
    print("-" * 72)
    print((BOOK.sections / f"{_sid}.md").read_text(encoding="utf-8")[:1200])
    print("\n" + "-" * 72)
    print("WHAT THE ARCHIVIST HARVESTED FROM IT")
    print("-" * 72)
    _diff = json.loads(BOOK.ledger_diffs.read_text(encoding="utf-8").strip().splitlines()[-1])
    print(json.dumps(_diff, indent=2)[:900])
print("=" * 72)

## Phase 6 — The Finishing Passes

Here is a truth about long-form writing that is easy to miss:

> **You cannot write a book that feels complete in one forward pass.** No human author does either.
> The first pass produces content. A second pass produces a book.

And because the ledger accumulated **structured facts rather than prose**, these passes are mostly
mechanical — this is where all the bookkeeping from Phases 1–5 finally pays out.

| Pass | What it does | Cost |
|---|---|---|
| 1. **Promise resolution** | Every unkept promise found before publication | embedding lookups |
| 2. **Glossary and index** | Fall out of the concept registry for free | zero LLM calls |
| 3. **Transitions** | A bridge wherever the seam between two sections shows | one call per rough seam |
| 4. **Contradiction sweep** | Claim clusters checked pairwise | a few dozen tiny calls |
| 5. **Coverage report** | Source material that never made the book | one set difference |

### Why the glossary is free

The concept registry already *is* a glossary: canonical name, definition, where it was defined,
where it was referenced.

> *"A glossary and an index are among the strongest signals of a finished book, and here they fall
> out of memory that was being kept anyway. Nothing else in this design has a better ratio of effort
> to perceived completeness."*

### Why the promise pass matters most

Every remaining open promise is a place where the book told the reader it would do something and
then did not. **That is precisely the feeling of incompleteness** — and now you have a list of them,
with the section that made each one.

Note the order of operations in Pass 1: before reporting a promise as broken, the book is *searched*
for it. A section may well have delivered on a promise without the Archivist noticing the connection.

### Why the coverage report is worth reading

```python
unused = [cid for cid in all_chunk_ids if not ledger.chunk_already_used(cid)]
```

Source material that never made it into the book. Sometimes that is correct — sources are often
redundant. Sometimes it is an entire topic the TOC clustering dropped, and **finding that out before
publication is the whole point.**

In [ ]:
# ============================================================================
# Phase 6.1 — Passes 1 and 2: promises, glossary, index
# ============================================================================

CONTRADICTION_SCHEMA = {"conflict": "boolean",
                        "explanation": "string — empty if no conflict"}


def resolve_promises(ledger: BookLedger, drafts: DraftStore,
                     similarity_threshold: float = 0.80) -> Dict[str, Any]:
    """
    Pass 1. Before reporting a promise as broken, search the book for it --
    a section may have delivered without the Archivist noticing.
    """
    resolved, gaps, orphaned = [], [], []

    for p in ledger.all_open_promises():
        candidates = drafts.find_similar(p["text"], k=3)
        if candidates and candidates[0]["similarity"] > similarity_threshold:
            p["status"] = "fulfilled"
            p["fulfilled_in"] = candidates[0]["section_id"]
            p["fulfilled_by"] = "resolution_pass"
            resolved.append(p)
        elif p.get("target_hint"):
            gaps.append(p)                 # a human decides: write it, or cut it
        else:
            p["status"] = "orphaned"       # the promise sentence gets edited out
            orphaned.append(p)

    ledger._flush()
    return {"auto_resolved": resolved, "gaps": gaps, "orphaned": orphaned}


def generate_glossary(ledger: BookLedger) -> str:
    """Pass 2a. The concept registry already IS a glossary."""
    lines = ["# Glossary\n"]
    for c in sorted(ledger._cache["concepts"].values(),
                    key=lambda x: x["canonical_name"].lower()):
        if not c.get("definition"):
            continue
        refs = ", ".join(c["referenced_in"][:5])
        entry = f"**{c['canonical_name']}** — {c['definition']}  \n*Introduced in {c['defined_in']}"
        if refs:
            entry += f". Also discussed in {refs}"
        entry += ".*\n"
        if c["aliases"]:
            entry += f"*Also written as: {', '.join(c['aliases'][:5])}.*\n"
        lines.append(entry)
    return "\n".join(lines)


def generate_index(ledger: BookLedger) -> str:
    """Pass 2b. Every concept, and every section that touches it."""
    lines = ["# Index\n"]
    entries = []
    for c in ledger._cache["concepts"].values():
        places = ([c["defined_in"]] if c["defined_in"] else []) + c["referenced_in"]
        if places:
            entries.append((c["canonical_name"], sorted(set(places))))
    for name, places in sorted(entries, key=lambda kv: kv[0].lower()):
        lines.append(f"**{name}** — {', '.join(places)}")
    return "\n".join(lines)


print("Phase 6.1 — promise resolution, glossary, index ready")

In [ ]:
# ============================================================================
# Phase 6.2 — Passes 3, 4, 5: transitions, contradictions, coverage
# ============================================================================


def find_rough_seams(toc: Dict, ledger: BookLedger, drafts: DraftStore,
                     max_report: int = 25) -> List[Dict]:
    """
    Pass 3. Read consecutive section pairs -- the closing_line of one and the
    opening of the next -- and flag the seams where a bridge is needed.

    Deliberately a REPORT, not an automatic rewrite: inserting generated
    sentences between finished sections is the one finishing pass that can make
    the prose worse, and it should be a human decision.
    """
    order = [s["section_id"] for s in toc["sections"]]
    summaries = ledger._cache["section_summaries"]
    seams = []

    for prev_id, next_id in zip(order, order[1:]):
        prev_sum = summaries.get(prev_id)
        if not prev_sum or next_id not in summaries:
            continue
        closing = (prev_sum.get("closing_line") or "").strip()
        opening = drafts.get_full(next_id)[:400].strip()
        if not closing or not opening:
            continue

        # A seam is rough when the closing line does not set up the opening at
        # all: no shared vocabulary, and no forward gesture.
        closing_words = set(re.findall(r"[a-z]{4,}", closing.lower()))
        opening_words = set(re.findall(r"[a-z]{4,}", opening.lower()))
        gestures_forward = bool(re.search(
            r"\bnext\b|\bnow\b|\bfollow|\bturn to\b|\bbut\b|\byet\b", closing.lower()))
        if not (closing_words & opening_words) and not gestures_forward:
            seams.append({"from": prev_id, "to": next_id,
                          "closing_line": closing, "opening": opening[:160]})
    return seams[:max_report]


def contradiction_sweep(ledger: BookLedger, llm, max_pairs: int = 40) -> List[Dict]:
    """
    Pass 4. Cluster the claims log by tag; within each cluster ask whether any
    pair conflicts. Because claims are short and typed, this is a few dozen tiny
    calls rather than a re-read of the whole book.
    """
    by_tag: Dict[str, List[Dict]] = defaultdict(list)
    for claim in ledger._cache["claims"]:
        for tag in claim.get("tags", []) or ["_untagged"]:
            by_tag[tag].append(claim)

    checked, conflicts, seen_pairs = 0, [], set()
    for tag, claims in by_tag.items():
        for i in range(len(claims)):
            for j in range(i + 1, len(claims)):
                a, b = claims[i], claims[j]
                if a["section_id"] == b["section_id"]:
                    continue
                pair = tuple(sorted((a.get("claim_id", str(i)), b.get("claim_id", str(j)))))
                if pair in seen_pairs or checked >= max_pairs:
                    continue
                seen_pairs.add(pair)
                checked += 1

                verdict = llm.generate_structured(
                    "You check whether two statements from the same book contradict "
                    "each other. Differences of emphasis or scope are NOT contradictions.",
                    f"A ({a['section_id']}): {a['text']}\nB ({b['section_id']}): {b['text']}\n\n"
                    f"Do these contradict?",
                    CONTRADICTION_SCHEMA, max_tokens=250)
                if verdict and verdict.get("conflict"):
                    conflicts.append({"tag": tag, "a": a, "b": b,
                                      "explanation": verdict.get("explanation", "")})
    return conflicts


def coverage_report(ledger: BookLedger, source: SourceMemory) -> Dict[str, Any]:
    """Pass 5. Source material that never made it into the book."""
    all_ids = source.all_chunk_ids()
    unused = [cid for cid in all_ids if not ledger.chunk_already_used(cid)]

    by_document: Dict[str, Dict[str, int]] = defaultdict(lambda: {"used": 0, "unused": 0})
    for cid in all_ids:
        doc = source.meta.get(cid, {}).get("source_document", "unknown")
        by_document[doc]["unused" if cid in set(unused) else "used"] += 1

    # A document that contributed nothing at all is the signal worth acting on:
    # usually a topic the TOC clustering dropped entirely.
    silent = [doc for doc, counts in by_document.items() if counts["used"] == 0]
    return {"chunks_total": len(all_ids), "chunks_unused": len(unused),
            "coverage_pct": round(100 * (len(all_ids) - len(unused)) /
                                  max(1, len(all_ids)), 1),
            "by_document": dict(by_document),
            "documents_never_used": silent,
            "unused_sample": unused[:20]}


print("Phase 6.2 — transitions, contradictions, coverage ready")

In [ ]:
# ============================================================================
# Phase 6.3 — Assemble the manuscript and the completeness report
# ============================================================================


def assemble_manuscript(toc: Dict, drafts: DraftStore, ledger: BookLedger) -> str:
    """Stitch the sections into one manuscript, in TOC order."""
    lines = [f"# {toc['book_title']}", ""]
    for chapter in toc["chapters"]:
        lines.append(f"\n# {chapter['order']}. {chapter['title']}\n")
        rollup = ledger._cache["chapter_rollups"].get(chapter["chapter_id"])
        if rollup:
            lines.append(f"> {rollup}\n")
        for section in toc["sections"]:
            if section["chapter_id"] != chapter["chapter_id"]:
                continue
            body = drafts.get_full(section["section_id"])
            if body:
                lines.append(body.rstrip() + "\n")
            else:
                lines.append(f"## {section['title']}\n\n*[NOT YET WRITTEN — "
                             f"{section['section_id']}]*\n")
    return "\n".join(lines)


def run_finishing_passes(run_contradictions: bool = True) -> Dict[str, Any]:
    """All five passes, then write the book directory."""
    BOOK.book.mkdir(parents=True, exist_ok=True)
    logC.info("=" * 70)
    logC.info("FINISHING PASSES")
    logC.info("=" * 70)

    logC.info("Pass 1: promise resolution")
    promises = resolve_promises(ledger, drafts)

    logC.info("Pass 2: glossary and index")
    (BOOK.book / "glossary.md").write_text(generate_glossary(ledger), encoding="utf-8")
    (BOOK.book / "index.md").write_text(generate_index(ledger), encoding="utf-8")

    logC.info("Pass 3: transitions")
    seams = find_rough_seams(toc, ledger, drafts)

    conflicts: List[Dict] = []
    if run_contradictions:
        logC.info("Pass 4: contradiction sweep")
        conflicts = contradiction_sweep(ledger, llm)

    logC.info("Pass 5: coverage")
    coverage = coverage_report(ledger, source)

    manuscript = assemble_manuscript(toc, drafts, ledger)
    (BOOK.book / "manuscript.md").write_text(manuscript, encoding="utf-8")

    written = set(ledger._cache["section_summaries"])
    planned = [s["section_id"] for s in toc["sections"]]
    report = {
        "book_title": toc["book_title"],
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "sections_written": len(written),
        "sections_planned": len(planned),
        "sections_missing": [s for s in planned if s not in written],
        "words": len(manuscript.split()),
        "estimated_pages": round(len(manuscript.split()) / 350),
        "promises": {
            "auto_resolved": len(promises["auto_resolved"]),
            "gaps": [{"promise_id": p["promise_id"], "text": p["text"],
                      "made_in": p["made_in"], "target": p.get("target_hint")}
                     for p in promises["gaps"]],
            "orphaned": len(promises["orphaned"]),
        },
        "rough_seams": seams,
        "contradictions": conflicts,
        "coverage": coverage,
        "ledger": ledger.stats(),
    }
    (BOOK.book / "completeness_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

    lines = [f"# Completeness report — {toc['book_title']}", "",
             f"Sections written: {report['sections_written']} / {report['sections_planned']}",
             f"Length: {report['words']:,} words ≈ {report['estimated_pages']} pages", ""]
    if report["sections_missing"]:
        lines += ["## Sections never written", ""] + \
                 [f"- {s}" for s in report["sections_missing"]] + [""]
    if report["promises"]["gaps"]:
        lines += ["## Promises the book did not keep", ""] + [
            f"- **{p['text']}** — promised in {p['made_in']}, due in {p['target']}"
            for p in report["promises"]["gaps"]] + [""]
    if conflicts:
        lines += ["## Possible contradictions", ""] + [
            f"- {c['a']['section_id']}: \"{c['a']['text']}\"  \n"
            f"  vs {c['b']['section_id']}: \"{c['b']['text']}\"  \n"
            f"  → {c['explanation']}" for c in conflicts] + [""]
    if seams:
        lines += ["## Rough seams between sections", ""] + [
            f"- {s['from']} → {s['to']}: ends \"{s['closing_line'][:80]}…\"" for s in seams] + [""]
    lines += ["## Source coverage", "",
              f"- {coverage['coverage_pct']}% of chunks used "
              f"({coverage['chunks_unused']} never used)"]
    if coverage["documents_never_used"]:
        lines += [f"- **Documents that contributed nothing:** "
                  f"{', '.join(coverage['documents_never_used'])}"]
    (BOOK.book / "completeness_report.md").write_text("\n".join(lines), encoding="utf-8")

    return report


# Run the passes only once sections exist.
if ledger._cache["section_summaries"]:
    final_report = run_finishing_passes(run_contradictions=True)

    print("=" * 72)
    print(f"PHASE 6 COMPLETE — \"{final_report['book_title']}\"")
    print("=" * 72)
    print(f"Sections     : {final_report['sections_written']} / "
          f"{final_report['sections_planned']}")
    print(f"Length       : {final_report['words']:,} words "
          f"≈ {final_report['estimated_pages']} pages")
    print()
    print(f"Promises auto-resolved : {final_report['promises']['auto_resolved']}")
    print(f"Promises UNKEPT        : {len(final_report['promises']['gaps'])}"
          "   <-- each is a place the book owes the reader")
    for p in final_report["promises"]["gaps"][:5]:
        print(f"    {p['made_in']} → {p['target']}: {p['text'][:70]}")
    print(f"Possible contradictions: {len(final_report['contradictions'])}")
    print(f"Rough seams            : {len(final_report['rough_seams'])}")
    print(f"Source coverage        : {final_report['coverage']['coverage_pct']}% "
          f"({final_report['coverage']['chunks_unused']} chunks never used)")
    if final_report["coverage"]["documents_never_used"]:
        print(f"  documents unused     : "
              f"{final_report['coverage']['documents_never_used'][:3]}")
    print()
    print("Written to output/book/:")
    for name in ("manuscript.md", "glossary.md", "index.md",
                 "completeness_report.md", "completeness_report.json"):
        path = BOOK.book / name
        print(f"  {name:28s} {path.stat().st_size:>8,} bytes"
              if path.exists() else f"  {name:28s} (missing)")
    print("=" * 72)
else:
    print("No sections written yet — run Phase 5.4 first, then re-run this cell.")

### Honest Caveats

Accurate observations about Pipeline C as implemented.

**1. The Archivist can be wrong, and its errors compound.** If it hallucinates a definition or
misses a promise, that error enters the ledger and *every later section inherits it*. Mitigations
here: a strict schema with a repair loop, temperature 0, an instruction to quote the section's own
words, and **a diff written to `output/ledger_diffs.jsonl` on every single write**. When a chapter
comes out wrong, read the diffs — that is what they are for.

**2. The ledger slice is a heuristic, and it is the weakest link.** It selects concepts by tag
overlap, plus defined prerequisites, plus a similarity fallback when tags are thin. If a section's
tags are wrong, the Writer gets the wrong fifteen concepts and the whole apparatus quietly
underperforms. The `[ledger slice selected by: …]` line at the top of the block tells you which path
fired.

**3. Alias resolution is exact string matching.** It catches "ReAct loop" versus "react loop" but not
"the loop we built in Chapter 3". Adding embedding similarity would help and would also introduce
false positives that silently merge two distinct concepts. Start exact, measure, loosen if needed.

**4. Nobody reviews the Editor.** The code masking and four guard checks cover the *detectable*
failures; they do not cover a subtly weakened explanation. Two things make this tolerable: the raw
drafts are kept in `output/raw_steps/`, so every raw→edited pair is a readable diff, and the fallback
ships raw rather than shipping something suspect. **If you read ten diffs after the first chapter and
the Editor is inventing things, turn it off** — it is one method in `process_section`.

**5. Two rewriting passes now sit between the source and the reader.** The Writer paraphrases the
chunks; the Editor paraphrases the Writer. Each hop is a chance for a fact to soften. The Archivist
reading the *edited* text limits the damage, since the book at least stays consistent with itself —
but self-consistency is not accuracy.

**6. Section-level granularity may be too coarse.** The Archivist runs once per section. Within a
section, steps see each other only through the conversation layer. If intra-section repetition shows
up, run a lightweight Archivist per step — but only if the problem is real.

**7. Nothing here fixes a bad table of contents.** If the curriculum puts chapter 6's prerequisite in
chapter 9, the Continuity Gate will *detect* it and then flag the same gap for every affected
section. **That is a signal to fix the TOC, not to keep writing.**

**8. The Student is the same model judging its own prose.** It is prompted to hold only the reader's
knowledge, but it cannot truly un-know the source. Expect it to be a *generous* reader. A low
`doubts_raised` count is not proof the prose is clear.

**9. Chapter rollups only appear once a chapter is complete.** Until then, other chapters contribute
titles alone to the spine. That is the design's pyramid working as intended — the ledger still
carries their concepts, claims and promises — but a first chapter written before any rollup exists
sees less than section 120 will.

**10. Degraded harvests are recorded, not hidden.** When the Archivist call fails outright, a
`_degraded` entry goes into the ledger with the mechanical facts only and no invented concepts. It
is counted in the run report as `archivist_failed`. A non-zero count means those sections contribute
nothing to the book's memory of itself.

---

### Summary Table of Every Technique in Pipeline C

| Technique | What it does | Failure it kills |
|---|---|---|
| **Constitution** | Fixed identity, style, conventions, running examples | #8 tone drift |
| **Forbidden patterns** | Bans the tics that mark generated text | #8 |
| **Concept registry** | Canonical terms, aliases, definitions, depth | #2, #5 |
| **Ledger seeding** | Pre-loads concepts from Pipeline B's tag output | #2, cold start |
| **Alias resolution** | Maps any variant back to its canonical concept | #2 |
| **Enrich in place** | An alias never creates a second concept | #2 |
| **Defined prerequisites in the slice** | A section sees the definitions it will lean on | #2, #5 |
| **Claims log** | Every assertion, with its section | #6 |
| **Promise ledger** | Forward references, open/fulfilled/orphaned | #3, #4 |
| **Deferred-doubt promotion** | Turns "we'll cover it later" into a tracked debt | #3 |
| **Example registry** | Running-example state across the whole book | #7 |
| **Coverage map** | Which chunks are used, by whom, how deeply | #1 |
| **Usage notes on source** | "already used in sec_04_01 — do not re-explain" | #1 |
| **Section summaries** | 60–100 word abstract of what each section taught | #1, #4 |
| **Draft store** | Every finished section, on disk and embedded | #1 |
| **Verbatim tail** | Previous section's last 800 characters, injected raw | #8 |
| **Neighbourhood retrieval** | Chunks arrive with their neighbours | Mid-thought fragments |
| **Semantic source expansion** | Relevant chunks the TOC did not assign | Coverage gaps |
| **Provenance in the header** | document · type · timestamp on every chunk | Weak grounding |
| **Context Assembler** | Budgeted selection from every layer | Overflow, dilution |
| **Declared budgets + logged truncation** | Overflow is visible, never silent | Slow, vague prompts |
| **Ledger slice** | Only the facts relevant to this section | Scale |
| **Similarity fallback** | Thin tags still get a useful slice | The weakest link |
| **Relevance over recency** | 3 nearest by meaning, not by position | #1, #4 |
| **Chapter rollups** | Pyramid summaries so the spine stays small | Scale |
| **Classroom memory** | Per-section plan, drift, doubts | Working state |
| **Anchored compaction** | Recent turns verbatim, older ones summarized | Context overflow |
| **Templated compaction** | Fixed schema so summaries stay comparable | Summary drift |
| **Rewrite-not-stack** | One merged summary, fixed size | Telephone-game drift |
| **The Editor** | Melts teaching steps and clarifications into one section | Seams, bolted-on Q&A |
| **Code masking** | Fenced blocks never reach the Editor's context | Broken code |
| **Edit guard** | Four deterministic checks, then fallback to raw | Silent content loss |
| **Deliberate closing line** | Section endings written on purpose | #8 |
| **The Archivist** | One structured call per section, harvests into memory | **all of them** |
| **Archivist reads the edited text** | Catalogues what the reader will see | Drift between book and memory |
| **Degraded-harvest fallback** | Records mechanical facts, invents nothing | A silent empty harvest |
| **Ledger diff log** | Every write traceable to its section | Compounding harvest errors |
| **Continuity Gate** | Five deterministic pre-write checks, zero LLM calls | #1, #2, #3, #5 |
| **Per-section checkpoint** | A crash at 90 resumes at 90 | Losing a night's run |
| **Promise resolution pass** | Every unkept promise found before publication | #3, #4 |
| **Glossary and index** | Fall out of the concept registry for free | Perceived completeness |
| **Transition report** | Seams flagged, not auto-rewritten | #8, and bad auto-edits |
| **Contradiction sweep** | Claim clusters checked pairwise | #6 |
| **Coverage report** | Source material that never made the book | Gaps |

---

### The One-Paragraph Takeaway

Pipeline C writes a 250-to-500 page book as roughly 175 separate sections, each generated in its own
conversation by a model that remembers none of the others. The whole design exists to close that
gap, and it closes it with plumbing rather than with a better model: at the centre sits the **Book
Ledger** — not prose, but a queryable database of small typed facts about the book itself, seeded for
free from Pipeline B's tag artifacts and kept current by the **Archivist**, an agent that reads each
finished section and harvests it back into memory in one structured call. A **Context Assembler**
ships a relevant *slice* of that ledger rather than dumping the whole thing, alongside the previous
section's verbatim ending and the three nearest already-written sections by meaning rather than by
recency. A **Continuity Gate** runs five deterministic checks on the teaching *plan* before a word is
generated. An **Editor** turns the raw teaching loop into one continuous section with a deliberate
opening and closing, with code blocks masked so it cannot break them and four guard checks so it
cannot quietly lose a definition. And because the ledger accumulates structured facts rather than
prose, the finishing passes are nearly mechanical: every unkept promise becomes a list, and the
glossary and index simply fall out of the concept registry. **A book feels complete not because every
paragraph is good, but because paragraph 4,000 knows what paragraph 40 said — and every layer here
exists to carry that knowledge forward, one section at a time.**